# ⚡ Fastest Qwen3.8-Flash-Next Jacobian Lens (J-Lens) Fitting Notebook

**Auto-adapts to randomly assigned Google Colab High-RAM runtimes: `G4` (RTX PRO 6000 Blackwell 96 GB), `A100 80 GB`, and `A100 40 GB`.**

---

### Why Naive J-Lens Fitting Fails on Qwen3.8-Flash-Next (`qwen4_exp`)
1. **4-Branch Gated Residual (`hc_count=4`)**: Decoder layers output `[B, T, 10240]` (`4 × 2560`), collapsed to `[B, T, 2560]` by a final `hyper_connection_mixer` before `lm_head` (there is no `model.norm`). Standard `jlens.from_hf` crashes immediately.
   - Our exact J-lens targets `h_final = hyper_connection_mixer(H_47)` (`2560` dims) w.r.t. each layer's full 4-branch residual `H_l` (`10240` dims), yielding **`J_l ∈ R^{2560 × 10240}`** in the **exact same number of output cotangents (`K = 2560`)**!
2. **172.78 GiB FP8 Checkpoint (`112.5 GiB` MoE + `47.7 GiB` 51B PLE + `8.2 GiB` dense BF16)**:
   - Naive `dim_batch=8` backprop streams `112.5 GiB` across PCIe **320× per prompt** (`36 TB/prompt` ≈ 24 minutes/prompt!) and OOM-kills A100-40G (83.5 GiB RAM) immediately.
   - Naive HF `AutoModelForCausalLM` triggers `transformers#48349` (silently corrupting BF16 dense layers) and downloads unneeded Vision + MTP shards.

---

### Core Breakthroughs in This Notebook
| Optimization | Impact |
|---|---|
| **Single-Sweep Layer-by-Layer Reverse Propagation** | Forward runs ONCE at `B = 1` (~120 MB saved inputs). Backward holds ALL `K = 2560` cotangents in ONE in-place GPU buffer (`6.25 GiB`) and computes all `2560` VJPs of layer `l` **before** moving to layer `l-1` -> **1× weight sweep/prompt** (40×–120× less PCIe/NVMe traffic!). |
| **Fused Active-Expert MoE VJP (`grouped_mm` + Triton)** | Each layer dequantizes ONLY hit experts **ONCE** via a custom 128×128 Triton FP8 kernel (~2.5 ms), runs Tensor-Core-saturated CUTLASS/cuBLAS `grouped_mm` across `n_e × K_blk` rows, fuses SwiGLU backward in Triton, and includes the exact top-k softmax router VJP without materializing `[T, K, 512]`. |
| **Zero-RAM 51B PLE Mmap / Pre-Extraction** | `MADV_RANDOM` mmap directly on PLE safetensors shards (`0 GiB` VRAM, `0 GiB` pinned RAM, `< 2 ms`/prompt), with automatic two-stage pre-extraction (`~64 MB`) if disk is tight. |
| **Interleaved 3-Tier Async Double-Buffering** | Permanently pins dense BF16 (`8.2 GiB`) + max MoE layers in **Tier 0 GPU HBM**, spills to **Tier 1 Pinned/Pageable RAM**, and (on A100 40G) interleaves **Tier 2 contiguous NVMe slabs** every ~2.4 layers so background NVMe preads & H2D DMAs hide **100% behind backward compute**! |
| **Hardware-Optimal Kernels** | **G4 Blackwell (SM120)**: `FlashQLA v0.1.3` (TileLang SM120 fused GDN fwd+bwd). **A100 (SM80)**: `FLA` Triton fused GDN fwd+bwd. **QSA (all GPUs)**: exact indexer bypass for `T ≤ 2048` + L2-cached exact `B=1` attention VJP (cuBLAS batched GEMMs). |

---

### Concrete Auto-Planned Profiles (Default `T = 128`, `Qwen/Qwen3.8-Flash-Next-FP8`)
| Colab Runtime | GPU Arch | Tier 0 GPU HBM | Tier 1 Host RAM | Tier 2 NVMe Slabs | GDN Kernel | Est. Speed |
|---|---|---|---|---|---|---|
| **G4 High RAM** (95.6 GiB / 177 GiB RAM) | SM120 | 8.2G dense + **26 MoE layers** | **22 MoE layers** (pinned, interleaved) | **0 layers** | **FlashQLA SM120** | **~7–10 s/prompt** |
| **A100 80G High RAM** (79.2 GiB / 167 GiB RAM) | SM80 | 8.2G dense + **19 MoE layers** | **29 MoE layers** (pinned, interleaved) | **0 layers** | **FLA Triton SM80** | **~18–22 s/prompt** |
| **A100 40G High RAM** (39.6 GiB / 83.5 GiB RAM) | SM80 | 8.2G dense + **5 MoE layers** | **23 MoE layers** (14 pin + 9 page) | **20 MoE layers** (interleaved) | **FLA Triton SM80** | **~22–28 s/prompt** |


## 1. Environment Setup & GPU-Adaptive Kernel Installation

In [ ]:
import os, subprocess, sys, torch

# Critical CUDA allocator & fast HF transfer settings (set before heavy allocations)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

assert torch.cuda.is_available(), "No CUDA GPU detected! Select A100 or G4 High-RAM runtime in Colab."
props = torch.cuda.get_device_properties(0)
cc = (props.major, props.minor)
print(f"Detected GPU: {props.name} | Compute Capability: SM{cc[0]}{cc[1]} | VRAM: {props.total_memory / 1024**3:.1f} GiB")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

def pip_install(pkgs, extra_args=(), optional=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *extra_args, *pkgs]
    print("Running:", " ".join(cmd))
    res = subprocess.run(cmd, check=False)
    if res.returncode != 0:
        if optional:
            print(f"  [optional install skipped with code {res.returncode}]: {pkgs}")
        else:
            raise RuntimeError(f"pip install failed for {pkgs}")

# Core dependencies
pip_install([
    "transformers>=5.17.0",
    "safetensors>=0.4.5",
    "huggingface_hub>=0.26.0",
    "hf_transfer",
    "datasets>=3.0.0",
    "psutil",
    "tqdm",
])

# Architecture-specific optimal GDN linear-attention kernel:
#   - SM90 / SM100 / SM120 (Colab G4 Blackwell, H100, B200): FlashQLA v0.1.3 (TileLang fused fwd+bwd) + FLA fallback
#   - SM80 (Colab A100 40G / 80G): FLA Triton fused chunk_gated_delta_rule (FlashQLA requires SM90+)
if cc in ((9, 0), (10, 0), (10, 3), (12, 0), (12, 1)):
    print(f"SM{cc[0]}{cc[1]} supports FlashQLA (2-3x faster GDN fwd/bwd)! Installing flash-qla + flash-linear-attention...")
    pip_install(["flash-qla"], optional=True)
    pip_install(["flash-linear-attention"], optional=True)
elif cc >= (8, 0):
    print(f"SM{cc[0]}{cc[1]} detected (A100 Ampere): installing flash-linear-attention (Triton GDN fused fwd+bwd)...")
    pip_install(["flash-linear-attention"], optional=True)
else:
    print(f"SM{cc[0]}{cc[1]}: using PyTorch chunked GDN backend.")

print("Setup complete!")


## 2. Fitting Configuration & Crash-Proof Async Checkpointing
Checkpoints are stored as a **single `.safetensors` file** holding the running mean `J_mean = jacobian_sum / n_done` (plus `n_done`, `next_idx`, and layer metadata in the safetensors header):
- **2× smaller (`2.34 GiB` in `float16` vs `4.69 GiB` in `float32`)** with zero fp16 overflow risk for any `n_prompts`.
- **Zero GPU stalls**: writes to local NVMe in `< 1 s`, then syncs asynchronously on a background thread to **Google Drive** and/or **Hugging Face Hub** (recommended — free unlimited LFS/Xet storage at ~300–800 MB/s, avoiding Google Drive's 15 GB free quota!).
- **Immediately usable on disconnect**: any interrupted checkpoint can be loaded directly as a valid `Qwen38JacobianLens`, and re-running automatically resumes from whichever copy (Local / Drive / HF Hub) has the highest `next_idx`.

In [ ]:
#@title Qwen3.8-Flash-Next J-Lens & Checkpoint Configuration { display-mode: "form" }
MODEL_ID = "Qwen/Qwen3.8-Flash-Next-FP8"  #@param {type:"string"}
N_PROMPTS = 100              #@param {type:"integer"}  # 100 = usable lens (paper §9.3); 1000 = full paper lens
PROMPT_OFFSET = 0            #@param {type:"integer"}  # Shift start index to shard across parallel Colab sessions & merge()
MAX_SEQ_LEN = 128            #@param {type:"integer"}  # Paper default: 128 tokens (keeps full K=2560 in 1 sweep!)
SKIP_FIRST = 16              #@param {type:"integer"}  # Exclude attention-sink prefix tokens (paper default: 16)
LAYER_STRIDE = 1             #@param {type:"integer"}  # 1 = all 48 layers (0..47); 2 = every 2nd layer
PLE_MODE = "auto"            #@param ["auto", "mmap", "preextract"]

#@markdown ### Async Checkpointing (Zero GPU Stall)
CKPT_DTYPE = "float16"       #@param ["float16", "float32"]  # float16 = 2.34 GiB (recommended); float32 = 4.69 GiB exact
CHECKPOINT_EVERY_PROMPTS = 10 #@param {type:"integer"}        # Trigger checkpoint every N prompts
CHECKPOINT_EVERY_MINUTES = 10.0 #@param {type:"number"}       # OR every M minutes (whichever comes first)
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}         # Async mirror to Google Drive (only 2.34 GiB in fp16!)
DRIVE_OUT_DIR = "/content/drive/MyDrive/qwen38_jlens_out" #@param {type:"string"}
HF_CHECKPOINT_REPO = ""      #@param {type:"string"}          # Optional e.g. "your-name/qwen38-jlens-ckpt" (unlimited storage, fastest!)
HF_REPO_PRIVATE = True       #@param {type:"boolean"}
ALSO_SAVE_LEGACY_PT = False  #@param {type:"boolean"}         # Also write `.pt` alongside `.safetensors` (doubles disk/Drive size)
LOCAL_WORK_DIR = "/content/qwen38_work" #@param {type:"string"}
HF_TOKEN = ""                #@param {type:"string"}

import os
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
SLAB_DIR = os.path.join(LOCAL_WORK_DIR, "moe_slabs")
PLE_DIR = os.path.join(LOCAL_WORK_DIR, "ple_shards")
LOCAL_CKPT_DIR = os.path.join(LOCAL_WORK_DIR, "checkpoints")
for _d in (SLAB_DIR, PLE_DIR, LOCAL_CKPT_DIR):
    os.makedirs(_d, exist_ok=True)

# Resolve optional HF_TOKEN from Colab Secrets if not pasted directly
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_TOKEN = HF_TOKEN or None
HF_CHECKPOINT_REPO = HF_CHECKPOINT_REPO.strip() or None

DRIVE_ENABLED = False
if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
        DRIVE_ENABLED = True
    except Exception as exc:
        print(f"Google Drive mount unavailable ({exc}); using local NVMe (+ HF Hub if configured).")

TAG = f"off{PROMPT_OFFSET}_n{N_PROMPTS}_s{LAYER_STRIDE}"
CKPT_FILENAME = f"qwen38_jlens_ckpt_{TAG}.safetensors"
LENS_FILENAME = f"qwen38_flash_next_jlens_{TAG}.safetensors"

LOCAL_CKPT_PATH = os.path.join(LOCAL_CKPT_DIR, CKPT_FILENAME)
DRIVE_CKPT_PATH = os.path.join(DRIVE_OUT_DIR, CKPT_FILENAME) if DRIVE_ENABLED else None
FINAL_LENS_LOCAL = os.path.join(LOCAL_CKPT_DIR, LENS_FILENAME)
FINAL_LENS_DRIVE = os.path.join(DRIVE_OUT_DIR, LENS_FILENAME) if DRIVE_ENABLED else None
OUT_DIR = DRIVE_OUT_DIR if DRIVE_ENABLED else LOCAL_CKPT_DIR

print("Checkpoint Destinations:")
print(f"  Local NVMe (fast <1s) : {LOCAL_CKPT_PATH}")
print(f"  Google Drive (async)  : {DRIVE_CKPT_PATH or 'disabled'}")
print(f"  HF Hub (async Xet)    : {f'{HF_CHECKPOINT_REPO}/{CKPT_FILENAME}' if HF_CHECKPOINT_REPO else 'disabled'}")
print(f"  Cadence & Dtype       : every {CHECKPOINT_EVERY_PROMPTS} prompts or {CHECKPOINT_EVERY_MINUTES} min | {CKPT_DTYPE}")


## 3. Install Self-Contained `qwen38_jlens` Engine Modules
Writing the verified, modular engine directly into `/content/qwen38_jlens/` so this notebook runs 100% standalone on Colab without needing external repo clones.

In [ ]:
import os
os.makedirs('qwen38_jlens', exist_ok=True)


In [ ]:
%%writefile qwen38_jlens/hardware.py
"""Hardware auto-detection and concrete 3-tier offloading planner for Qwen3.8-Flash-Next J-lens.

Targets randomly assigned Google Colab High-RAM runtimes:
  - Colab G4 High RAM: NVIDIA RTX PRO 6000 Blackwell (SM120, ~95.6 GiB VRAM, ~177 GiB RAM)
  - Colab A100 80G High RAM: NVIDIA A100-SXM4-80GB (SM80, ~79.2 GiB VRAM, ~167 GiB RAM)
  - Colab A100 40G High RAM: NVIDIA A100-SXM4-40GB (SM80, ~39.6 GiB VRAM, ~83.5 GiB RAM)
  - Also adapts cleanly to H100 80G (SM90), B200 (SM100), L4/V100, or synthetic CPU/GPU models.

Core memory insights for Qwen/Qwen3.8-Flash-Next-FP8 (48 layers, 512 experts, hc_count=4):
  - Dense BF16 backbone (excluding vision, MTP, lm_head): ~8.2 GiB -> ALWAYS Tier 0 (GPU HBM).
  - Routed MoE experts (FP8 E4M3 + 128x128 fp32 scales): 2.3443 GiB/layer -> 112.53 GiB across 48 layers.
  - 51B PLE n-gram embedding table (FP8 E4M3 + scalar scale, layer 2 only): 47.68 GiB -> NEVER in RAM/VRAM;
    served via MADV_RANDOM NVMe mmap (0 GiB RAM) or pre-extracted into a ~64 MB cache if disk is tight.
  - Tier placement interleaves slower tiers (Tier 2 NVMe, Tier 1 Host RAM) evenly between faster tiers
    so background NVMe reads and async H2D DMA transfers overlap 100% behind backward compute.
"""

from __future__ import annotations

import math
import os
import shutil
from dataclasses import dataclass, field
from enum import Enum
from typing import Sequence

import torch


class StorageTier(str, Enum):
    """Where a decoder layer's routed MoE FP8 expert weights reside."""

    GPU_HBM = "tier0_gpu_hbm"
    HOST_PINNED = "tier1_host_pinned"
    HOST_PAGEABLE = "tier1_host_pageable"
    NVME_SLAB = "tier2_nvme_slab"


@dataclass(frozen=True)
class HardwareProfile:
    """Detected hardware capabilities of the current runtime."""

    gpu_name: str
    compute_capability: tuple[int, int]
    vram_total_gib: float
    vram_free_gib: float
    ram_total_gib: float
    ram_avail_gib: float
    disk_free_gib: float
    cpu_count: int
    has_cuda: bool = True

    @property
    def arch_label(self) -> str:
        major, minor = self.compute_capability
        if major == 12:
            return f"Blackwell-Workstation (SM{major}{minor})"
        if major == 10:
            return f"Blackwell-Datacenter (SM{major}{minor})"
        if major == 9:
            return f"Hopper (SM{major}{minor})"
        if major == 8:
            return f"Ampere (SM{major}{minor})"
        return f"SM{major}{minor}"

    @property
    def supports_flash_qla(self) -> bool:
        """FlashQLA v0.1.3 supports SM90, SM100, SM103, SM120, SM121."""
        return self.has_cuda and self.compute_capability in (
            (9, 0),
            (10, 0),
            (10, 3),
            (12, 0),
            (12, 1),
        )

    @property
    def supports_native_fp8_tc(self) -> bool:
        """Hardware FP8 Tensor Cores exist on SM89+ (Ada/Hopper/Blackwell)."""
        return self.has_cuda and self.compute_capability >= (8, 9)

    @property
    def supports_grouped_mm(self) -> bool:
        """PyTorch CUTLASS/cuBLAS grouped_mm supports SM80+ on torch>=2.9."""
        return self.has_cuda and self.compute_capability >= (8, 0)


@dataclass
class OffloadPlan:
    """Concrete, deterministic offloading & kernel execution plan."""

    profile: HardwareProfile
    num_layers: int
    d_model: int
    hc_count: int
    max_seq_len: int
    bytes_per_moe_layer: int
    dense_backbone_gib: float

    # Tier assignment per layer index (0 .. num_layers - 1)
    layer_tiers: dict[int, StorageTier]

    # J-lens sweep & micro-chunk tuning parameters
    k_pass: int                  # Cotangents held in GPU G_curr per layer sweep (2560 => 1 single sweep!)
    attn_k_chunk: int            # Cotangents processed per GDN/QSA kernel call
    moe_k_chunk: int             # Cotangents processed per MoE grouped_mm VJP call (multiple of 16!)
    expert_dequant_chunk: int    # Max active experts dequantized simultaneously into BF16 (256 or 512)
    gpu_staging_slots: int       # Number of double-buffered FP8 GPU staging slots (0 or 2)
    host_pinned_staging_slots: int  # Pinned host staging slots for pageable/NVMe layers (0 or 2)

    # PLE & Disk strategy
    ple_mode: str                # "mmap" (keep 48 GiB PLE shards on disk) or "preextract" (~64 MB cache)
    evict_raw_shards: bool       # Delete raw HF safetensors shards after unpacking each layer group
    download_batch_gib: float    # Max raw shard download batch before packing + evicting

    # Preferred kernels
    gdn_backend: str             # "flash_qla", "fla", or "torch"
    attn_backend: str            # "l2_cached_exact" (T<=512) or "sdpa_flash"
    moe_backend: str             # "grouped_mm_bf16" (1x Triton FP8 dequant) or "loop_mm_bf16"
    use_triton_fp8_dequant: bool
    use_triton_swiglu_vjp: bool

    # Estimated memory usage summary (GiB)
    est_vram_peak_gib: float = 0.0
    est_ram_peak_gib: float = 0.0
    est_disk_peak_gib: float = 0.0
    notes: list[str] = field(default_factory=list)

    def tier_counts(self) -> dict[StorageTier, int]:
        counts = {t: 0 for t in StorageTier}
        for t in self.layer_tiers.values():
            counts[t] += 1
        return counts

    def summary(self) -> str:
        c = self.tier_counts()
        moe_gb = self.bytes_per_moe_layer / (1024**3)
        n_sweeps = math.ceil(self.d_model / self.k_pass)
        lines = [
            "========================================================================",
            f" Hardware Detected : {self.profile.gpu_name} [{self.profile.arch_label}]",
            f"   VRAM            : {self.profile.vram_free_gib:.1f} / {self.profile.vram_total_gib:.1f} GiB free",
            f"   System RAM      : {self.profile.ram_avail_gib:.1f} / {self.profile.ram_total_gib:.1f} GiB avail",
            f"   Disk Free       : {self.profile.disk_free_gib:.1f} GiB   |   CPUs: {self.profile.cpu_count}",
            "------------------------------------------------------------------------",
            " Concrete 3-Tier Offloading Plan (Qwen3.8-Flash-Next-FP8)",
            f"   Dense Backbone  : {self.dense_backbone_gib:.2f} GiB BF16 -> Tier 0 GPU HBM (permanent)",
            f"   Tier 0 (GPU HBM): {c[StorageTier.GPU_HBM]:2d}/{self.num_layers} MoE layers ({c[StorageTier.GPU_HBM]*moe_gb:5.1f} GiB FP8, 0 ms transfer)",
            f"   Tier 1 (Pinned) : {c[StorageTier.HOST_PINNED]:2d}/{self.num_layers} MoE layers ({c[StorageTier.HOST_PINNED]*moe_gb:5.1f} GiB FP8, async H2D DMA)",
            f"   Tier 1 (Pageabl): {c[StorageTier.HOST_PAGEABLE]:2d}/{self.num_layers} MoE layers ({c[StorageTier.HOST_PAGEABLE]*moe_gb:5.1f} GiB FP8, threaded pin->DMA)",
            f"   Tier 2 (NVMe)   : {c[StorageTier.NVME_SLAB]:2d}/{self.num_layers} MoE layers ({c[StorageTier.NVME_SLAB]*moe_gb:5.1f} GiB FP8, interleaved pread)",
            f"   51B PLE Table   : mode={self.ple_mode!r} (0 GiB VRAM, 0 GiB pinned RAM)",
            f"   Shard Eviction  : {self.evict_raw_shards} (download batches <= {self.download_batch_gib:.0f} GiB)",
            "------------------------------------------------------------------------",
            " Kernel & VJP Schedule",
            f"   GDN Kernel      : {self.gdn_backend}",
            f"   QSA Attention   : {self.attn_backend} (QSA indexer bypassed for T <= 2048)",
            f"   MoE VJP Kernel  : {self.moe_backend} | Triton FP8 dequant={self.use_triton_fp8_dequant} | Triton SwiGLU={self.use_triton_swiglu_vjp}",
            f"   Weight Sweeps   : {n_sweeps}x/prompt (k_pass={self.k_pass}, attn_k_chunk={self.attn_k_chunk}, moe_k_chunk={self.moe_k_chunk}, expert_dequant_chunk={self.expert_dequant_chunk})",
            f"   Est. Peak Usage : VRAM ~{self.est_vram_peak_gib:.1f} GiB | RAM ~{self.est_ram_peak_gib:.1f} GiB | Disk ~{self.est_disk_peak_gib:.1f} GiB",
            "========================================================================",
        ]
        if self.notes:
            lines.append(" Planner Notes:")
            for note in self.notes:
                lines.append(f"   - {note}")
        return "\n".join(lines)


def detect_hardware(disk_path: str = ".") -> HardwareProfile:
    """Inspect live GPU, Host RAM, Disk, and CPU resources."""
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram_total_gib = vm.total / (1024**3)
        ram_avail_gib = vm.available / (1024**3)
    except ImportError:
        # Fallback to /proc/meminfo on Linux
        ram_total_gib, ram_avail_gib = 64.0, 48.0
        if os.path.exists("/proc/meminfo"):
            info = {}
            with open("/proc/meminfo", "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 2:
                        info[parts[0].rstrip(":")] = int(parts[1])
            ram_total_gib = info.get("MemTotal", 64 * 1024 * 1024) / (1024**2)
            ram_avail_gib = info.get("MemAvailable", 48 * 1024 * 1024) / (1024**2)

    check_path = disk_path if os.path.exists(disk_path) else "."
    disk_free_gib = shutil.disk_usage(check_path).free / (1024**3)
    cpu_count = os.cpu_count() or 4

    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        free_b, total_b = torch.cuda.mem_get_info(0)
        return HardwareProfile(
            gpu_name=props.name,
            compute_capability=(props.major, props.minor),
            vram_total_gib=total_b / (1024**3),
            vram_free_gib=free_b / (1024**3),
            ram_total_gib=ram_total_gib,
            ram_avail_gib=ram_avail_gib,
            disk_free_gib=disk_free_gib,
            cpu_count=cpu_count,
            has_cuda=True,
        )
    return HardwareProfile(
        gpu_name="CPU-only",
        compute_capability=(0, 0),
        vram_total_gib=0.0,
        vram_free_gib=0.0,
        ram_total_gib=ram_total_gib,
        ram_avail_gib=ram_avail_gib,
        disk_free_gib=disk_free_gib,
        cpu_count=cpu_count,
        has_cuda=False,
    )


def _evenly_spaced_indices(n_total: int, count: int, excluded: set[int] | None = None) -> list[int]:
    """Select `count` maximally spread-out indices from `[0, n_total) \\ excluded`.

    Used to interleave Tier 2 NVMe layers (and Tier 1 Host layers) evenly among
    faster tiers so background NVMe reads and async H2D DMAs hide 100% behind
    backward compute of adjacent layers.
    """
    available = [i for i in range(n_total) if excluded is None or i not in excluded]
    if count <= 0:
        return []
    if count >= len(available):
        return list(available)
    m = len(available)
    # Pick evenly spaced positions in [0, m - 1]
    chosen_pos: list[int] = []
    used: set[int] = set()
    for k in range(count):
        ideal = (k + 0.5) * m / count - 0.5
        pos = min(m - 1, max(0, int(round(ideal))))
        # Resolve rare rounding collision to nearest unused slot
        if pos in used:
            for delta in range(1, m):
                if pos + delta < m and (pos + delta) not in used:
                    pos = pos + delta
                    break
                if pos - delta >= 0 and (pos - delta) not in used:
                    pos = pos - delta
                    break
        used.add(pos)
        chosen_pos.append(pos)
    chosen_pos.sort()
    return [available[p] for p in chosen_pos]


def compute_moe_layer_bytes(
    num_experts: int = 512,
    hidden_size: int = 2560,
    moe_intermediate_size: int = 640,
    block_size: int = 128,
    weight_bytes: int = 1,
) -> int:
    """Exact byte footprint of one layer's routed MoE experts (weights + 128x128 fp32 scales)."""
    gate_up_w = num_experts * (2 * moe_intermediate_size) * hidden_size * weight_bytes
    down_w = num_experts * hidden_size * moe_intermediate_size * weight_bytes
    if weight_bytes == 1:
        gate_up_s = (
            num_experts
            * math.ceil((2 * moe_intermediate_size) / block_size)
            * math.ceil(hidden_size / block_size)
            * 4
        )
        down_s = (
            num_experts
            * math.ceil(hidden_size / block_size)
            * math.ceil(moe_intermediate_size / block_size)
            * 4
        )
    else:
        gate_up_s = 0
        down_s = 0
    return gate_up_w + down_w + gate_up_s + down_s


def build_offload_plan(
    profile: HardwareProfile,
    *,
    num_layers: int = 48,
    d_model: int = 2560,
    hc_count: int = 4,
    num_experts: int = 512,
    moe_intermediate_size: int = 640,
    max_seq_len: int = 128,
    num_source_layers: int | None = None,
    dense_backbone_gib: float = 8.2,
    ple_table_gib: float = 47.68,
    force_ple_mode: str | None = None,
    weight_bytes: int = 1,
) -> OffloadPlan:
    """Compute the optimal 3-tier placement and kernel configuration for `profile`."""
    notes: list[str] = []
    d_hc = d_model * hc_count
    n_src = num_layers if num_source_layers is None else num_source_layers
    bytes_per_layer = compute_moe_layer_bytes(
        num_experts=num_experts,
        hidden_size=d_model,
        moe_intermediate_size=moe_intermediate_size,
        weight_bytes=weight_bytes,
    )
    layer_gib = bytes_per_layer / (1024**3)
    vram_usable = min(profile.vram_free_gib, profile.vram_total_gib) if profile.has_cuda else 0.0

    # 1. Choose k_pass (cotangents held in GPU G_curr), attn_k_chunk, moe_k_chunk, expert_dequant_chunk
    #    For standard max_seq_len=128: full K=2560 G_curr is 6.25 GiB -> fits in 1 single sweep on A100-40G/80G/G4!
    full_g_curr_gib = (d_model * max_seq_len * d_hc * 2) / (1024**3)
    max_g_curr_gib = 6.5 if vram_usable < 60.0 else 13.0
    if full_g_curr_gib <= max_g_curr_gib or not profile.has_cuda:
        k_pass = d_model
    else:
        # Pick largest divisor of d_model (or multiple of 128) fitting max_g_curr_gib
        raw_k = max(128, int((max_g_curr_gib * (1024**3)) / (max_seq_len * d_hc * 2)))
        k_pass = max(128, (raw_k // 128) * 128)
        notes.append(f"max_seq_len={max_seq_len} exceeds 1-sweep G_curr budget; using k_pass={k_pass} ({math.ceil(d_model / k_pass)} sweeps/prompt).")

    g_curr_gib = (k_pass * max_seq_len * d_hc * 2) / (1024**3)

    if vram_usable >= 70.0:
        attn_k_chunk = 128
        moe_k_chunk = 256
        expert_dequant_chunk = num_experts  # Dequantize all active experts in 1 shot (~3.8-4.7 GiB max)
        vram_safety_gib = 4.0
    elif vram_usable >= 34.0:
        # Colab A100 40G profile
        attn_k_chunk = 64
        moe_k_chunk = 128
        expert_dequant_chunk = min(num_experts, 256)  # Chunked BF16 dequant caps peak scratch at 2.35 GiB
        vram_safety_gib = 3.5
    else:
        attn_k_chunk = 32
        moe_k_chunk = 64
        expert_dequant_chunk = min(num_experts, 128)
        vram_safety_gib = 2.0

    # Ensure chunks divide/bound cleanly by k_pass
    attn_k_chunk = min(attn_k_chunk, k_pass)
    moe_k_chunk = min(moe_k_chunk, k_pass)

    # Active BF16 expert dequant peak (2 bytes/param * expert_dequant_chunk / num_experts)
    dequant_bf16_gib = (
        (expert_dequant_chunk / max(1, num_experts))
        * (num_experts * 3 * moe_intermediate_size * d_model * 2)
        / (1024**3)
    )
    # Scratch for GDN/QSA + MoE VJP intermediates
    # MoE VJP peak: ~ (S * moe_k_chunk * (2*D + 3*inter) * 2B) where S = max_seq_len * 10
    s_pairs = max_seq_len * 10
    moe_scratch_gib = (s_pairs * moe_k_chunk * (2 * d_model + 3 * moe_intermediate_size) * 2) / (1024**3)
    attn_scratch_gib = max(0.8, (attn_k_chunk * max_seq_len * 48 * 128 * 10 * 2) / (1024**3))
    compute_scratch_gib = max(moe_scratch_gib, attn_scratch_gib)

    # Check if ALL layers fit directly in GPU HBM (no staging slots needed)
    all_gpu_vram = (
        dense_backbone_gib
        + g_curr_gib
        + dequant_bf16_gib
        + compute_scratch_gib
        + vram_safety_gib
        + num_layers * layer_gib
    )
    if profile.has_cuda and all_gpu_vram <= vram_usable:
        n_tier0 = num_layers
        gpu_staging_slots = 0
    elif profile.has_cuda:
        gpu_staging_slots = 2  # Double-buffered async H2D prefetch slots
        fixed_vram = (
            dense_backbone_gib
            + g_curr_gib
            + gpu_staging_slots * layer_gib
            + dequant_bf16_gib
            + compute_scratch_gib
            + vram_safety_gib
        )
        avail_for_tier0 = max(0.0, vram_usable - fixed_vram)
        n_tier0 = min(num_layers, int(avail_for_tier0 // layer_gib))
    else:
        n_tier0 = 0
        gpu_staging_slots = 0

    remaining_after_gpu = num_layers - n_tier0

    # 2. Host RAM budget (Tier 1 Pinned + Tier 1 Pageable) vs Tier 2 NVMe Slabs
    # Fixed Host RAM overhead:
    #   - Running fp32 Jacobian sums: n_src * d_model * d_hc * 4B (~4.69 GiB for all 48 layers)
    #   - lm_head on CPU (~1.18 GiB)
    #   - Saved B=1 layer inputs H_l across 48 layers (~0.12 GiB)
    #   - Python / PyTorch / OS page cache + streaming load temp buffer safety margin
    jac_sum_gib = (n_src * d_model * d_hc * 4) / (1024**3)
    lm_head_cpu_gib = 1.2 if d_model >= 2048 else 0.1
    ram_safety_gib = 14.0 if profile.ram_total_gib >= 120.0 else 13.5

    # Cap total pinned memory to <= 45% of total RAM (prevents Linux mlock/OOM-killer starvation)
    max_pinned_gib = min(
        0.45 * profile.ram_total_gib,
        max(0.0, profile.ram_avail_gib - ram_safety_gib - jac_sum_gib - lm_head_cpu_gib),
    )

    if remaining_after_gpu == 0:
        n_tier1_pinned = 0
        n_tier1_pageable = 0
        n_tier2_nvme = 0
        host_pinned_staging_slots = 0
    else:
        # First check if all remaining layers fit in directly pinned host RAM
        if remaining_after_gpu * layer_gib <= max_pinned_gib:
            n_tier1_pinned = remaining_after_gpu
            n_tier1_pageable = 0
            n_tier2_nvme = 0
            host_pinned_staging_slots = 0
        else:
            # Reserve 2 pinned host staging slots for pageable RAM / NVMe layers
            host_pinned_staging_slots = 2 if profile.has_cuda else 0
            staging_ram_gib = host_pinned_staging_slots * layer_gib
            avail_ram_for_layers = max(
                0.0,
                profile.ram_avail_gib - ram_safety_gib - jac_sum_gib - lm_head_cpu_gib - staging_ram_gib,
            )
            max_host_layers = min(remaining_after_gpu, int(avail_ram_for_layers // layer_gib))
            avail_pinned_for_layers = max(0.0, max_pinned_gib - staging_ram_gib)
            n_tier1_pinned = min(max_host_layers, int(avail_pinned_for_layers // layer_gib))
            n_tier1_pageable = max(0, max_host_layers - n_tier1_pinned)
            n_tier2_nvme = remaining_after_gpu - max_host_layers

    # 3. Interleaved layer-to-tier assignment!
    #    - Space Tier 2 NVMe layers evenly across [0..num_layers-1] so each NVMe read (~0.8s)
    #      hides behind 2-3 consecutive Tier 0/1 layers (~1.0-1.3s compute on A100).
    #    - Space Tier 0 GPU layers evenly among remaining layers so on G4/A100-80G every
    #      Tier 1 H2D DMA (~90 ms) hides behind Tier 0 + Tier 1 compute!
    layer_tiers: dict[int, StorageTier] = {}
    t2_indices = set(_evenly_spaced_indices(num_layers, n_tier2_nvme))
    for l in t2_indices:
        layer_tiers[l] = StorageTier.NVME_SLAB

    t0_indices = set(_evenly_spaced_indices(num_layers, n_tier0, excluded=t2_indices))
    for l in t0_indices:
        layer_tiers[l] = StorageTier.GPU_HBM

    assigned = t2_indices | t0_indices
    t1_pageable_indices = set(_evenly_spaced_indices(num_layers, n_tier1_pageable, excluded=assigned))
    for l in t1_pageable_indices:
        layer_tiers[l] = StorageTier.HOST_PAGEABLE

    for l in range(num_layers):
        if l not in layer_tiers:
            layer_tiers[l] = StorageTier.HOST_PINNED

    # 4. Disk & PLE strategy
    tier2_disk_gib = n_tier2_nvme * layer_gib
    download_batch_gib = 15.0 if profile.disk_free_gib >= 80.0 else 8.0
    if force_ple_mode is not None:
        ple_mode = force_ple_mode
    else:
        # Need room for: Tier 2 slabs + PLE shards (47.7 GiB) + download batch + 20 GiB OS/checkpoint buffer
        needed_for_mmap = tier2_disk_gib + ple_table_gib + download_batch_gib + 20.0
        if profile.disk_free_gib >= needed_for_mmap:
            ple_mode = "mmap"
        else:
            ple_mode = "preextract"
            notes.append(
                f"Disk free ({profile.disk_free_gib:.1f} GiB) < mmap threshold ({needed_for_mmap:.1f} GiB); "
                "auto-selecting two-stage PLE pre-extraction + shard cleanup!"
            )

    # Always evict raw MoE safetensors shards after unpacking if keeping full 164 GiB raw repo + slabs risks disk full
    evict_raw_shards = profile.disk_free_gib < (175.0 + tier2_disk_gib + 25.0)
    if evict_raw_shards:
        notes.append("Streaming shard-by-shard pack-and-evict enabled (keeps peak disk well below Colab 235 GiB limit).")

    # 5. Best kernel selection per GPU architecture
    if profile.supports_flash_qla:
        gdn_backend = "flash_qla"
    elif profile.has_cuda and profile.compute_capability >= (8, 0):
        gdn_backend = "fla"
    else:
        gdn_backend = "torch"

    attn_backend = "l2_cached_exact" if max_seq_len <= 512 else "sdpa_flash"
    moe_backend = "grouped_mm_bf16" if profile.supports_grouped_mm else "loop_mm_bf16"
    use_triton_fp8_dequant = profile.has_cuda and weight_bytes == 1
    use_triton_swiglu_vjp = profile.has_cuda

    est_vram_peak = (
        dense_backbone_gib
        + n_tier0 * layer_gib
        + gpu_staging_slots * layer_gib
        + g_curr_gib
        + dequant_bf16_gib
        + compute_scratch_gib
    ) if profile.has_cuda else 0.0

    est_ram_peak = (
        (n_tier1_pinned + n_tier1_pageable + host_pinned_staging_slots) * layer_gib
        + jac_sum_gib
        + lm_head_cpu_gib
        + 4.0  # Python/PyTorch base overhead
    )

    ple_disk = ple_table_gib if ple_mode == "mmap" else 0.5
    raw_disk = download_batch_gib if evict_raw_shards else (num_layers * layer_gib + dense_backbone_gib)
    est_disk_peak = tier2_disk_gib + ple_disk + raw_disk + 2.5  # +2.5 GiB saved lens/ckpt

    if n_tier2_nvme > 0:
        spacing = num_layers / n_tier2_nvme
        notes.append(
            f"Tier 2 NVMe layers ({n_tier2_nvme}) interleaved every ~{spacing:.1f} layers "
            "for zero-stall async pread overlap."
        )

    return OffloadPlan(
        profile=profile,
        num_layers=num_layers,
        d_model=d_model,
        hc_count=hc_count,
        max_seq_len=max_seq_len,
        bytes_per_moe_layer=bytes_per_layer,
        dense_backbone_gib=dense_backbone_gib,
        layer_tiers=layer_tiers,
        k_pass=k_pass,
        attn_k_chunk=attn_k_chunk,
        moe_k_chunk=moe_k_chunk,
        expert_dequant_chunk=expert_dequant_chunk,
        gpu_staging_slots=gpu_staging_slots,
        host_pinned_staging_slots=host_pinned_staging_slots,
        ple_mode=ple_mode,
        evict_raw_shards=evict_raw_shards,
        download_batch_gib=download_batch_gib,
        gdn_backend=gdn_backend,
        attn_backend=attn_backend,
        moe_backend=moe_backend,
        use_triton_fp8_dequant=use_triton_fp8_dequant,
        use_triton_swiglu_vjp=use_triton_swiglu_vjp,
        est_vram_peak_gib=est_vram_peak,
        est_ram_peak_gib=est_ram_peak,
        est_disk_peak_gib=est_disk_peak,
        notes=notes,
    )


In [ ]:
%%writefile qwen38_jlens/ple_mmap.py
"""Zero-RAM mmap reader & exact n-gram hasher for Qwen3.8-Flash-Next's 51B PLE embedding table.

In Qwen/Qwen3.8-Flash-Next-FP8:
  - PLE is active at layer 2 (1-indexed -> decoder layer index 1).
  - The n-gram table has ~320,001,536 rows x 160 dims = 51.2B params (~47.68 GiB in FP8 E4M3)
    split across `split_ngram_parts = 128` shard tensors (`ngram_embedding.shard_0..127.weight`),
    plus a single per-tensor scale (`ngram_embedding.weight_scale`, shape `[1]`).
  - Each token only reads 16 rows x 160 bytes = 2.5 KB! For a 128-token J-lens prompt, that is
    only 320 KB of reads -> < 2 ms via MADV_RANDOM mmap with ZERO RAM footprint.
  - Because `ple_embedding(input_ids)` depends ONLY on `input_ids` (never on hidden states),
    when disk is tight we can also pre-extract embeddings (~640 KB/prompt) and delete the 48 GiB
    PLE shards before downloading MoE shards!
"""

from __future__ import annotations

import json
import math
import mmap as _mmap
import os
import re
import struct
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Iterable, Sequence

import numpy as np
import torch


# SplitMix64 constants matching transformers/models/qwen4_exp/modeling_qwen4_exp.py exactly
_MASK64 = (1 << 64) - 1
_SPLITMIX_GAMMA = 0x9E3779B97F4A7C15
_SPLITMIX_M1 = 0xBF58476D1CE4E5B9
_SPLITMIX_M2 = 0x94D049BB133111EB
_PRIME_1 = 10007

_SHARD_IDX_RE = re.compile(r"ngram_embedding\.shard_(\d+)\.weight$")

_SAFETENSORS_DTYPE_INFO: dict[str, tuple[torch.dtype, int]] = {
    "F8_E4M3": (torch.float8_e4m3fn, 1),
    "F8_E5M2": (torch.float8_e5m2, 1),
    "BF16": (torch.bfloat16, 2),
    "F16": (torch.float16, 2),
    "F32": (torch.float32, 4),
}


def _splitmix64(value: int) -> int:
    value = (value + _SPLITMIX_GAMMA) & _MASK64
    value = ((value ^ (value >> 30)) * _SPLITMIX_M1) & _MASK64
    value = ((value ^ (value >> 27)) * _SPLITMIX_M2) & _MASK64
    return (value ^ (value >> 31)) & _MASK64


def build_layer_multipliers(
    unigram_vocab_size: int, ngram_size: int, ple_layer_index: int, seed: int
) -> torch.Tensor:
    """Bit-identical to Qwen4Exp `_build_layer_multipliers`."""
    max_long = (1 << 63) - 1
    multiplier_max = max_long // max(unigram_vocab_size, 1)
    half_bound = max(1, multiplier_max // 2)
    base_seed = seed + _PRIME_1 * ple_layer_index
    multipliers = []
    for index in range(ngram_size):
        value = (base_seed + _SPLITMIX_GAMMA * (index + 1)) & _MASK64
        multipliers.append(2 * (_splitmix64(value) % half_bound) + 1)
    return torch.tensor(multipliers, dtype=torch.long)


def _is_prime(value: int) -> bool:
    if value < 2:
        return False
    if value % 2 == 0:
        return value == 2
    for divisor in range(3, math.isqrt(value) + 1, 2):
        if value % divisor == 0:
            return False
    return True


def _find_nth_prime_after(start: int, count: int) -> int:
    prime = start
    for _ in range(count):
        prime += 1
        while not _is_prime(prime):
            prime += 1
    return prime


def parse_safetensors_header(path: str) -> tuple[dict, int]:
    """Return `(header_dict, data_start_byte_offset)` of a safetensors file."""
    with open(path, "rb") as f:
        (header_len,) = struct.unpack("<Q", f.read(8))
        header = json.loads(f.read(header_len))
    header.pop("__metadata__", None)
    return header, 8 + header_len


@dataclass
class PleHashSpec:
    """Deterministic prime-vocab n-gram hashing specification matching Qwen4ExpTextNGramEmbedding."""

    vocab_size: int
    ngram_size: int = 3
    heads_per_ngram: int = 8
    ngram_vocab_size_base: int = 20_000_000
    make_ngram_vocab_size_divisible_by: int = 128
    ple_embed_dim: int = 2560
    seed: int = 1234
    eos_token_id: int = 248044
    ple_layer_index: int = 0

    def __post_init__(self) -> None:
        self.context_len = self.ngram_size - 1
        self.ngram_heads = (self.ngram_size - 1) * self.heads_per_ngram
        self.head_dim_per_ngram = self.ple_embed_dim // self.ngram_heads

        self.head_vocab_sizes: list[int] = []
        self.head_offsets: list[int] = []
        total = 0
        for head_idx in range(self.ngram_heads):
            global_head_idx = self.ple_layer_index * self.ngram_heads + head_idx
            size = _find_nth_prime_after(self.ngram_vocab_size_base - 1, global_head_idx + 1)
            self.head_vocab_sizes.append(size)
            self.head_offsets.append(total)
            total += size
        self.total_vocab_size = total
        div = self.make_ngram_vocab_size_divisible_by
        self.padded_vocab_size = math.ceil(total / div) * div

        self.layer_multipliers = build_layer_multipliers(
            self.vocab_size, self.ngram_size, self.ple_layer_index, self.seed
        )
        self.ngram_heads_vocab_sizes = torch.tensor(self.head_vocab_sizes, dtype=torch.long)
        self.ngram_heads_offsets = torch.tensor(self.head_offsets, dtype=torch.long)

    def _shift_right_ignore_eos(self, token_ids: torch.Tensor, shift: int) -> torch.Tensor:
        if shift == 0:
            return token_ids
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device, dtype=torch.long)
        eos_positions = torch.where(token_ids == self.eos_token_id, positions, -1)
        previous_eos_inclusive = torch.cummax(eos_positions, dim=1).values
        previous_eos = torch.cat(
            [eos_positions.new_full((batch_size, 1), -1), previous_eos_inclusive[:, :-1]], dim=1
        )
        segment_start = previous_eos + 1
        position_in_segment = positions.unsqueeze(0) - segment_start
        source_positions = positions - shift
        gather_positions = source_positions.clamp_min(0).unsqueeze(0).expand(batch_size, -1)
        shifted = token_ids.gather(dim=1, index=gather_positions)
        valid = (position_in_segment >= shift) & (source_positions.unsqueeze(0) >= 0)
        return torch.where(valid, shifted, token_ids.new_full((), self.eos_token_id))

    def compute_ngram_ids(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Compute `[B, T, ngram_heads]` global row indices bit-identical to Qwen4Exp."""
        device = input_ids.device
        input_ids = input_ids.long()
        prev_ctx = input_ids.new_full((input_ids.shape[0], self.context_len), self.eos_token_id)
        token_history = torch.cat([prev_ctx, input_ids], dim=-1)
        shifted_tokens = [
            self._shift_right_ignore_eos(token_history, shift) for shift in range(self.ngram_size)
        ]

        mult = self.layer_multipliers.to(device)
        vocab_sizes = self.ngram_heads_vocab_sizes.to(device)
        offsets = self.ngram_heads_offsets.to(device)

        blocks = []
        for ngram in range(2, self.ngram_size + 1):
            start_idx = (ngram - 2) * self.heads_per_ngram
            end_idx = start_idx + self.heads_per_ngram
            mixed_ids = shifted_tokens[0] * mult[0]
            for pos in range(1, ngram):
                mixed_ids = torch.bitwise_xor(mixed_ids, shifted_tokens[pos] * mult[pos])
            h_sizes = vocab_sizes[start_idx:end_idx]
            h_offsets = offsets[start_idx:end_idx]
            ngram_ids = torch.remainder(mixed_ids.unsqueeze(-1), h_sizes.view(1, 1, -1))
            blocks.append(ngram_ids + h_offsets.view(1, 1, -1))

        return torch.cat(blocks, dim=-1)[:, -input_ids.shape[1] :]


class MmapPleTable:
    """Zero-RAM NVMe mmap lookup table for Qwen3.8-Flash-Next's sharded FP8/BF16 PLE embedding.

    Supports both:
      - Sharded `ngram_embedding.shard_0..N.weight` safetensors layout (real Qwen3.8-Flash-Next-FP8),
      - Single-tensor `ngram_embedding.weight` safetensors or in-memory tensor (unit tests / small checkpoints).
    """

    def __init__(
        self,
        hash_spec: PleHashSpec,
        shards: Sequence[tuple[str, int, int]],
        row_dim: int,
        torch_dtype: torch.dtype,
        weight_scale: float | torch.Tensor | None = None,
        workers: int = 16,
    ) -> None:
        self.hash_spec = hash_spec
        self.row_dim = int(row_dim)
        self.torch_dtype = torch_dtype
        self.elem_bytes = torch.tensor([], dtype=torch_dtype).element_size()
        self.row_bytes = self.row_dim * self.elem_bytes
        if weight_scale is None:
            self.weight_scale = 1.0
        elif torch.is_tensor(weight_scale):
            self.weight_scale = float(weight_scale.float().item())
        else:
            self.weight_scale = float(weight_scale)

        self._mmaps: list[np.memmap] = []
        self._shard_starts: list[int] = []
        self._shard_ends: list[int] = []
        cursor = 0
        for path, byte_offset, n_rows in shards:
            mm = np.memmap(path, dtype=np.uint8, mode="r", offset=byte_offset, shape=(n_rows, self.row_bytes))
            # Avoid Linux readahead polluting page cache on random 160-byte hashed rows
            raw_mm = getattr(mm, "_mmap", None)
            if raw_mm is not None and hasattr(_mmap, "MADV_RANDOM"):
                try:
                    raw_mm.madvise(_mmap.MADV_RANDOM)
                except OSError:
                    pass
            self._mmaps.append(mm)
            self._shard_starts.append(cursor)
            cursor += n_rows
            self._shard_ends.append(cursor)
        self.total_rows = cursor
        self._shard_starts_np = np.array(self._shard_starts, dtype=np.int64)
        self._pool = ThreadPoolExecutor(max_workers=max(1, min(workers, len(self._mmaps))))

    @classmethod
    def from_safetensors_files(
        cls,
        hash_spec: PleHashSpec,
        shard_paths: Iterable[str],
        weight_scale: float | torch.Tensor | None = None,
        workers: int = 16,
    ) -> "MmapPleTable":
        """Discover all PLE n-gram shards from safetensors headers and open zero-copy mmaps."""
        indexed_shards: dict[int, tuple[str, int, int]] = {}
        single_entry: tuple[str, int, int] | None = None
        row_dim: int | None = None
        torch_dtype: torch.dtype | None = None
        discovered_scale = weight_scale

        for path in shard_paths:
            header, data_start = parse_safetensors_header(path)
            for key, meta in header.items():
                if key.endswith("ngram_embedding.weight_scale") and discovered_scale is None:
                    off0, off1 = meta["data_offsets"]
                    dt_str = meta["dtype"]
                    dt, _ = _SAFETENSORS_DTYPE_INFO[dt_str]
                    with open(path, "rb") as f:
                        f.seek(data_start + off0)
                        raw = f.read(off1 - off0)
                    t = torch.frombuffer(bytearray(raw), dtype=dt)
                    discovered_scale = float(t.float().item())
                    continue

                m = _SHARD_IDX_RE.search(key)
                if m is not None:
                    s_idx = int(m.group(1))
                    n_rows, r_dim = meta["shape"]
                    dt, _ = _SAFETENSORS_DTYPE_INFO[meta["dtype"]]
                    off0 = data_start + meta["data_offsets"][0]
                    indexed_shards[s_idx] = (path, off0, int(n_rows))
                    row_dim = int(r_dim)
                    torch_dtype = dt
                elif key.endswith("ngram_embedding.weight"):
                    n_rows, r_dim = meta["shape"]
                    dt, _ = _SAFETENSORS_DTYPE_INFO[meta["dtype"]]
                    off0 = data_start + meta["data_offsets"][0]
                    single_entry = (path, off0, int(n_rows))
                    row_dim = int(r_dim)
                    torch_dtype = dt

        if indexed_shards:
            ordered = [indexed_shards[i] for i in sorted(indexed_shards)]
        elif single_entry is not None:
            ordered = [single_entry]
        else:
            raise FileNotFoundError("No `ngram_embedding.shard_*.weight` or `ngram_embedding.weight` found in shards.")

        assert row_dim is not None and torch_dtype is not None
        return cls(
            hash_spec=hash_spec,
            shards=ordered,
            row_dim=row_dim,
            torch_dtype=torch_dtype,
            weight_scale=discovered_scale,
            workers=workers,
        )

    def gather_rows(self, row_ids: np.ndarray) -> torch.Tensor:
        """Gather `row_ids` (`[...]` int64) -> dequantized `bfloat16` tensor `[..., row_dim]` on CPU."""
        orig_shape = row_ids.shape
        flat_ids = np.ascontiguousarray(row_ids, dtype=np.int64).reshape(-1)
        if flat_ids.size == 0:
            return torch.empty((*orig_shape, self.row_dim), dtype=torch.bfloat16)

        uniq, inverse = np.unique(flat_ids, return_inverse=True)
        if uniq[0] < 0 or uniq[-1] >= self.total_rows:
            raise IndexError(f"PLE row id out of range [{uniq[0]}, {uniq[-1]}] vs total_rows={self.total_rows}")

        # Map each unique global row id to (shard_idx, local_row)
        shard_ids = np.searchsorted(self._shard_starts_np, uniq, side="right") - 1
        local_rows = uniq - self._shard_starts_np[shard_ids]

        out_bytes = np.empty((uniq.size, self.row_bytes), dtype=np.uint8)
        bounds = np.flatnonzero(np.diff(shard_ids)) + 1
        starts = np.concatenate(([0], bounds))
        ends = np.concatenate((bounds, [uniq.size]))

        if uniq.size <= 512 or len(starts) == 1:
            for s, e in zip(starts.tolist(), ends.tolist()):
                si = int(shard_ids[s])
                out_bytes[s:e] = self._mmaps[si][local_rows[s:e]]
        else:
            def _fetch(span: tuple[int, int]) -> None:
                s, e = span
                si = int(shard_ids[s])
                out_bytes[s:e] = self._mmaps[si][local_rows[s:e]]

            list(self._pool.map(_fetch, zip(starts.tolist(), ends.tolist())))

        # Re-expand to original order and reinterpret bytes as native torch dtype (FP8 / BF16)
        gathered_bytes = out_bytes[inverse]
        raw_tensor = torch.from_numpy(gathered_bytes).view(self.torch_dtype).view(*orig_shape, self.row_dim)
        if self.torch_dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
            return raw_tensor.to(torch.bfloat16).mul_(self.weight_scale)
        return raw_tensor.to(torch.bfloat16)

    def lookup(self, input_ids: torch.Tensor, device: torch.device | str | None = None) -> torch.Tensor:
        """Compute n-gram IDs for `input_ids` `[B, T]` and return `[B, T, ple_embed_dim]` BF16."""
        ngram_ids = self.hash_spec.compute_ngram_ids(input_ids.cpu())  # [B, T, ngram_heads]
        emb = self.gather_rows(ngram_ids.numpy()).flatten(-2)  # [B, T, ple_embed_dim]
        target_dev = input_ids.device if device is None else torch.device(device)
        return emb.to(target_dev, non_blocking=True)

    def preextract_corpus(self, tokenized_prompts: Sequence[torch.Tensor]) -> list[torch.Tensor]:
        """Pre-gather PLE embeddings (`[1, T, ple_embed_dim]` BF16 CPU) for an entire prompt list.

        ~640 KB per 128-token prompt -> 64 MB for 100 prompts, 640 MB for 1000 prompts.
        Allows safely deleting the 48 GiB PLE shards on disk-constrained Colab VMs!
        """
        return [self.lookup(ids, device="cpu").contiguous() for ids in tokenized_prompts]

    def close(self) -> None:
        self._pool.shutdown(wait=False)
        for mm in self._mmaps:
            raw_mm = getattr(mm, "_mmap", None)
            if raw_mm is not None:
                try:
                    raw_mm.close()
                except Exception:
                    pass
        self._mmaps.clear()


class PreextractedPleProvider:
    """Zero-disk, zero-latency PLE embedding provider backed by pre-extracted CPU tensors."""

    def __init__(self, cache_by_key: dict[bytes, torch.Tensor]) -> None:
        self._cache = cache_by_key

    @staticmethod
    def _key(input_ids: torch.Tensor) -> bytes:
        return input_ids.detach().cpu().to(torch.int64).contiguous().numpy().tobytes()

    @classmethod
    def from_mmap_and_prompts(
        cls, mmap_table: MmapPleTable, tokenized_prompts: Sequence[torch.Tensor]
    ) -> "PreextractedPleProvider":
        cache: dict[bytes, torch.Tensor] = {}
        for ids in tokenized_prompts:
            k = cls._key(ids)
            if k not in cache:
                cache[k] = mmap_table.lookup(ids, device="cpu").contiguous()
        return cls(cache)

    def lookup(self, input_ids: torch.Tensor, device: torch.device | str | None = None) -> torch.Tensor:
        k = self._key(input_ids)
        if k not in self._cache:
            raise KeyError(
                "Prompt not in PreextractedPleProvider cache. Use `ple_mode='mmap'` to support arbitrary prompts."
            )
        target_dev = input_ids.device if device is None else torch.device(device)
        return self._cache[k].to(target_dev, non_blocking=True)

    def save(self, path: str) -> None:
        torch.save(self._cache, path)

    @classmethod
    def load(cls, path: str) -> "PreextractedPleProvider":
        return cls(torch.load(path, map_location="cpu", weights_only=True))


class InMemoryPleProvider:
    """Simple in-memory PLE provider for unit tests and tiny synthetic models."""

    def __init__(
        self,
        hash_spec: PleHashSpec,
        weight: torch.Tensor,
        weight_scale: float = 1.0,
    ) -> None:
        self.hash_spec = hash_spec
        self.weight = weight
        self.weight_scale = float(weight_scale)

    def lookup(self, input_ids: torch.Tensor, device: torch.device | str | None = None) -> torch.Tensor:
        ngram_ids = self.hash_spec.compute_ngram_ids(input_ids.to(self.weight.device))
        rows = torch.nn.functional.embedding(ngram_ids, self.weight)
        if rows.dtype in (torch.float8_e4m3fn, torch.float8_e5m2):
            rows = rows.to(torch.bfloat16) * self.weight_scale
        out = rows.flatten(-2)
        target_dev = input_ids.device if device is None else torch.device(device)
        return out.to(target_dev)


In [ ]:
%%writefile qwen38_jlens/kernels.py
"""Best-in-class GPU kernels with hardware auto-dispatch and verified fallbacks.

Kernel stack:
  1. Gated DeltaNet (36/48 layers):
       - SM90/100/103/120/121 (Colab G4 Blackwell, H100, B200): FlashQLA v0.1.3 (TileLang fused fwd + bwd)
       - SM80 (Colab A100 40G/80G): flash-linear-attention (FLA Triton fused chunk fwd + bwd)
       - Universal fallback: exact chunked PyTorch Gated Delta Rule (`chunk_size=64`)
  2. FP8 128x128 Block-Wise Weight Dequantization (48 MoE layers):
       - Custom fused Triton 2D-block kernel (single HBM pass, zero temp allocations, ~1.6+ TB/s)
       - Vectorized PyTorch fallback for CPU / non-Triton environments
  3. Fused SwiGLU Backward + Pack `[dG | dU]` (48 MoE layers):
       - Custom Triton kernel reading L2-resident `[S, H]` gate/up and writing `[S, K, 2*H]` directly
       - Vectorized PyTorch fallback
  4. Grouped MoE GEMM (48 MoE layers):
       - `torch.nn.functional.grouped_mm` (PyTorch >= 2.10) or `torch._grouped_mm` (PyTorch >= 2.9 SM80+)
       - Clean per-expert `torch.mm` fallback for CPU / older PyTorch
  5. Causal Conv1D (36 GDN layers):
       - `causal_conv1d.causal_conv1d_fn` when installed, else depthwise `F.conv1d` + `F.silu`
"""

from __future__ import annotations

import logging
import math
from typing import Any

import torch
import torch.nn.functional as F

logger = logging.getLogger(__name__)

# Optional Triton kernels
try:
    import triton
    import triton.language as tl

    _HAS_TRITON = True
except Exception:
    triton = None  # type: ignore[assignment]
    tl = None  # type: ignore[assignment]
    _HAS_TRITON = False


# --------------------------------------------------------------------------- #
# 1. Custom Fused Triton Kernel: 128x128 Block-Wise FP8 -> BF16 Dequantization
# --------------------------------------------------------------------------- #
if _HAS_TRITON:

    @triton.jit
    def _dequant_fp8_block_kernel(
        w_ptr,
        s_ptr,
        out_ptr,
        E,
        M,
        N,
        stride_we,
        stride_wm,
        stride_wn,
        stride_se,
        stride_sm,
        stride_sn,
        stride_oe,
        stride_om,
        stride_on,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
    ):
        pid_e = tl.program_id(0)
        pid_m = tl.program_id(1)
        pid_n = tl.program_id(2)

        # Single scalar scale for this (expert, m_block, n_block) tile
        scale = tl.load(s_ptr + pid_e * stride_se + pid_m * stride_sm + pid_n * stride_sn).to(tl.float32)

        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

        w_ptrs = w_ptr + pid_e * stride_we + offs_m[:, None] * stride_wm + offs_n[None, :] * stride_wn
        o_ptrs = out_ptr + pid_e * stride_oe + offs_m[:, None] * stride_om + offs_n[None, :] * stride_on

        w_fp8 = tl.load(w_ptrs, mask=mask, other=0.0)
        w_bf16 = (w_fp8.to(tl.float32) * scale).to(tl.bfloat16)
        tl.store(o_ptrs, w_bf16, mask=mask)

    @triton.jit
    def _swiglu_vjp_pack_kernel(
        da_ptr,      # [S, K, H]
        da_dg_ptr,   # [S, H]
        da_du_ptr,   # [S, H]
        out_ptr,     # [S, K, 2*H]
        S,
        K,
        H,
        stride_da_s,
        stride_da_k,
        stride_da_h,
        stride_g_s,
        stride_g_h,
        stride_o_s,
        stride_o_k,
        stride_o_h,
        BLOCK_H: tl.constexpr,
    ):
        pid_s = tl.program_id(0)
        pid_k = tl.program_id(1)
        pid_h = tl.program_id(2)

        offs_h = pid_h * BLOCK_H + tl.arange(0, BLOCK_H)
        mask = offs_h < H

        da = tl.load(
            da_ptr + pid_s * stride_da_s + pid_k * stride_da_k + offs_h * stride_da_h,
            mask=mask,
            other=0.0,
        ).to(tl.float32)
        dg_coeff = tl.load(da_dg_ptr + pid_s * stride_g_s + offs_h * stride_g_h, mask=mask, other=0.0).to(tl.float32)
        du_coeff = tl.load(da_du_ptr + pid_s * stride_g_s + offs_h * stride_g_h, mask=mask, other=0.0).to(tl.float32)

        out_base = out_ptr + pid_s * stride_o_s + pid_k * stride_o_k
        tl.store(out_base + offs_h * stride_o_h, (da * dg_coeff).to(tl.bfloat16), mask=mask)
        tl.store(out_base + (H + offs_h) * stride_o_h, (da * du_coeff).to(tl.bfloat16), mask=mask)


def dequantize_fp8_blockwise(
    weight: torch.Tensor,
    scale_inv: torch.Tensor,
    block_size: tuple[int, int] = (128, 128),
    out_dtype: torch.dtype = torch.bfloat16,
    use_triton: bool = True,
) -> torch.Tensor:
    """Dequantize `[E, M, N]` FP8 E4M3 weights with `[E, M//bm, N//bn]` block scales.

    Uses a single-pass Triton 2D-block kernel on CUDA when available (~1.6+ TB/s, zero temp buffers).
    """
    if weight.element_size() > 1:
        return weight.to(out_dtype)

    bm, bn = block_size
    if weight.ndim == 2:
        return dequantize_fp8_blockwise(
            weight.unsqueeze(0), scale_inv.unsqueeze(0), block_size=block_size, out_dtype=out_dtype, use_triton=use_triton
        ).squeeze(0)

    E, M, N = weight.shape
    if (
        use_triton
        and _HAS_TRITON
        and weight.is_cuda
        and out_dtype == torch.bfloat16
        and bm == 128
        and bn == 128
        and M % 128 == 0
        and N % 128 == 0
    ):
        w_c = weight.contiguous()
        s_c = scale_inv.contiguous()
        out = torch.empty((E, M, N), device=weight.device, dtype=torch.bfloat16)
        grid = (E, M // 128, N // 128)
        _dequant_fp8_block_kernel[grid](
            w_c,
            s_c,
            out,
            E,
            M,
            N,
            w_c.stride(0),
            w_c.stride(1),
            w_c.stride(2),
            s_c.stride(0),
            s_c.stride(1),
            s_c.stride(2),
            out.stride(0),
            out.stride(1),
            out.stride(2),
            BLOCK_M=128,
            BLOCK_N=128,
            num_warps=8,
        )
        return out

    # Universal PyTorch exact fallback (handles arbitrary shapes & CPU testing)
    sm, sn = scale_inv.shape[-2], scale_inv.shape[-1]
    if M % sm == 0 and N % sn == 0:
        b_m = M // sm
        b_n = N // sn
        w_f = weight.to(torch.float32).reshape(E, sm, b_m, sn, b_n)
        s_f = scale_inv.to(torch.float32).reshape(E, sm, 1, sn, 1)
        return (w_f * s_f).reshape(E, M, N).to(out_dtype)

    # Non-divisible edge case
    s_rep = scale_inv.to(torch.float32).repeat_interleave(bm, dim=-2).repeat_interleave(bn, dim=-1)[..., :M, :N]
    return (weight.to(torch.float32) * s_rep).to(out_dtype)


def swiglu_vjp_pack(
    dA: torch.Tensor,      # [S, K, H]
    da_dg: torch.Tensor,   # [S, H]
    da_du: torch.Tensor,   # [S, H]
    use_triton: bool = True,
) -> torch.Tensor:
    """Fused compute & pack `dGU = cat([dA * da_dg, dA * da_du], dim=-1)` -> `[S, K, 2*H]`."""
    S, K, H = dA.shape
    if use_triton and _HAS_TRITON and dA.is_cuda and dA.dtype == torch.bfloat16:
        dA_c = dA.contiguous()
        dg_c = da_dg.contiguous()
        du_c = da_du.contiguous()
        out = torch.empty((S, K, 2 * H), device=dA.device, dtype=torch.bfloat16)
        BLOCK_H = 128 if H >= 128 else 64
        grid = (S, K, triton.cdiv(H, BLOCK_H))
        _swiglu_vjp_pack_kernel[grid](
            dA_c,
            dg_c,
            du_c,
            out,
            S,
            K,
            H,
            dA_c.stride(0),
            dA_c.stride(1),
            dA_c.stride(2),
            dg_c.stride(0),
            dg_c.stride(1),
            out.stride(0),
            out.stride(1),
            out.stride(2),
            BLOCK_H=BLOCK_H,
            num_warps=4,
        )
        return out

    dG = dA * da_dg.unsqueeze(1)
    dU = dA * da_du.unsqueeze(1)
    return torch.cat([dG, dU], dim=-1)


# --------------------------------------------------------------------------- #
# 2. Grouped MoE GEMM Dispatcher (CUTLASS/cuBLAS grouped_mm + safe fallback)
# --------------------------------------------------------------------------- #
def _has_native_cuda_grouped_mm(device: torch.device, dtype: torch.dtype) -> bool:
    if device.type != "cuda" or dtype != torch.bfloat16:
        return False
    cc = torch.cuda.get_device_capability(device)
    if hasattr(torch.nn.functional, "grouped_mm"):
        return cc >= (8, 0)
    if hasattr(torch, "_grouped_mm"):
        # torch>=2.9 supports SM80+; torch 2.8 supports SM90
        parts = torch.__version__.split("+")[0].split(".")
        ver = (int(parts[0]), int(parts[1]))
        return cc >= (8, 0) if ver >= (2, 9) else cc[0] >= 9
    return False


def grouped_mm(
    mat_a: torch.Tensor,    # [M_total, K_in]
    mat_b: torch.Tensor,    # [E, K_in, N_out]
    offs: torch.Tensor,     # [E] int32 cumulative end offsets
    offs_list: list[int] | None = None,
) -> torch.Tensor:
    """Fast grouped GEMM: `out[start_e:end_e] = mat_a[start_e:end_e] @ mat_b[e]`.

    Dispatches to CUTLASS/cuBLAS `torch.nn.functional.grouped_mm` / `torch._grouped_mm`
    on CUDA SM80+ (A100, H100, Blackwell G4) when inputs are BF16, with automatic safe
    fallback if a driver/torch build rejects unaligned `B=1` forward offsets.
    """
    if _has_native_cuda_grouped_mm(mat_a.device, mat_a.dtype) and mat_b.dtype == mat_a.dtype:
        offs_i32 = offs.to(dtype=torch.int32, device=mat_a.device)
        try:
            if hasattr(torch.nn.functional, "grouped_mm"):
                return torch.nn.functional.grouped_mm(mat_a, mat_b, offs=offs_i32)
            return torch._grouped_mm(mat_a, mat_b, offs=offs_i32)
        except RuntimeError as exc:
            logger.debug("Native grouped_mm declined shape %s (%s); using per-expert mm fallback.", tuple(mat_a.shape), exc)

    # Clean fallback (CPU unit tests / fp32 verification / unaligned edge cases)
    ends = offs.tolist() if offs_list is None else offs_list
    out = torch.empty((mat_a.size(0), mat_b.size(2)), device=mat_a.device, dtype=mat_a.dtype)
    start = 0
    for e, end in enumerate(ends):
        if end > start:
            torch.mm(mat_a[start:end], mat_b[e], out=out[start:end])
        start = end
    return out


# --------------------------------------------------------------------------- #
# 3. Gated DeltaNet (GDN) Kernel Dispatcher: FlashQLA > FLA Triton > PyTorch
# --------------------------------------------------------------------------- #
def l2norm_torch(x: torch.Tensor, eps: float = 1e-6) -> tuple[torch.Tensor, torch.Tensor]:
    """L2 normalization matching FLA/Qwen4Exp: returns `(x_norm, rstd)` in fp32."""
    xf = x.float()
    rstd = torch.rsqrt((xf * xf).sum(dim=-1, keepdim=True) + eps)
    return (xf * rstd).to(x.dtype), rstd


def l2norm_bwd_torch(x_norm: torch.Tensor, rstd: torch.Tensor, dy: torch.Tensor) -> torch.Tensor:
    """Exact VJP of `y = x * rstd` given saved `x_norm = x * rstd` and `rstd`."""
    yf = x_norm.float()
    dyf = dy.float()
    dot = (dyf * yf).sum(dim=-1, keepdim=True)
    return (rstd * (dyf - yf * dot)).to(dy.dtype)


def torch_chunk_gated_delta_rule(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    g: torch.Tensor,
    beta: torch.Tensor,
    chunk_size: int = 64,
    use_qk_l2norm_in_kernel: bool = True,
) -> torch.Tensor:
    """Exact reference chunked Gated Delta Rule matching `modeling_qwen4_exp.py`."""
    initial_dtype = query.dtype
    batch_size, sequence_length, num_k_heads, k_head_dim = key.shape
    num_v_heads, v_head_dim = value.shape[-2:]

    if num_k_heads != num_v_heads:
        rep = num_v_heads // num_k_heads
        query = query.repeat_interleave(rep, dim=2)
        key = key.repeat_interleave(rep, dim=2)

    q, k, v, b, decay = [
        x.transpose(1, 2).to(torch.float32).contiguous() for x in (query, key, value, beta, g)
    ]
    if use_qk_l2norm_in_kernel:
        q = q * torch.rsqrt((q * q).sum(dim=-1, keepdim=True) + 1e-6)
        k = k * torch.rsqrt((k * k).sum(dim=-1, keepdim=True) + 1e-6)
    q = q * (k_head_dim ** -0.5)

    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    if pad_size:
        q = F.pad(q, (0, 0, 0, pad_size))
        k = F.pad(k, (0, 0, 0, pad_size))
        v = F.pad(v, (0, 0, 0, pad_size))
        b = F.pad(b, (0, pad_size))
        decay = F.pad(decay, (0, pad_size))

    num_chunks = (sequence_length + pad_size) // chunk_size
    v_beta = v * b.unsqueeze(-1)
    k_beta = k * b.unsqueeze(-1)

    q, k, k_beta, v_beta = [
        x.reshape(batch_size, num_v_heads, num_chunks, chunk_size, x.shape[-1])
        for x in (q, k, k_beta, v_beta)
    ]
    decay = decay.reshape(batch_size, num_v_heads, num_chunks, chunk_size)

    strictly_upper = torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=q.device).triu(1)
    cum_decay = decay.cumsum(dim=3)
    pairwise = (cum_decay.unsqueeze(4) - cum_decay.unsqueeze(3)).masked_fill(strictly_upper, float("-inf")).exp()

    ut_system = (k_beta @ k.transpose(-1, -2)) * pairwise
    intra_attn = (q @ k.transpose(-1, -2)) * pairwise
    decayed_k_beta = k_beta * cum_decay.exp().unsqueeze(-1)

    new_values = torch.linalg.solve_triangular(ut_system, v_beta, upper=False, unitriangular=True)
    k_cumdecay = torch.linalg.solve_triangular(ut_system, decayed_k_beta, upper=False, unitriangular=True)

    state = torch.zeros((batch_size, num_v_heads, k_head_dim, v_head_dim), dtype=torch.float32, device=q.device)
    q_dec = q * cum_decay.exp().unsqueeze(-1)
    k_dec = k * (cum_decay[..., -1:] - cum_decay).exp().unsqueeze(-1)
    chunk_decay = cum_decay[..., -1].exp()[..., None, None]

    out_chunks = []
    for i in range(num_chunks):
        v_new = new_values[:, :, i] - k_cumdecay[:, :, i] @ state
        inter = q_dec[:, :, i] @ state
        out_chunks.append(inter + intra_attn[:, :, i] @ v_new)
        state = state * chunk_decay[:, :, i] + k_dec[:, :, i].transpose(-1, -2) @ v_new

    core_out = torch.cat(out_chunks, dim=2)[:, :, :sequence_length]
    return core_out.transpose(1, 2).contiguous().to(initial_dtype)


class GatedDeltaRuleBackend:
    """Unified GDN forward + batched-cotangent backward backend with startup parity verification.

    Automatically probes and validates:
      1. FlashQLA v0.1.3 (TileLang SM90/100/120 fused fwd & bwd — direct low-level or high-level API)
      2. FLA Triton (`flash-linear-attention` SM80+ fused chunk fwd & bwd — direct or high-level API)
      3. Pure PyTorch chunked fallback (universal, CPU-compatible)
    """

    def __init__(self, preferred: str = "auto", verify_on_init: bool = True) -> None:
        self.name = "torch"
        self._flash_qla: dict[str, Any] | None = None
        self._fla: dict[str, Any] | None = None
        self._use_direct = True
        self._init_backend(preferred, verify_on_init=verify_on_init)

    def _init_backend(self, preferred: str, verify_on_init: bool = True) -> None:
        if not torch.cuda.is_available() or preferred == "torch":
            self.name = "torch"
            return

        cc = torch.cuda.get_device_capability(0)
        order = (
            ["flash_qla", "fla", "torch"]
            if preferred in ("auto", "flash_qla")
            else ["fla", "flash_qla", "torch"]
        )

        for cand in order:
            if cand == "flash_qla" and cc in ((9, 0), (10, 0), (10, 3), (12, 0), (12, 1)):
                try:
                    import flash_qla
                    from flash_qla.ops.gated_delta_rule.chunk import (
                        chunk_gated_delta_rule_fwd,
                        chunk_gated_delta_rule_bwd,
                    )
                    from flash_qla.utils import l2norm_fwd, l2norm_bwd

                    self._flash_qla = {
                        "high": flash_qla.chunk_gated_delta_rule,
                        "fwd": chunk_gated_delta_rule_fwd,
                        "bwd": chunk_gated_delta_rule_bwd,
                        "l2_fwd": l2norm_fwd,
                        "l2_bwd": l2norm_bwd,
                    }
                    self.name = "flash_qla"
                    self._use_direct = True
                    if not verify_on_init or self._self_test_gva():
                        logger.info("GDN backend verified: FlashQLA direct (SM%d%d TileLang)", cc[0], cc[1])
                        return
                    self._use_direct = False
                    if self._self_test_gva():
                        logger.info("GDN backend verified: FlashQLA high-level (SM%d%d TileLang)", cc[0], cc[1])
                        return
                except Exception as exc:
                    logger.warning("FlashQLA unavailable (%s); trying next GDN backend.", exc)

            if cand == "fla" and cc >= (8, 0):
                try:
                    from fla.ops.gated_delta_rule import chunk_gated_delta_rule
                    from fla.ops.gated_delta_rule.chunk import (
                        chunk_gated_delta_rule_fwd,
                        chunk_gated_delta_rule_bwd,
                    )
                    from fla.modules.norm.l2norm import l2norm_fwd, l2norm_bwd

                    self._fla = {
                        "high": chunk_gated_delta_rule,
                        "fwd": chunk_gated_delta_rule_fwd,
                        "bwd": chunk_gated_delta_rule_bwd,
                        "l2_fwd": l2norm_fwd,
                        "l2_bwd": l2norm_bwd,
                    }
                    self.name = "fla"
                    self._use_direct = True
                    if not verify_on_init or self._self_test_gva():
                        logger.info("GDN backend verified: FLA Triton direct (SM%d%d)", cc[0], cc[1])
                        return
                    self._use_direct = False
                    if self._self_test_gva():
                        logger.info("GDN backend verified: FLA Triton high-level (SM%d%d)", cc[0], cc[1])
                        return
                except Exception as exc:
                    logger.warning("FLA unavailable (%s); falling back to PyTorch chunked GDN.", exc)

        self.name = "torch"

    def _self_test_gva(self) -> bool:
        """1-ms numerical parity smoke test on Qwen3.8's exact 3:1 GVA head ratio & D=128."""
        try:
            dev = torch.device("cuda:0")
            q = torch.randn(1, 64, 2, 128, device=dev, dtype=torch.bfloat16) * 0.2
            k = torch.randn(1, 64, 2, 128, device=dev, dtype=torch.bfloat16) * 0.2
            v = torch.randn(1, 64, 6, 128, device=dev, dtype=torch.bfloat16) * 0.2
            g = -F.softplus(torch.randn(1, 64, 6, device=dev, dtype=torch.float32))
            beta = torch.sigmoid(torch.randn(1, 64, 6, device=dev, dtype=torch.bfloat16))
            do = torch.randn(2, 64, 6, 128, device=dev, dtype=torch.bfloat16) * 0.2

            o_fast, saved = self.forward_single(q, k, v, g, beta)
            dq, dk, dv, dg, db = self.backward_batched(saved, do)

            with torch.enable_grad():
                q_r = q.expand(2, -1, -1, -1).clone().requires_grad_(True)
                k_r = k.expand(2, -1, -1, -1).clone().requires_grad_(True)
                v_r = v.expand(2, -1, -1, -1).clone().requires_grad_(True)
                g_r = g.expand(2, -1, -1).clone().requires_grad_(True)
                b_r = beta.expand(2, -1, -1).clone().requires_grad_(True)
                o_r = torch_chunk_gated_delta_rule(q_r, k_r, v_r, g_r, b_r, use_qk_l2norm_in_kernel=True)
                dq_r, dk_r, dv_r, dg_r, db_r = torch.autograd.grad(o_r, (q_r, k_r, v_r, g_r, b_r), grad_outputs=do)

            ok = (
                (o_fast.float() - o_r[:1].float()).abs().max().item() < 5e-2
                and (dq.float() - dq_r.float()).abs().max().item() < 1e-1
                and (dv.float() - dv_r.float()).abs().max().item() < 1e-1
                and torch.isfinite(dg).all().item()
                and torch.isfinite(db).all().item()
            )
            return bool(ok)
        except Exception as exc:
            logger.warning("GDN kernel self-test failed (%s direct=%s): %s", self.name, self._use_direct, exc)
            return False

    def forward_single(
        self,
        q: torch.Tensor,     # [1, T, H_k, D]
        k: torch.Tensor,     # [1, T, H_k, D]
        v: torch.Tensor,     # [1, T, H_v, D]
        g: torch.Tensor,     # [1, T, H_v] float32 log-decay <= 0
        beta: torch.Tensor,  # [1, T, H_v]
    ) -> tuple[torch.Tensor, dict[str, Any]]:
        """Run ONE `B=1` forward pass and return `(core_attn_out, saved_ctx)` for fast multi-cotangent bwd."""
        scale = k.shape[-1] ** -0.5

        if self.name == "flash_qla" and q.is_cuda and q.dtype in (torch.bfloat16, torch.float16):
            api = self._flash_qla
            assert api is not None
            if self._use_direct:
                q_n, q_rstd = api["l2_fwd"](q.contiguous())
                k_n, k_rstd = api["l2_fwd"](k.contiguous())
                v_c = v.contiguous()
                g_c = g.float().contiguous()
                b_c = beta.to(q.dtype).contiguous()
                g_cum, A, o, *_ = api["fwd"](
                    q=q_n,
                    k=k_n,
                    v=v_c,
                    g=g_c,
                    beta=b_c,
                    scale=scale,
                    initial_state=None,
                    output_final_state=False,
                    auto_cp=False,
                    enable_fwd_cp_cache=False,
                    is_train=True,
                )
                return o.to(q.dtype), {
                    "mode": "flash_qla_direct",
                    "q_n": q_n, "q_rstd": q_rstd,
                    "k_n": k_n, "k_rstd": k_rstd,
                    "v": v_c, "g_cum": g_cum, "beta": b_c, "A": A, "scale": scale,
                }
            o, _ = api["high"](
                q.contiguous(), k.contiguous(), v.contiguous(),
                g=g.float().contiguous(), beta=beta.to(q.dtype).contiguous(),
                scale=scale, output_final_state=False, use_qk_l2norm_in_kernel=True, auto_cp=False,
            )
            return o.to(q.dtype), {"mode": "flash_qla_high", "q": q, "k": k, "v": v, "g": g, "beta": beta, "scale": scale}

        if self.name == "fla" and q.is_cuda and q.dtype in (torch.bfloat16, torch.float16):
            api = self._fla
            assert api is not None
            if self._use_direct:
                q_n, q_rstd = api["l2_fwd"](q.contiguous())
                k_n, k_rstd = api["l2_fwd"](k.contiguous())
                v_c = v.contiguous()
                g_c = g.float().contiguous()
                b_c = beta.to(q.dtype).contiguous()
                g_cum, o, A, *_ = api["fwd"](
                    q=q_n,
                    k=k_n,
                    v=v_c,
                    g=g_c,
                    beta=b_c,
                    scale=scale,
                    initial_state=None,
                    output_final_state=False,
                )
                return o.to(q.dtype), {
                    "mode": "fla_direct",
                    "q_n": q_n, "q_rstd": q_rstd,
                    "k_n": k_n, "k_rstd": k_rstd,
                    "v": v_c, "g_cum": g_cum, "beta": b_c, "A": A, "scale": scale,
                }
            o, _ = api["high"](
                q.contiguous(), k.contiguous(), v.contiguous(),
                g=g.float().contiguous(), beta=beta.to(q.dtype).contiguous(),
                scale=scale, output_final_state=False, use_qk_l2norm_in_kernel=True,
            )
            return o.to(q.dtype), {"mode": "fla_high", "q": q, "k": k, "v": v, "g": g, "beta": beta, "scale": scale}

        # Universal PyTorch exact path
        o = torch_chunk_gated_delta_rule(q, k, v, g, beta, use_qk_l2norm_in_kernel=True)
        return o, {"mode": "torch", "q": q, "k": k, "v": v, "g": g, "beta": beta}

    def backward_batched(
        self,
        saved: dict[str, Any],
        do: torch.Tensor,  # [K_sub, T, H_v, D]
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Compute `(dq, dk, dv, dg, dbeta)` for `K_sub` cotangents at once reusing `B=1` forward state."""
        K_sub = do.shape[0]
        mode = saved["mode"]

        if mode == "flash_qla_direct":
            api = self._flash_qla
            assert api is not None
            q_e = saved["q_n"].expand(K_sub, -1, -1, -1).contiguous()
            k_e = saved["k_n"].expand(K_sub, -1, -1, -1).contiguous()
            v_e = saved["v"].expand(K_sub, -1, -1, -1).contiguous()
            g_e = saved["g_cum"].expand(K_sub, -1, -1).contiguous()
            b_e = saved["beta"].expand(K_sub, -1, -1).contiguous()
            A_e = saved["A"].expand(K_sub, -1, -1, -1).contiguous()
            res = api["bwd"](
                q=q_e,
                k=k_e,
                v=v_e,
                g=g_e,
                beta=b_e,
                A=A_e,
                do=do.to(q_e.dtype).contiguous(),
                dht=None,
                scale=saved["scale"],
                initial_state=None,
                auto_cp=False,
            )
            dq, dk, dv, db, dg = res[0], res[1], res[2], res[3], res[4]
            q_rstd_e = saved["q_rstd"].expand(K_sub, -1, -1, -1).contiguous()
            k_rstd_e = saved["k_rstd"].expand(K_sub, -1, -1, -1).contiguous()
            dq = api["l2_bwd"](q_e, q_rstd_e, dq)
            dk = api["l2_bwd"](k_e, k_rstd_e, dk)
            return dq.to(do.dtype), dk.to(do.dtype), dv.to(do.dtype), dg.float(), db.to(do.dtype)

        if mode == "fla_direct":
            api = self._fla
            assert api is not None
            q_e = saved["q_n"].expand(K_sub, -1, -1, -1).contiguous()
            k_e = saved["k_n"].expand(K_sub, -1, -1, -1).contiguous()
            v_e = saved["v"].expand(K_sub, -1, -1, -1).contiguous()
            g_e = saved["g_cum"].expand(K_sub, -1, -1).contiguous()
            b_e = saved["beta"].expand(K_sub, -1, -1).contiguous()
            A_e = saved["A"].expand(K_sub, -1, -1, -1).contiguous()
            res = api["bwd"](
                q=q_e,
                k=k_e,
                v=v_e,
                g=g_e,
                beta=b_e,
                A=A_e,
                scale=saved["scale"],
                initial_state=None,
                do=do.to(q_e.dtype).contiguous(),
                dht=None,
            )
            dq, dk, dv, db, dg = res[0], res[1], res[2], res[3], res[4]
            q_rstd_e = saved["q_rstd"].expand(K_sub, -1, -1, -1).contiguous()
            k_rstd_e = saved["k_rstd"].expand(K_sub, -1, -1, -1).contiguous()
            dq = api["l2_bwd"](q_e, q_rstd_e, dq)
            dk = api["l2_bwd"](k_e, k_rstd_e, dk)
            return dq.to(do.dtype), dk.to(do.dtype), dv.to(do.dtype), dg.float(), db.to(do.dtype)

        if mode in ("flash_qla_high", "fla_high"):
            api = self._flash_qla if mode == "flash_qla_high" else self._fla
            assert api is not None
            extra = {"auto_cp": False} if mode == "flash_qla_high" else {}
            with torch.enable_grad():
                q_e = saved["q"].detach().expand(K_sub, -1, -1, -1).contiguous().requires_grad_(True)
                k_e = saved["k"].detach().expand(K_sub, -1, -1, -1).contiguous().requires_grad_(True)
                v_e = saved["v"].detach().expand(K_sub, -1, -1, -1).contiguous().requires_grad_(True)
                g_e = saved["g"].detach().float().expand(K_sub, -1, -1).contiguous().requires_grad_(True)
                b_e = saved["beta"].detach().to(q_e.dtype).expand(K_sub, -1, -1).contiguous().requires_grad_(True)
                o_e, _ = api["high"](
                    q_e, k_e, v_e, g=g_e, beta=b_e, scale=saved["scale"],
                    output_final_state=False, use_qk_l2norm_in_kernel=True, **extra,
                )
                dq, dk, dv, dg, db = torch.autograd.grad(
                    outputs=o_e, inputs=(q_e, k_e, v_e, g_e, b_e), grad_outputs=do.to(o_e.dtype),
                )
            return dq.to(do.dtype), dk.to(do.dtype), dv.to(do.dtype), dg.float(), db.to(do.dtype)

        # Pure PyTorch exact chunked backward
        with torch.enable_grad():
            q_e = saved["q"].detach().expand(K_sub, -1, -1, -1).clone().requires_grad_(True)
            k_e = saved["k"].detach().expand(K_sub, -1, -1, -1).clone().requires_grad_(True)
            v_e = saved["v"].detach().expand(K_sub, -1, -1, -1).clone().requires_grad_(True)
            g_e = saved["g"].detach().expand(K_sub, -1, -1).clone().requires_grad_(True)
            b_e = saved["beta"].detach().expand(K_sub, -1, -1).clone().requires_grad_(True)
            o_e = torch_chunk_gated_delta_rule(q_e, k_e, v_e, g_e, b_e, use_qk_l2norm_in_kernel=True)
            dq, dk, dv, dg, db = torch.autograd.grad(
                outputs=o_e,
                inputs=(q_e, k_e, v_e, g_e, b_e),
                grad_outputs=do.to(o_e.dtype),
            )
        return dq, dk, dv, dg.float(), db


# --------------------------------------------------------------------------- #
# 4. Causal Conv1D Forward + Exact Backward
# --------------------------------------------------------------------------- #
try:
    from causal_conv1d import causal_conv1d_fn as _cuda_causal_conv1d_fn
except Exception:
    _cuda_causal_conv1d_fn = None


def causal_conv1d_silu_fwd(
    x: torch.Tensor,        # [B, C, T]
    weight: torch.Tensor,   # [C, K_conv]
) -> tuple[torch.Tensor, torch.Tensor]:
    """Exact depthwise causal conv1d + SiLU; returns `(out_silu, pre_silu)`."""
    C, K_conv = weight.shape
    T = x.shape[-1]
    pre = F.conv1d(x.to(weight.dtype), weight.unsqueeze(1), bias=None, padding=K_conv - 1, groups=C)[:, :, :T]
    out = F.silu(pre).to(x.dtype)
    return out, pre


def causal_conv1d_silu_bwd(
    d_out: torch.Tensor,     # [K_sub, C, T]
    pre_silu: torch.Tensor,  # [1, C, T]
    weight: torch.Tensor,    # [C, K_conv]
) -> torch.Tensor:
    """Exact VJP of depthwise causal conv1d + SiLU w.r.t. input `x` -> `[K_sub, C, T]`."""
    C, K_conv = weight.shape
    T = d_out.shape[-1]
    # d_silu(u) = sig(u) * (1 + u * (1 - sig(u)))
    u = pre_silu.float()
    sig = torch.sigmoid(u)
    d_silu = (sig * (1.0 + u * (1.0 - sig))).to(d_out.dtype)
    d_pre = (d_out * d_silu).to(weight.dtype)
    # Forward was F.conv1d(x, w, padding=K_conv-1)[:, :, :T] == F.conv1d(F.pad(x, (K_conv-1, 0)), w)
    # Exact adjoint w.r.t. x: conv_transpose1d(d_pre, w)[:, :, K_conv-1 : K_conv-1 + T]
    pad_l = K_conv - 1
    d_x = F.conv_transpose1d(d_pre, weight.unsqueeze(1), bias=None, padding=0, groups=C)
    return d_x[:, :, pad_l : pad_l + T].to(d_out.dtype)


In [ ]:
%%writefile qwen38_jlens/model_ops.py
"""Exact Qwen3.8-Flash-Next (qwen4_exp) forward and batched-cotangent analytical VJP operators.

Every operator executes:
  1. Forward ONCE at batch size `B = 1` (saving minimal `B = 1` activations, ~18 MB/layer total).
  2. Exact VJP across `K` cotangents (`G` of shape `[K, T, D_hc]`, up to `K = 2560`) reusing
     the single `B = 1` forward state and single FP8->BF16 active-expert dequantization!

Mathematical parity with HuggingFace `transformers/models/qwen4_exp/modeling_qwen4_exp.py`:
  - `Qwen4ExpTextRMSNorm`: zero-centered `(1.0 + weight)` with optional `group_size=hidden_size`.
  - `Qwen4ExpTextRMSNormGated`: direct `weight` (NOT `1 + weight`) and configurable `sigmoid` / `silu`.
  - `Qwen4ExpTextGatedResidual`: 4-branch Hyper-Connection with bottleneck rank 320.
  - `Qwen4ExpTextPLELayer`: layer-2 hashed n-gram lookup (constant wrt residual) + sqrt-sign gate + dilated conv1d.
  - `Qwen4ExpTextAttention` (QSA): exact indexer bypass for `T <= 2048` + L2-cached exact `B=1` attention VJP.
  - `Qwen4ExpTextGatedDeltaNet` (GDN): FlashQLA (SM90/100/120) / FLA Triton (SM80) / PyTorch fallback.
  - `Qwen4ExpTextSparseMoeBlock`: exact router softmax+topk+norm VJP + 1x active-expert FP8 dequant + `grouped_mm` VJP + shared expert VJP.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any

import torch
import torch.nn.functional as F

from qwen38_jlens.kernels import (
    GatedDeltaRuleBackend,
    causal_conv1d_silu_bwd,
    causal_conv1d_silu_fwd,
    dequantize_fp8_blockwise,
    grouped_mm,
    swiglu_vjp_pack,
)


# --------------------------------------------------------------------------- #
# Weight containers (pure flat tensor dataclasses — zero nn.Module overhead)
# --------------------------------------------------------------------------- #
@dataclass
class GatedResidualWeights:
    hc_norm_weight: torch.Tensor          # [hc_count * D]
    input_mix_weight_down: torch.Tensor   # [hc_lowrank, hc_count * D]
    input_mix_weight_up: torch.Tensor     # [hc_count * D, hc_lowrank]
    block_inject_weight: torch.Tensor | None  # [hc_count, hc_count * D] or None
    hc_count: int
    hidden_size: int
    eps: float = 1e-6


@dataclass
class PleLayerWeights:
    key_proj: torch.Tensor       # [hc_count * D, ple_embed_dim]
    value_proj: torch.Tensor     # [D, ple_embed_dim]
    norm_key: torch.Tensor       # [hc_count * D]
    norm_query: torch.Tensor     # [hc_count * D]
    norm_conv: torch.Tensor      # [hc_count * D]
    conv1d_weight: torch.Tensor  # [hc_count * D, 1, conv_kernel_size]
    hc_count: int
    hidden_size: int
    ngram_size: int = 3
    eps: float = 1e-6


@dataclass
class GdnLayerWeights:
    in_proj_qkv: torch.Tensor    # [2*key_dim + value_dim, D]
    in_proj_z: torch.Tensor      # [value_dim, D]
    in_proj_b: torch.Tensor      # [num_v_heads, D]
    in_proj_a: torch.Tensor      # [num_v_heads, D]
    conv1d_weight: torch.Tensor  # [conv_dim, conv_kernel_size] (squeezed)
    dt_bias: torch.Tensor        # [num_v_heads]
    A_log: torch.Tensor          # [num_v_heads]
    norm_weight: torch.Tensor    # [head_v_dim]
    out_proj: torch.Tensor       # [D, value_dim]
    num_k_heads: int
    num_v_heads: int
    head_k_dim: int
    head_v_dim: int
    output_gate_type: str = "sigmoid"
    eps: float = 1e-6


@dataclass
class QsaLayerWeights:
    q_proj: torch.Tensor         # [num_q_heads * head_dim * 2, D]
    k_proj: torch.Tensor         # [num_kv_heads * head_dim, D]
    v_proj: torch.Tensor         # [num_kv_heads * head_dim, D]
    o_proj: torch.Tensor         # [D, num_q_heads * head_dim]
    q_norm: torch.Tensor         # [head_dim]
    k_norm: torch.Tensor         # [head_dim]
    num_q_heads: int
    num_kv_heads: int
    head_dim: int
    indexer_budget: int = 2048
    indexer_compress_ratio: int = 4
    eps: float = 1e-6
    # Optional indexer weights (only used when seq_len > indexer_budget)
    index_qk_proj: torch.Tensor | None = None
    index_q_norm: torch.Tensor | None = None
    index_k_norm: torch.Tensor | None = None
    index_n_heads: int = 4
    index_head_dim: int = 128


@dataclass
class MoeDenseWeights:
    router_weight: torch.Tensor        # [num_experts, D]
    shared_gate_proj: torch.Tensor     # [shared_inter, D]
    shared_up_proj: torch.Tensor       # [shared_inter, D]
    shared_down_proj: torch.Tensor     # [D, shared_inter]
    shared_expert_gate: torch.Tensor   # [1, D]
    num_experts: int
    top_k: int
    norm_topk_prob: bool = True
    fp8_block_size: tuple[int, int] = (128, 128)


@dataclass
class MoeExpertPayload:
    """Routed MoE expert tensors for one layer (either FP8+scales or BF16/FP32)."""

    gate_up_proj: torch.Tensor                  # [E, 2*inter, D] fp8 or bf16/fp32
    down_proj: torch.Tensor                     # [E, D, inter]   fp8 or bf16/fp32
    gate_up_scale: torch.Tensor | None = None   # [E, 2*inter//128, D//128] fp32
    down_scale: torch.Tensor | None = None      # [E, D//128, inter//128]   fp32


@dataclass
class DecoderLayerDenseWeights:
    layer_idx: int
    layer_type: str  # "linear_attention" or "indexed_attention"
    attn_hc: GatedResidualWeights
    mlp_hc: GatedResidualWeights
    moe_dense: MoeDenseWeights
    gdn: GdnLayerWeights | None = None
    qsa: QsaLayerWeights | None = None
    ple: PleLayerWeights | None = None


# --------------------------------------------------------------------------- #
# 1. Exact RMSNorm (zero-centered 1+w) & RMSNormGated (direct w)
# --------------------------------------------------------------------------- #
def rmsnorm_fwd(
    x: torch.Tensor,
    weight: torch.Tensor,
    eps: float = 1e-6,
    group_size: int | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Qwen4ExpTextRMSNorm forward: `y = x_hat * (1 + weight)` with optional group_size."""
    orig_shape = x.shape
    xf = x.float()
    wf = 1.0 + weight.float()
    if group_size is not None:
        xf = xf.reshape(*orig_shape[:-1], -1, group_size)
        wf = wf.reshape(-1, group_size)
    rstd = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + eps)
    x_hat = xf * rstd
    out = (x_hat * wf).to(x.dtype)
    if group_size is not None:
        out = out.reshape(orig_shape)
    return out, x_hat, rstd


def rmsnorm_vjp(
    dy: torch.Tensor,      # [K, *orig_shape_no_batch] or [K, T, D]
    x_hat: torch.Tensor,   # [1, ..., (G,) gs] float32
    rstd: torch.Tensor,    # [1, ..., (G,) 1]  float32
    weight: torch.Tensor,  # [D]
    group_size: int | None = None,
) -> torch.Tensor:
    """Exact batched-cotangent VJP of Qwen4ExpTextRMSNorm."""
    orig_dy_shape = dy.shape
    dyf = dy.float()
    wf = 1.0 + weight.float()
    if group_size is not None:
        dyf = dyf.reshape(*orig_dy_shape[:-1], -1, group_size)
        wf = wf.reshape(-1, group_size)
    g = dyf * wf
    dot = (g * x_hat).mean(dim=-1, keepdim=True)
    dx = (rstd * (g - x_hat * dot)).to(dy.dtype)
    if group_size is not None:
        dx = dx.reshape(orig_dy_shape)
    return dx


def rmsnorm_gated_fwd(
    x: torch.Tensor,         # [1, T, H_v, D_v]
    gate: torch.Tensor,      # [1, T, H_v, D_v]
    weight: torch.Tensor,    # [D_v] (NOT 1+weight!)
    activation: str = "sigmoid",
    eps: float = 1e-6,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """Qwen4ExpTextRMSNormGated exact forward."""
    input_dtype = x.dtype
    xf = x.float()
    rstd = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + eps)
    x_hat_f = xf * rstd
    normed = (weight.to(input_dtype) * x_hat_f.to(input_dtype)).float()
    gf = gate.float()
    if activation == "sigmoid":
        act_g = torch.sigmoid(gf)
        d_act_g = act_g * (1.0 - act_g)
    elif activation == "silu":
        sg = torch.sigmoid(gf)
        act_g = gf * sg
        d_act_g = sg * (1.0 + gf * (1.0 - sg))
    else:
        raise ValueError(f"Unsupported RMSNormGated activation: {activation}")
    out = (normed * act_g).to(input_dtype)
    return out, {
        "x_hat_f": x_hat_f,
        "rstd": rstd,
        "normed": normed,
        "act_g": act_g,
        "d_act_g": d_act_g,
        "weight_f": weight.float(),
    }


def rmsnorm_gated_vjp(
    dy: torch.Tensor,  # [K, T, H_v, D_v]
    saved: dict[str, torch.Tensor],
) -> tuple[torch.Tensor, torch.Tensor]:
    """Exact VJP of Qwen4ExpTextRMSNormGated -> `(dx, dgate)`."""
    dyf = dy.float()
    d_gate = (dyf * saved["normed"] * saved["d_act_g"]).to(dy.dtype)
    g = dyf * saved["act_g"] * saved["weight_f"]
    x_hat_f = saved["x_hat_f"]
    dot = (g * x_hat_f).mean(dim=-1, keepdim=True)
    dx = (saved["rstd"] * (g - x_hat_f * dot)).to(dy.dtype)
    return dx, d_gate


# --------------------------------------------------------------------------- #
# 2. Exact Gated Residual (Hyper-Connection) Forward & VJP
# --------------------------------------------------------------------------- #
def gated_residual_fwd(
    hyper_input: torch.Tensor,  # [1, T, hc_count * D]
    w: GatedResidualWeights,
) -> tuple[torch.Tensor, torch.Tensor | None, dict[str, Any]]:
    """Exact forward of `Qwen4ExpTextGatedResidual`."""
    h_norm, x_hat, rstd = rmsnorm_fwd(
        hyper_input, w.hc_norm_weight, eps=w.eps, group_size=w.hidden_size
    )
    u1 = F.linear(h_norm, w.input_mix_weight_down) / w.hc_count
    u1_f = u1.float()
    sig1 = torch.sigmoid(u1_f)
    s1 = (u1_f * sig1).to(h_norm.dtype)
    d_silu_u1 = (sig1 * (1.0 + u1_f * (1.0 - sig1))).to(h_norm.dtype)

    u2 = F.linear(s1, w.input_mix_weight_up)
    m_flat = torch.sigmoid(u2.float())
    m_4 = m_flat.unflatten(-1, (w.hc_count, w.hidden_size))
    h_norm_4 = h_norm.float().unflatten(-1, (w.hc_count, w.hidden_size))
    mixed_input = (m_4 * h_norm_4).mean(dim=-2).to(hyper_input.dtype)

    inj_weights = None
    d_inj_u3 = None
    if w.block_inject_weight is not None:
        u3 = F.linear(h_norm, w.block_inject_weight) / w.hc_count
        sig3 = torch.sigmoid(u3.float())
        inj_weights = (2.0 * sig3).to(hyper_input.dtype)
        d_inj_u3 = (2.0 * sig3 * (1.0 - sig3)).to(hyper_input.dtype)

    saved = {
        "x_hat": x_hat,
        "rstd": rstd,
        "h_norm_4": h_norm_4.to(hyper_input.dtype),
        "m_4": m_4.to(hyper_input.dtype),
        "dm_du2": (m_4 * (1.0 - m_4)).to(hyper_input.dtype),
        "d_silu_u1": d_silu_u1,
        "d_inj_u3": d_inj_u3,
    }
    return mixed_input, inj_weights, saved


def gated_residual_vjp(
    d_mixed: torch.Tensor,          # [K, T, D]
    d_inj: torch.Tensor | None,     # [K, T, hc_count] or None
    w: GatedResidualWeights,
    saved: dict[str, Any],
) -> torch.Tensor:
    """Exact VJP of `Qwen4ExpTextGatedResidual` w.r.t. `hyper_input` -> `[K, T, hc_count * D]`."""
    inv_hc = 1.0 / w.hc_count
    d_prod = (d_mixed * inv_hc).unsqueeze(-2)  # [K, T, 1, D]

    # Direct branch through h_norm * m
    d_h_norm = (d_prod * saved["m_4"]).flatten(-2)

    # Gate branch through m = sigmoid(W_up(silu(W_down(h_norm) / hc)))
    d_u2 = (d_prod * saved["h_norm_4"] * saved["dm_du2"]).flatten(-2)  # [K, T, hc*D]
    d_s1 = torch.matmul(d_u2, w.input_mix_weight_up)  # [K, T, hc_lowrank]
    d_u1 = d_s1 * saved["d_silu_u1"]
    d_h_norm = d_h_norm + torch.matmul(d_u1 * inv_hc, w.input_mix_weight_down)

    # Injection gate branch
    if d_inj is not None and w.block_inject_weight is not None:
        d_u3 = d_inj * saved["d_inj_u3"]  # [K, T, hc_count]
        d_h_norm = d_h_norm + torch.matmul(d_u3 * inv_hc, w.block_inject_weight)

    return rmsnorm_vjp(d_h_norm, saved["x_hat"], saved["rstd"], w.hc_norm_weight, group_size=w.hidden_size)


# --------------------------------------------------------------------------- #
# 3. Exact PLE Layer Forward & VJP (Layer 2 only; n-gram lookup is constant wrt H)
# --------------------------------------------------------------------------- #
def ple_layer_fwd(
    hidden_states: torch.Tensor,    # [1, T, hc_count * D]
    ple_embeddings: torch.Tensor,   # [1, T, ple_embed_dim] (pre-gathered from MmapPleTable)
    w: PleLayerWeights,
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Exact forward of `Qwen4ExpTextPLELayer` returning `(delta_out, saved)`."""
    hc = w.hc_count
    D = w.hidden_size
    key_normed, _, _ = rmsnorm_fwd(
        F.linear(ple_embeddings, w.key_proj), w.norm_key, eps=w.eps, group_size=D
    )
    key_normed_4 = key_normed.unflatten(-1, (hc, D))  # [1, T, hc, D]
    value = F.linear(ple_embeddings, w.value_proj)    # [1, T, D]

    query_normed, q_xhat, q_rstd = rmsnorm_fwd(
        hidden_states, w.norm_query, eps=w.eps, group_size=D
    )
    query_normed_4 = query_normed.unflatten(-1, (hc, D))

    inv_sqrt_d = 1.0 / math.sqrt(D)
    raw_gate = (key_normed_4.float() * query_normed_4.float()).sum(dim=-1, keepdim=True) * inv_sqrt_d
    abs_gate = raw_gate.abs()
    sqrt_gate = abs_gate.clamp_min(1e-6).sqrt() * raw_gate.sign()
    d_sqrt_gate = torch.where(
        abs_gate > 1e-6, 0.5 * torch.rsqrt(abs_gate.clamp_min(1e-6)), torch.zeros_like(abs_gate)
    )
    sig_gate = torch.sigmoid(sqrt_gate)
    d_sig_raw = sig_gate * (1.0 - sig_gate) * d_sqrt_gate  # [1, T, hc, 1]

    gated_value = (sig_gate * value.float().unsqueeze(-2)).to(hidden_states.dtype).flatten(-2)
    gv_normed, c_xhat, c_rstd = rmsnorm_fwd(gated_value, w.norm_conv, eps=w.eps, group_size=D)

    # Dilated causal depthwise conv1d + silu
    conv_k = w.conv1d_weight.shape[-1]
    dil = w.ngram_size
    pad_l = (conv_k - 1) * dil
    x_conv = F.pad(gv_normed.transpose(1, 2), (pad_l, 0))
    pre_silu = F.conv1d(x_conv, w.conv1d_weight, bias=None, dilation=dil, groups=hc * D)
    pre_f = pre_silu.float()
    sig_c = torch.sigmoid(pre_f)
    conv_out = (pre_f * sig_c).to(hidden_states.dtype).transpose(1, 2)
    d_silu_conv = (sig_c * (1.0 + pre_f * (1.0 - sig_c))).to(hidden_states.dtype)

    out = gated_value + conv_out
    saved = {
        "key_normed_4": key_normed_4.float(),
        "value_f": value.float(),
        "q_xhat": q_xhat,
        "q_rstd": q_rstd,
        "d_sig_raw": d_sig_raw,
        "c_xhat": c_xhat,
        "c_rstd": c_rstd,
        "d_silu_conv": d_silu_conv,
        "pad_l": pad_l,
        "dil": dil,
    }
    return out, saved


def ple_layer_vjp(
    d_out: torch.Tensor,  # [K, T, hc_count * D]
    w: PleLayerWeights,
    saved: dict[str, Any],
) -> torch.Tensor:
    """Exact VJP of `Qwen4ExpTextPLELayer` w.r.t. `hidden_states` -> `[K, T, hc_count * D]`."""
    hc = w.hc_count
    D = w.hidden_size
    T = d_out.shape[1]

    # Backward through short_conv (dilated depthwise conv1d + silu)
    d_pre = (d_out.transpose(1, 2) * saved["d_silu_conv"]).to(w.conv1d_weight.dtype)
    pad_l = saved["pad_l"]
    d_gv_normed = F.conv_transpose1d(
        d_pre, w.conv1d_weight, bias=None, dilation=saved["dil"], groups=hc * D
    )[:, :, pad_l : pad_l + T].transpose(1, 2).to(d_out.dtype)

    d_gated_value = d_out + rmsnorm_vjp(
        d_gv_normed, saved["c_xhat"], saved["c_rstd"], w.norm_conv, group_size=D
    )

    # Backward through gated_value = sig_gate * value
    d_gv_4 = d_gated_value.float().unflatten(-1, (hc, D))
    d_sig = (d_gv_4 * saved["value_f"].unsqueeze(-2)).sum(dim=-1, keepdim=True)  # [K, T, hc, 1]
    d_raw_gate = d_sig * saved["d_sig_raw"] * (1.0 / math.sqrt(D))  # [K, T, hc, 1]

    # Backward through raw_gate = sum(key_normed_4 * query_normed_4) / sqrt(D)
    d_query_normed = (d_raw_gate * saved["key_normed_4"]).flatten(-2).to(d_out.dtype)
    return rmsnorm_vjp(d_query_normed, saved["q_xhat"], saved["q_rstd"], w.norm_query, group_size=D)


# --------------------------------------------------------------------------- #
# 4. Exact Qwen Sparse Attention (QSA) Forward & L2-Cached Attention VJP
# --------------------------------------------------------------------------- #
def _rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_partial_rope(
    x: torch.Tensor,      # [..., T, head_dim]
    cos: torch.Tensor,    # [1, 1, T, rotary_dim]
    sin: torch.Tensor,    # [1, 1, T, rotary_dim]
) -> torch.Tensor:
    rot_dim = cos.shape[-1]
    x_rot = x[..., :rot_dim]
    x_pass = x[..., rot_dim:]
    return torch.cat([(x_rot * cos) + (_rotate_half(x_rot) * sin), x_pass], dim=-1)


def build_rope_cos_sin(
    seq_len: int,
    head_dim: int,
    partial_rotary_factor: float,
    rope_theta: float,
    device: torch.device,
    dtype: torch.dtype,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Exact Qwen4Exp 1D text RoPE cos/sin (for text-only sequences, all 3 M-RoPE dims equal position_ids)."""
    rot_dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (rope_theta ** (torch.arange(0, rot_dim, 2, device=device, dtype=torch.float32) / rot_dim))
    pos = torch.arange(seq_len, device=device, dtype=torch.float32)
    freqs = torch.outer(pos, inv_freq)  # [T, rot_dim // 2]
    emb = torch.cat([freqs, freqs], dim=-1)  # [T, rot_dim]
    cos = emb.cos().to(dtype).view(1, 1, seq_len, rot_dim)
    sin = emb.sin().to(dtype).view(1, 1, seq_len, rot_dim)
    return cos, sin


def qsa_layer_fwd(
    hidden_states: torch.Tensor,  # [1, T, D]
    w: QsaLayerWeights,
    cos: torch.Tensor,            # [1, 1, T, rot_dim]
    sin: torch.Tensor,            # [1, 1, T, rot_dim]
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Exact QSA forward.

    Mathematical guarantee: when `T <= indexer_budget` (2048), QSA's top-k block indexer
    selects 100% of causal positions and has zero gradient (boolean mask). Bypassing the
    indexer eliminates O(B*T) Python loops and GPU syncs while remaining bit-identical!
    """
    B, T, _ = hidden_states.shape
    H_q, H_kv, Dh = w.num_q_heads, w.num_kv_heads, w.head_dim
    n_rep = H_q // H_kv
    scaling = Dh ** -0.5

    qg = F.linear(hidden_states, w.q_proj).view(B, T, H_q, Dh * 2)
    q_raw, gate = torch.chunk(qg, 2, dim=-1)
    gate_flat = gate.reshape(B, T, H_q * Dh)

    q_normed, q_xhat, q_rstd = rmsnorm_fwd(q_raw, w.q_norm, eps=w.eps)
    k_raw = F.linear(hidden_states, w.k_proj).view(B, T, H_kv, Dh)
    k_normed, k_xhat, k_rstd = rmsnorm_fwd(k_raw, w.k_norm, eps=w.eps)
    v = F.linear(hidden_states, w.v_proj).view(B, T, H_kv, Dh).transpose(1, 2)  # [1, H_kv, T, Dh]

    q_rot = apply_partial_rope(q_normed.transpose(1, 2), cos, sin)  # [1, H_q, T, Dh]
    k_rot = apply_partial_rope(k_normed.transpose(1, 2), cos, sin)  # [1, H_kv, T, Dh]

    k_rep = k_rot.unsqueeze(2).expand(B, H_kv, n_rep, T, Dh).reshape(B, H_q, T, Dh)
    v_rep = v.unsqueeze(2).expand(B, H_kv, n_rep, T, Dh).reshape(B, H_q, T, Dh)

    if T <= 512 and T <= w.indexer_budget:
        # L2-cached exact attention weights P [H_q, T, T] (~768 KB at T=128)
        scores = torch.matmul(q_rot[0].float(), k_rep[0].float().transpose(-1, -2)) * scaling
        causal_mask = torch.ones((T, T), device=hidden_states.device, dtype=torch.bool).triu(1)
        scores.masked_fill_(causal_mask, float("-inf"))
        P = torch.softmax(scores, dim=-1)  # [H_q, T, T] fp32
        attn_out = torch.matmul(P.to(v_rep.dtype), v_rep[0]).transpose(0, 1).contiguous().view(B, T, H_q * Dh)
        attn_mode = "l2_cached"
    else:
        attn_out = F.scaled_dot_product_attention(q_rot, k_rep, v_rep, is_causal=True, scale=scaling)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, H_q * Dh)
        P = None
        attn_mode = "sdpa"

    sig_gate = torch.sigmoid(gate_flat.float())
    gated_out = (attn_out.float() * sig_gate).to(hidden_states.dtype)
    out = F.linear(gated_out, w.o_proj)

    saved = {
        "attn_mode": attn_mode,
        "P": P,
        "q_rot": q_rot[0],
        "k_rep": k_rep[0],
        "v_rep": v_rep[0],
        "q_xhat": q_xhat,
        "q_rstd": q_rstd,
        "k_xhat": k_xhat,
        "k_rstd": k_rstd,
        "attn_out": attn_out[0],
        "sig_gate": sig_gate[0].to(hidden_states.dtype),
        "d_sig_gate": (sig_gate[0] * (1.0 - sig_gate[0])).to(hidden_states.dtype),
        "cos": cos,
        "sin": sin,
        "scaling": scaling,
    }
    return out, saved


def qsa_layer_vjp(
    d_out: torch.Tensor,  # [K_sub, T, D]
    w: QsaLayerWeights,
    saved: dict[str, Any],
) -> torch.Tensor:
    """Exact batched-cotangent VJP of `Qwen4ExpTextAttention` -> `[K_sub, T, D]`."""
    K_sub, T, _ = d_out.shape
    H_q, H_kv, Dh = w.num_q_heads, w.num_kv_heads, w.head_dim
    n_rep = H_q // H_kv
    dtype = d_out.dtype

    # 1. o_proj & sigmoid output gate VJP
    d_gated = torch.matmul(d_out, w.o_proj)  # [K_sub, T, H_q * Dh]
    d_gate = d_gated * saved["attn_out"].unsqueeze(0) * saved["d_sig_gate"].unsqueeze(0)
    d_attn_flat = d_gated * saved["sig_gate"].unsqueeze(0)
    dO = d_attn_flat.view(K_sub, T, H_q, Dh).transpose(1, 2).contiguous()  # [K_sub, H_q, T, Dh]

    # 2. Core causal attention VJP
    if saved["attn_mode"] == "l2_cached":
        P = saved["P"]                                # [H_q, T, T] fp32
        P_t = P.to(dtype)                             # [H_q, T, T]
        Q_1 = saved["q_rot"]                          # [H_q, T, Dh]
        K_1 = saved["k_rep"]                          # [H_q, T, Dh]
        V_1 = saved["v_rep"]                          # [H_q, T, Dh]

        # Reorder dO to [H_q, K_sub, T, Dh] -> [H_q, K_sub*T, Dh] for pure 3D batched GEMMs!
        dO_h = dO.transpose(0, 1).contiguous()        # [H_q, K_sub, T, Dh]

        # dV = P^T @ dO: [H_q, T, T] @ [H_q, T, K_sub*Dh] -> [H_q, T, K_sub, Dh]
        dO_for_v = dO_h.permute(0, 2, 1, 3).reshape(H_q, T, K_sub * Dh)
        dV_rep = torch.bmm(P_t.transpose(1, 2), dO_for_v).view(H_q, T, K_sub, Dh).permute(2, 0, 1, 3)

        # dP = dO @ V^T: [H_q, K_sub*T, Dh] @ [H_q, Dh, T] -> [H_q, K_sub, T, T]
        dP = torch.bmm(dO_h.view(H_q, K_sub * T, Dh), V_1.transpose(1, 2)).view(H_q, K_sub, T, T)

        # Softmax VJP in fp32: dS = P * (dP - sum(dP * P)) * scaling
        dP_f = dP.float()
        P_b = P.unsqueeze(1)
        D_dot = (dP_f * P_b).sum(dim=-1, keepdim=True)
        dS = ((P_b * (dP_f - D_dot)) * saved["scaling"]).to(dtype)  # [H_q, K_sub, T, T]

        # dQ = dS @ K: [H_q, K_sub*T, T] @ [H_q, T, Dh] -> [K_sub, H_q, T, Dh]
        dQ_rot = torch.bmm(dS.view(H_q, K_sub * T, T), K_1).view(H_q, K_sub, T, Dh).transpose(0, 1)

        # dK = dS^T @ Q: reshape dS^T to [H_q, T, K_sub, T]? Loop-free via matmul:
        # dS_perm: [H_q, K_sub, T_k, T_q] @ Q_1 [H_q, 1, T_q, Dh] -> [H_q, K_sub, T_k, Dh]
        dK_rep = torch.matmul(dS.transpose(-1, -2), Q_1.unsqueeze(1)).transpose(0, 1)  # [K_sub, H_q, T, Dh]
    else:
        with torch.enable_grad():
            q_e = saved["q_rot"].detach().unsqueeze(0).expand(K_sub, -1, -1, -1).clone().requires_grad_(True)
            k_e = saved["k_rep"].detach().unsqueeze(0).expand(K_sub, -1, -1, -1).clone().requires_grad_(True)
            v_e = saved["v_rep"].detach().unsqueeze(0).expand(K_sub, -1, -1, -1).clone().requires_grad_(True)
            o_e = F.scaled_dot_product_attention(q_e, k_e, v_e, is_causal=True, scale=saved["scaling"])
            dQ_rot, dK_rep, dV_rep = torch.autograd.grad(o_e, (q_e, k_e, v_e), grad_outputs=dO)

    # Reduce GQA heads H_q -> H_kv
    dK_rot = dK_rep.view(K_sub, H_kv, n_rep, T, Dh).sum(dim=2)  # [K_sub, H_kv, T, Dh]
    dV = dV_rep.view(K_sub, H_kv, n_rep, T, Dh).sum(dim=2)      # [K_sub, H_kv, T, Dh]

    # 3. Inverse partial RoPE: exact transpose is (cos, -sin)
    cos, sin = saved["cos"], saved["sin"]
    dQ_normed = apply_partial_rope(dQ_rot, cos, -sin).transpose(1, 2)  # [K_sub, T, H_q, Dh]
    dK_normed = apply_partial_rope(dK_rot, cos, -sin).transpose(1, 2)  # [K_sub, T, H_kv, Dh]

    # 4. q_norm & k_norm VJP
    dQ_raw = rmsnorm_vjp(dQ_normed, saved["q_xhat"], saved["q_rstd"], w.q_norm)
    dK_raw = rmsnorm_vjp(dK_normed, saved["k_xhat"], saved["k_rstd"], w.k_norm)

    # 5. Pack [dQ_raw, d_gate] matching q_proj output layout [K_sub, T, H_q, 2*Dh]
    dQG = torch.cat([dQ_raw, d_gate.view(K_sub, T, H_q, Dh)], dim=-1).reshape(K_sub, T, H_q * Dh * 2)
    dK_flat = dK_raw.reshape(K_sub, T, H_kv * Dh)
    dV_flat = dV.transpose(1, 2).reshape(K_sub, T, H_kv * Dh)

    d_x = (
        torch.matmul(dQG, w.q_proj)
        + torch.matmul(dK_flat, w.k_proj)
        + torch.matmul(dV_flat, w.v_proj)
    )
    return d_x


# --------------------------------------------------------------------------- #
# 5. Exact Gated DeltaNet (GDN) Forward & Direct Kernel VJP
# --------------------------------------------------------------------------- #
def gdn_layer_fwd(
    hidden_states: torch.Tensor,  # [1, T, D]
    w: GdnLayerWeights,
    gdn_backend: GatedDeltaRuleBackend,
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Exact GDN forward using FlashQLA (SM90/100/120) or FLA Triton (SM80) or PyTorch."""
    B, T, _ = hidden_states.shape
    key_dim = w.num_k_heads * w.head_k_dim
    val_dim = w.num_v_heads * w.head_v_dim

    mixed_qkv = F.linear(hidden_states, w.in_proj_qkv).transpose(1, 2)  # [1, conv_dim, T]
    z = F.linear(hidden_states, w.in_proj_z).view(B, T, w.num_v_heads, w.head_v_dim)
    b = F.linear(hidden_states, w.in_proj_b)  # [1, T, H_v]
    a = F.linear(hidden_states, w.in_proj_a)  # [1, T, H_v]

    mixed_qkv_act, conv_pre_silu = causal_conv1d_silu_fwd(mixed_qkv, w.conv1d_weight)
    mixed_qkv_t = mixed_qkv_act.transpose(1, 2)
    q, k, v = torch.split(mixed_qkv_t, [key_dim, key_dim, val_dim], dim=-1)
    q = q.reshape(B, T, w.num_k_heads, w.head_k_dim)
    k = k.reshape(B, T, w.num_k_heads, w.head_k_dim)
    v = v.reshape(B, T, w.num_v_heads, w.head_v_dim)

    beta = torch.sigmoid(b)
    neg_exp_A = -w.A_log.float().exp()
    a_plus_dt = a.float() + w.dt_bias.float()
    g = neg_exp_A * F.softplus(a_plus_dt)

    core_out, gdn_saved = gdn_backend.forward_single(q, k, v, g, beta)
    norm_out, norm_saved = rmsnorm_gated_fwd(
        core_out, z, w.norm_weight, activation=w.output_gate_type, eps=w.eps
    )
    out = F.linear(norm_out.reshape(B, T, val_dim), w.out_proj)

    saved = {
        "conv_pre_silu": conv_pre_silu,
        "beta": beta,
        "neg_exp_A": neg_exp_A,
        "sig_a_plus_dt": torch.sigmoid(a_plus_dt),
        "gdn_saved": gdn_saved,
        "norm_saved": norm_saved,
    }
    return out, saved


def gdn_layer_vjp(
    d_out: torch.Tensor,  # [K_sub, T, D]
    w: GdnLayerWeights,
    saved: dict[str, Any],
    gdn_backend: GatedDeltaRuleBackend,
) -> torch.Tensor:
    """Exact batched-cotangent VJP of `Qwen4ExpTextGatedDeltaNet` -> `[K_sub, T, D]`."""
    K_sub, T, _ = d_out.shape
    val_dim = w.num_v_heads * w.head_v_dim
    dtype = d_out.dtype

    # 1. out_proj & RMSNormGated VJP
    d_norm_out = torch.matmul(d_out, w.out_proj).view(K_sub, T, w.num_v_heads, w.head_v_dim)
    d_core_out, d_z = rmsnorm_gated_vjp(d_norm_out, saved["norm_saved"])

    # 2. Core Gated Delta Rule VJP (FlashQLA SM120 / FLA SM80 / PyTorch)
    dq, dk, dv, dg, dbeta = gdn_backend.backward_batched(saved["gdn_saved"], d_core_out)

    # 3. Gate activations VJP: beta = sigmoid(b), g = -exp(A_log) * softplus(a + dt_bias)
    beta_f = saved["beta"].float()
    db = (dbeta.float() * beta_f * (1.0 - beta_f)).to(dtype)
    da = (dg.float() * saved["neg_exp_A"] * saved["sig_a_plus_dt"]).to(dtype)

    # 4. Pack [dq, dk, dv] -> causal_conv1d + SiLU VJP
    d_mixed_t = torch.cat(
        [dq.reshape(K_sub, T, -1), dk.reshape(K_sub, T, -1), dv.reshape(K_sub, T, -1)],
        dim=-1,
    )
    d_conv_in = causal_conv1d_silu_bwd(
        d_mixed_t.transpose(1, 2), saved["conv_pre_silu"], w.conv1d_weight
    ).transpose(1, 2)  # [K_sub, T, conv_dim]

    # 5. Input projections VJP
    d_x = (
        torch.matmul(d_conv_in, w.in_proj_qkv)
        + torch.matmul(d_z.reshape(K_sub, T, val_dim), w.in_proj_z)
        + torch.matmul(db, w.in_proj_b)
        + torch.matmul(da, w.in_proj_a)
    )
    return d_x


# --------------------------------------------------------------------------- #
# 6. Exact Sparse MoE Forward & Fused Active-Expert Grouped-GEMM VJP
# --------------------------------------------------------------------------- #
def _extract_active_experts_bf16(
    experts: MoeExpertPayload,
    active_ids: torch.Tensor,  # [E_hit] int64 on target device
    block_size: tuple[int, int],
    out_dtype: torch.dtype,
    use_triton: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Gather ONLY the `E_hit` active experts and dequantize FP8 -> `out_dtype` ONCE."""
    gu_sel = experts.gate_up_proj.index_select(0, active_ids)
    dn_sel = experts.down_proj.index_select(0, active_ids)
    if gu_sel.element_size() == 1 and experts.gate_up_scale is not None:
        gu_s = experts.gate_up_scale.index_select(0, active_ids)
        dn_s = experts.down_scale.index_select(0, active_ids)  # type: ignore[union-attr]
        W_gu = dequantize_fp8_blockwise(gu_sel, gu_s, block_size=block_size, out_dtype=out_dtype, use_triton=use_triton)
        W_dn = dequantize_fp8_blockwise(dn_sel, dn_s, block_size=block_size, out_dtype=out_dtype, use_triton=use_triton)
    else:
        W_gu = gu_sel.to(out_dtype)
        W_dn = dn_sel.to(out_dtype)
    return W_gu, W_dn


def moe_block_fwd(
    hidden_states: torch.Tensor,  # [1, T, D]
    dense: MoeDenseWeights,
    experts: MoeExpertPayload,
    *,
    dequant_chunk_size: int = 512,
    use_triton_dequant: bool = True,
    keep_dequantized_for_bwd: bool = False,
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Exact `B=1` forward of `Qwen4ExpTextSparseMoeBlock`.

    When `keep_dequantized_for_bwd=True` and `E_hit <= dequant_chunk_size`, keeps the
    active experts' BF16 weights (`W_gu`, `W_dn`) directly in `saved` so backward reuses
    them across ALL 2560 cotangents with ZERO redundant FP8 dequantization!
    """
    _, T, D = hidden_states.shape
    x = hidden_states[0]  # [T, D]
    dtype = x.dtype
    top_k = dense.top_k

    # 1. Shared expert + gate
    sh_gate = F.linear(x, dense.shared_gate_proj)
    sh_up = F.linear(x, dense.shared_up_proj)
    sh_gf = sh_gate.float()
    sh_sig = torch.sigmoid(sh_gf)
    sh_act = (sh_gf * sh_sig * sh_up.float()).to(dtype)
    sh_out = F.linear(sh_act, dense.shared_down_proj)  # [T, D]
    sg_logit = F.linear(x, dense.shared_expert_gate)   # [T, 1]
    sg_val = torch.sigmoid(sg_logit.float())           # [T, 1] fp32

    # 2. Top-K Router (exact match with Qwen4ExpTextTopKRouter)
    router_logits = F.linear(x, dense.router_weight)              # [T, E]
    router_probs = torch.softmax(router_logits.float(), dim=-1)   # [T, E] fp32
    top_vals_raw, top_idx = torch.topk(router_probs, top_k, dim=-1)  # [T, k]
    if dense.norm_topk_prob:
        top_sum = top_vals_raw.sum(dim=-1, keepdim=True)
        routing_weights_f = top_vals_raw / top_sum
    else:
        top_sum = None
        routing_weights_f = top_vals_raw
    routing_weights = routing_weights_f.to(dtype)

    # 3. Compact active-expert sorting (ONLY hit experts!)
    S = T * top_k
    flat_eids = top_idx.reshape(S)
    sorted_eids, perm = torch.sort(flat_eids)
    inv_perm = torch.empty_like(perm)
    inv_perm[perm] = torch.arange(S, device=x.device)
    tok_of_sorted = perm // top_k  # [S]
    w_sorted = routing_weights.reshape(S)[perm]  # [S]

    active_ids, counts = torch.unique_consecutive(sorted_eids, return_counts=True)
    E_hit = int(active_ids.numel())
    offs_1 = torch.cumsum(counts, dim=0, dtype=torch.int32)
    offs_1_list = offs_1.tolist()

    x_sorted = x[tok_of_sorted]  # [S, D]

    # Run active experts (chunked if E_hit > dequant_chunk_size to cap BF16 peak VRAM on A100-40G!)
    inter = dense.shared_gate_proj.shape[0]
    # Note: routed intermediate size comes from gate_up_proj shape[1] // 2
    r_inter = experts.gate_up_proj.shape[1] // 2
    y_exp_sorted = torch.empty((S, D), device=x.device, dtype=dtype)
    da_dg_sorted = torch.empty((S, r_inter), device=x.device, dtype=dtype)
    da_du_sorted = torch.empty((S, r_inter), device=x.device, dtype=dtype)

    cached_W_gu = None
    cached_W_dn = None

    if E_hit <= dequant_chunk_size:
        W_gu, W_dn = _extract_active_experts_bf16(
            experts, active_ids, dense.fp8_block_size, dtype, use_triton_dequant
        )
        # Forward: [S, D] @ [E_hit, D, 2*r_inter] -> [S, 2*r_inter]
        gu_out = grouped_mm(x_sorted, W_gu.transpose(-2, -1).contiguous(), offs_1, offs_1_list)
        g_act, u_act = gu_out.chunk(2, dim=-1)
        gf = g_act.float()
        sig = torch.sigmoid(gf)
        silu_g = gf * sig
        a_out = (silu_g * u_act.float()).to(dtype)
        da_dg_sorted.copy_((sig * (1.0 + gf * (1.0 - sig)) * u_act.float()).to(dtype))
        da_du_sorted.copy_(silu_g.to(dtype))
        y_exp_sorted.copy_(grouped_mm(a_out, W_dn.transpose(-2, -1).contiguous(), offs_1, offs_1_list))
        if keep_dequantized_for_bwd:
            cached_W_gu = W_gu
            cached_W_dn = W_dn
        else:
            del W_gu, W_dn
    else:
        for c_start in range(0, E_hit, dequant_chunk_size):
            c_end = min(c_start + dequant_chunk_size, E_hit)
            row_lo = offs_1_list[c_start - 1] if c_start > 0 else 0
            row_hi = offs_1_list[c_end - 1]
            sub_offs = offs_1[c_start:c_end] - row_lo
            sub_offs_list = [v - row_lo for v in offs_1_list[c_start:c_end]]
            W_gu, W_dn = _extract_active_experts_bf16(
                experts, active_ids[c_start:c_end], dense.fp8_block_size, dtype, use_triton_dequant
            )
            gu_out = grouped_mm(x_sorted[row_lo:row_hi], W_gu.transpose(-2, -1).contiguous(), sub_offs, sub_offs_list)
            g_act, u_act = gu_out.chunk(2, dim=-1)
            gf = g_act.float()
            sig = torch.sigmoid(gf)
            silu_g = gf * sig
            a_out = (silu_g * u_act.float()).to(dtype)
            da_dg_sorted[row_lo:row_hi].copy_((sig * (1.0 + gf * (1.0 - sig)) * u_act.float()).to(dtype))
            da_du_sorted[row_lo:row_hi].copy_(silu_g.to(dtype))
            y_exp_sorted[row_lo:row_hi].copy_(
                grouped_mm(a_out, W_dn.transpose(-2, -1).contiguous(), sub_offs, sub_offs_list)
            )
            del W_gu, W_dn

    weighted_sorted = y_exp_sorted * w_sorted.unsqueeze(-1)
    routed_out = weighted_sorted[inv_perm].view(T, top_k, D).sum(dim=1).to(dtype)
    total_out = (routed_out + (sg_val * sh_out.float()).to(dtype)).unsqueeze(0)  # [1, T, D]

    # Precompute router expectation E_p[W_router] = router_probs @ W_router ONCE ([T, D]) for O(1) softmax VJP!
    p_dot_W_router = torch.matmul(router_probs.to(dtype), dense.router_weight)  # [T, D]
    W_router_top = dense.router_weight[top_idx]  # [T, top_k, D]

    saved = {
        "T": T,
        "D": D,
        "r_inter": r_inter,
        "S": S,
        "E_hit": E_hit,
        "active_ids": active_ids,
        "offs_1": offs_1,
        "offs_1_list": offs_1_list,
        "perm": perm,
        "inv_perm": inv_perm,
        "tok_of_sorted": tok_of_sorted,
        "w_sorted": w_sorted,
        "y_exp_sorted": y_exp_sorted,
        "da_dg_sorted": da_dg_sorted,
        "da_du_sorted": da_du_sorted,
        "cached_W_gu": cached_W_gu,
        "cached_W_dn": cached_W_dn,
        # Router VJP state
        "top_vals_raw": top_vals_raw,          # [T, k] fp32
        "top_sum": top_sum,                    # [T, 1] fp32 or None
        "routing_weights_f": routing_weights_f,  # [T, k] fp32
        "p_dot_W_router": p_dot_W_router,      # [T, D] dtype
        "W_router_top": W_router_top,          # [T, k, D] dtype
        # Shared expert VJP state
        "sh_out": sh_out,                      # [T, D]
        "sh_da_dg": (sh_sig * (1.0 + sh_gf * (1.0 - sh_sig)) * sh_up.float()).to(dtype),
        "sh_da_du": (sh_gf * sh_sig).to(dtype),
        "sg_val": sg_val.to(dtype),            # [T, 1]
        "d_sg": (sg_val * (1.0 - sg_val)).to(dtype),  # [T, 1]
    }
    return total_out, saved


def moe_block_vjp_all(
    d_out_all: torch.Tensor,  # [K_total, T, D]
    dense: MoeDenseWeights,
    experts: MoeExpertPayload,
    saved: dict[str, Any],
    *,
    moe_k_chunk: int = 256,
    dequant_chunk_size: int = 512,
    use_triton_dequant: bool = True,
    use_triton_swiglu: bool = True,
) -> torch.Tensor:
    """Exact VJP of `Qwen4ExpTextSparseMoeBlock` across ALL `K_total` cotangents.

    Key optimization:
      - When `E_hit <= dequant_chunk_size` (A100-80G / G4-96G): active experts are already
        dequantized ONCE in `saved["cached_W_gu"]` / `saved["cached_W_dn"]` and reused for
        every `moe_k_chunk` slice with 100% Tensor-Core-saturated `grouped_mm`!
      - When `E_hit > dequant_chunk_size` (A100-40G chunked mode): loops over expert chunks
        OUTSIDE the `K` loop so EVERY active expert is STILL dequantized ONLY ONCE across
        all `K_total = 2560` cotangents!!
    """
    K_total, T, D = d_out_all.shape
    dtype = d_out_all.dtype
    top_k = dense.top_k
    S = saved["S"]
    E_hit = saved["E_hit"]
    offs_1 = saved["offs_1"]
    offs_1_list = saved["offs_1_list"]
    tok_of_sorted = saved["tok_of_sorted"]
    w_sorted = saved["w_sorted"]
    inv_perm = saved["inv_perm"]

    # Transpose to [T, K_total, D] for O(1) token gather into expert-grouped layout [S, K_blk, D]
    dY_TK = d_out_all.transpose(0, 1).contiguous()  # [T, K_total, D]
    dX_TK = torch.zeros_like(dY_TK)

    # 1. Shared expert + router VJP (tiny dense BF16 ops -> run in K chunks, < 1 ms total)
    y_exp_sorted = saved["y_exp_sorted"]  # [S, D]
    sh_out = saved["sh_out"]              # [T, D]
    sh_da_dg = saved["sh_da_dg"]          # [T, sh_inter]
    sh_da_du = saved["sh_da_du"]          # [T, sh_inter]
    sg_val = saved["sg_val"]              # [T, 1]
    d_sg = saved["d_sg"]                  # [T, 1]
    top_vals_raw = saved["top_vals_raw"]  # [T, k] fp32
    top_sum = saved["top_sum"]            # [T, 1] fp32 or None
    rw_f = saved["routing_weights_f"]     # [T, k] fp32
    p_dot_W = saved["p_dot_W_router"]     # [T, D]
    W_r_top = saved["W_router_top"]       # [T, k, D]

    for k0 in range(0, K_total, moe_k_chunk):
        k1 = min(k0 + moe_k_chunk, K_total)
        dy_tk = dY_TK[:, k0:k1, :]  # [T, K_blk, D]

        # 1a. Shared expert VJP
        d_sh_out = dy_tk * sg_val.unsqueeze(1)  # [T, K_blk, D]
        d_sg_logit = (dy_tk.float() * sh_out.float().unsqueeze(1)).sum(dim=-1, keepdim=True).to(dtype) * d_sg.unsqueeze(1)
        d_sh_act = torch.matmul(d_sh_out, dense.shared_down_proj)  # [T, K_blk, sh_inter]
        d_sh_g = d_sh_act * sh_da_dg.unsqueeze(1)
        d_sh_u = d_sh_act * sh_da_du.unsqueeze(1)
        dx_shared = (
            torch.matmul(d_sh_g, dense.shared_gate_proj)
            + torch.matmul(d_sh_u, dense.shared_up_proj)
            + torch.matmul(d_sg_logit, dense.shared_expert_gate)
        )

        # 1b. Exact Top-K Router VJP (without materializing [T, K_blk, 512]!)
        # dw_sorted[s, k] = <dY[tok(s), k, :], y_exp_sorted[s, :]>
        dy_sorted_raw = dy_tk[tok_of_sorted]  # [S, K_blk, D]
        dw_sorted = (dy_sorted_raw.float() * y_exp_sorted.float().unsqueeze(1)).sum(dim=-1)  # [S, K_blk]
        dw = dw_sorted[inv_perm].view(T, top_k, k1 - k0).transpose(1, 2)  # [T, K_blk, k] fp32
        if top_sum is not None:
            dot_w = (dw * rw_f.unsqueeze(1)).sum(dim=-1, keepdim=True)
            d_top_v = (dw - dot_w) / top_sum.unsqueeze(1)
        else:
            d_top_v = dw
        # Softmax VJP: sum_e p_e (dp_e - dot_p) W_e = sum_r (p_r * d_top_v_r) W_{idx_r} - dot_p * (p @ W)
        p_times_dv = d_top_v * top_vals_raw.unsqueeze(1)  # [T, K_blk, k] fp32
        dot_p = p_times_dv.sum(dim=-1, keepdim=True)      # [T, K_blk, 1] fp32
        dx_router = (
            torch.bmm(p_times_dv.to(dtype), W_r_top)
            - dot_p.to(dtype) * p_dot_W.unsqueeze(1)
        )
        dX_TK[:, k0:k1, :].add_(dx_shared + dx_router)

    # 2. Routed experts VJP (each expert chunk dequantized ONLY ONCE across ALL K_total!)
    da_dg_sorted = saved["da_dg_sorted"]
    da_du_sorted = saved["da_du_sorted"]
    active_ids = saved["active_ids"]

    if saved["cached_W_gu"] is not None:
        expert_chunks = [(0, E_hit, saved["cached_W_gu"], saved["cached_W_dn"])]
    else:
        expert_chunks = [
            (c0, min(c0 + dequant_chunk_size, E_hit), None, None)
            for c0 in range(0, E_hit, dequant_chunk_size)
        ]

    for c0, c1, pre_W_gu, pre_W_dn in expert_chunks:
        row_lo = offs_1_list[c0 - 1] if c0 > 0 else 0
        row_hi = offs_1_list[c1 - 1]
        n_rows = row_hi - row_lo
        if n_rows == 0:
            continue

        if pre_W_gu is not None and pre_W_dn is not None:
            W_gu, W_dn = pre_W_gu, pre_W_dn
        else:
            W_gu, W_dn = _extract_active_experts_bf16(
                experts, active_ids[c0:c1], dense.fp8_block_size, dtype, use_triton_dequant
            )

        sub_tok = tok_of_sorted[row_lo:row_hi]         # [n_rows]
        sub_w = w_sorted[row_lo:row_hi]                # [n_rows]
        sub_dg = da_dg_sorted[row_lo:row_hi]           # [n_rows, r_inter]
        sub_du = da_du_sorted[row_lo:row_hi]           # [n_rows, r_inter]
        sub_offs_1 = offs_1[c0:c1] - row_lo            # [c1 - c0] int32
        sub_offs_1_list = [v - row_lo for v in offs_1_list[c0:c1]]

        # Notice W_dn is [E_c, D, r_inter] and W_gu is [E_c, 2*r_inter, D]:
        # ALREADY in exact contiguous [E_c, K_in, N_out] layout for backward grouped_mm!
        for k0 in range(0, K_total, moe_k_chunk):
            k1 = min(k0 + moe_k_chunk, K_total)
            K_blk = k1 - k0

            # Gather [n_rows, K_blk, D] and scale by routing weights
            dy_s = dY_TK[sub_tok, k0:k1, :] * sub_w[:, None, None]  # [n_rows, K_blk, D]
            offs_K = sub_offs_1 * K_blk
            offs_K_list = [v * K_blk for v in sub_offs_1_list]

            # Down-proj backward: [n_rows * K_blk, D] @ [E_c, D, r_inter] -> [n_rows * K_blk, r_inter]
            dA = grouped_mm(dy_s.reshape(n_rows * K_blk, D), W_dn, offs_K, offs_K_list).view(n_rows, K_blk, -1)

            # Fused SwiGLU backward + pack [dG | dU] -> [n_rows * K_blk, 2 * r_inter]
            dGU = swiglu_vjp_pack(dA, sub_dg, sub_du, use_triton=use_triton_swiglu).reshape(n_rows * K_blk, -1)
            del dA

            # Gate-Up-proj backward: [n_rows * K_blk, 2*r_inter] @ [E_c, 2*r_inter, D] -> [n_rows, K_blk, D]
            dX_s = grouped_mm(dGU, W_gu, offs_K, offs_K_list).view(n_rows, K_blk, D)
            del dGU

            # Deterministic accumulation back to token axis (no race conditions / atomicAdd!)
            if c0 == 0 and c1 == E_hit:
                # Full-active fast path: direct inv_perm + sum(dim=1) (zero index_add_)
                dX_TK[:, k0:k1, :].add_(dX_s[inv_perm].view(T, top_k, K_blk, D).sum(dim=1))
            else:
                # Chunked path: accumulate in float32 per token cleanly
                dX_TK[:, k0:k1, :].index_add_(0, sub_tok, dX_s)
            del dX_s

        if pre_W_gu is None:
            del W_gu, W_dn

    return dX_TK.transpose(0, 1).contiguous()  # [K_total, T, D]


# --------------------------------------------------------------------------- #
# 7. Complete Single-Layer Forward (`B=1`) & Full-Cotangent In-Place Backward
# --------------------------------------------------------------------------- #
def decoder_layer_fwd(
    H_in: torch.Tensor,  # [1, T, hc_count * D]
    layer_w: DecoderLayerDenseWeights,
    experts: MoeExpertPayload,
    cos: torch.Tensor,
    sin: torch.Tensor,
    gdn_backend: GatedDeltaRuleBackend,
    *,
    ple_embeddings: torch.Tensor | None = None,
    dequant_chunk_size: int = 512,
    use_triton_dequant: bool = True,
    keep_experts_dequantized: bool = False,
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Run ONE `B=1` forward of `Qwen4ExpTextDecoderLayer`; returns `(H_out, saved)`."""
    saved: dict[str, Any] = {}

    # 1. Optional PLE injection (layer 2 only)
    if layer_w.ple is not None:
        assert ple_embeddings is not None, f"Layer {layer_w.layer_idx} requires ple_embeddings."
        ple_delta, ple_saved = ple_layer_fwd(H_in, ple_embeddings, layer_w.ple)
        H_ple = H_in + ple_delta
        saved["ple_saved"] = ple_saved
    else:
        H_ple = H_in

    # 2. Attention Hyper-Connection + Token Mixer (GDN or QSA) + Residual Injection
    x_attn, w_attn, attn_hc_saved = gated_residual_fwd(H_ple, layer_w.attn_hc)
    assert w_attn is not None
    if layer_w.layer_type == "linear_attention":
        assert layer_w.gdn is not None
        y_attn, mixer_saved = gdn_layer_fwd(x_attn, layer_w.gdn, gdn_backend)
    else:
        assert layer_w.qsa is not None
        y_attn, mixer_saved = qsa_layer_fwd(x_attn, layer_w.qsa, cos, sin)

    H_mid = H_ple + (y_attn.unsqueeze(-2) * w_attn.unsqueeze(-1)).flatten(-2)

    # 3. MLP Hyper-Connection + Sparse MoE + Residual Injection
    x_mlp, w_mlp, mlp_hc_saved = gated_residual_fwd(H_mid, layer_w.mlp_hc)
    assert w_mlp is not None
    y_mlp, moe_saved = moe_block_fwd(
        x_mlp,
        layer_w.moe_dense,
        experts,
        dequant_chunk_size=dequant_chunk_size,
        use_triton_dequant=use_triton_dequant,
        keep_dequantized_for_bwd=keep_experts_dequantized,
    )
    H_out = H_mid + (y_mlp.unsqueeze(-2) * w_mlp.unsqueeze(-1)).flatten(-2)

    saved.update(
        {
            "attn_hc_saved": attn_hc_saved,
            "mixer_saved": mixer_saved,
            "y_attn": y_attn[0],
            "w_attn": w_attn[0],
            "mlp_hc_saved": mlp_hc_saved,
            "moe_saved": moe_saved,
            "y_mlp": y_mlp[0],
            "w_mlp": w_mlp[0],
        }
    )
    return H_out, saved


def decoder_layer_vjp_inplace(
    G: torch.Tensor,  # [K_total, T, hc_count * D] — UPDATED IN-PLACE TO G_in!
    layer_w: DecoderLayerDenseWeights,
    experts: MoeExpertPayload,
    saved: dict[str, Any],
    gdn_backend: GatedDeltaRuleBackend,
    *,
    attn_k_chunk: int = 128,
    moe_k_chunk: int = 256,
    dequant_chunk_size: int = 512,
    use_triton_dequant: bool = True,
    use_triton_swiglu: bool = True,
) -> torch.Tensor:
    """Exact in-place VJP of `Qwen4ExpTextDecoderLayer` across ALL `K_total` cotangents.

    Updates `G` in-place so peak GPU cotangent memory stays at ONLY 1x `G_curr` (6.25 GiB)!
    """
    K_total, T, D_hc = G.shape
    hc = layer_w.mlp_hc.hc_count
    D = layer_w.mlp_hc.hidden_size

    # ---- Stage A: MLP Injection + SparseMoE + MLP Hyper-Connection ----
    y_mlp = saved["y_mlp"]  # [T, D]
    w_mlp = saved["w_mlp"]  # [T, hc]

    # Compute d_y_mlp [K_total, T, D] and d_w_mlp [K_total, T, hc] in small chunks to avoid extra 6.25 GiB temp
    d_y_mlp = torch.empty((K_total, T, D), device=G.device, dtype=G.dtype)
    d_w_mlp = torch.empty((K_total, T, hc), device=G.device, dtype=G.dtype)
    for k0 in range(0, K_total, moe_k_chunk):
        k1 = min(k0 + moe_k_chunk, K_total)
        g4 = G[k0:k1].view(k1 - k0, T, hc, D)
        d_y_mlp[k0:k1] = (g4 * w_mlp[None, :, :, None]).sum(dim=2)
        d_w_mlp[k0:k1] = (g4.float() * y_mlp.float()[None, :, None, :]).sum(dim=-1).to(G.dtype)

    d_x_mlp = moe_block_vjp_all(
        d_y_mlp,
        layer_w.moe_dense,
        experts,
        saved["moe_saved"],
        moe_k_chunk=moe_k_chunk,
        dequant_chunk_size=dequant_chunk_size,
        use_triton_dequant=use_triton_dequant,
        use_triton_swiglu=use_triton_swiglu,
    )
    del d_y_mlp
    # Free cached BF16 active experts immediately after MoE backward!
    saved["moe_saved"]["cached_W_gu"] = None
    saved["moe_saved"]["cached_W_dn"] = None

    # Update G in-place: G_mid = G + gated_residual_vjp(d_x_mlp, d_w_mlp)
    for k0 in range(0, K_total, moe_k_chunk):
        k1 = min(k0 + moe_k_chunk, K_total)
        G[k0:k1].add_(
            gated_residual_vjp(d_x_mlp[k0:k1], d_w_mlp[k0:k1], layer_w.mlp_hc, saved["mlp_hc_saved"])
        )
    del d_x_mlp, d_w_mlp

    # ---- Stage B: Attention Injection + Token Mixer (GDN/QSA) + Attn Hyper-Connection ----
    y_attn = saved["y_attn"]
    w_attn = saved["w_attn"]
    for k0 in range(0, K_total, attn_k_chunk):
        k1 = min(k0 + attn_k_chunk, K_total)
        g4 = G[k0:k1].view(k1 - k0, T, hc, D)
        d_y_attn = (g4 * w_attn[None, :, :, None]).sum(dim=2)
        d_w_attn = (g4.float() * y_attn.float()[None, :, None, :]).sum(dim=-1).to(G.dtype)

        if layer_w.layer_type == "linear_attention":
            assert layer_w.gdn is not None
            d_x_attn = gdn_layer_vjp(d_y_attn, layer_w.gdn, saved["mixer_saved"], gdn_backend)
        else:
            assert layer_w.qsa is not None
            d_x_attn = qsa_layer_vjp(d_y_attn, layer_w.qsa, saved["mixer_saved"])

        G[k0:k1].add_(
            gated_residual_vjp(d_x_attn, d_w_attn, layer_w.attn_hc, saved["attn_hc_saved"])
        )

    # ---- Stage C: Optional PLE Layer Backward (layer 2 only) ----
    if layer_w.ple is not None:
        ple_saved = saved["ple_saved"]
        for k0 in range(0, K_total, attn_k_chunk):
            k1 = min(k0 + attn_k_chunk, K_total)
            G[k0:k1].add_(ple_layer_vjp(G[k0:k1], layer_w.ple, ple_saved))

    return G


In [ ]:
%%writefile qwen38_jlens/loader.py
"""Streaming shard-by-shard safetensors loader & 3-tier MoE storage manager for Qwen3.8-Flash-Next.

Why a custom loader instead of `AutoModelForCausalLM.from_pretrained(..., device_map="auto")`?
  1. Avoids HF `transformers#48349` (multimodal `modules_to_not_convert` prefix mismatch silently
     corrupting BF16 PLE/GDN/GatedResidual weights into unscaled FP8!).
  2. Avoids HF `transformers#48350` & massive temporary CPU buffers during 51B PLE concatenation.
  3. Skips downloading/loading Vision encoder (`model.visual.*`) and MTP head (`mtp.*`) shards.
  4. Streams raw shards in small batches (~15 GiB), packs routed MoE experts directly into
     Tier 0 (GPU HBM), Tier 1 (Pinned/Pageable Host RAM), or Tier 2 (contiguous 2.34 GiB
     binary NVMe slabs for max sequential pread), and immediately deletes consumed raw HF blobs!
     Peak disk usage drops from >210 GiB to ~65-95 GiB (100% safe on Google Colab's 235 GiB disk!).
"""

from __future__ import annotations

import gc
import json
import logging
import os
import re
import struct
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch
from safetensors import safe_open

from qwen38_jlens.hardware import HardwareProfile, OffloadPlan, StorageTier, build_offload_plan
from qwen38_jlens.model_ops import (
    DecoderLayerDenseWeights,
    GatedResidualWeights,
    GdnLayerWeights,
    MoeDenseWeights,
    MoeExpertPayload,
    PleLayerWeights,
    QsaLayerWeights,
)
from qwen38_jlens.ple_mmap import MmapPleTable, PleHashSpec, PreextractedPleProvider

logger = logging.getLogger(__name__)

_EXPERT_SPLIT_RE = re.compile(
    r"^(?:model\.language_model\.|model\.)?layers\.(\d+)\.mlp\.experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.(weight|weight_scale_inv|weight_scale)$"
)
_EXPERT_FUSED_RE = re.compile(
    r"^(?:model\.language_model\.|model\.)?layers\.(\d+)\.mlp\.experts\.(gate_up_proj|down_proj|gate_up_proj_scale_inv|down_proj_scale_inv)$"
)
_LAYER_RE = re.compile(r"^(?:model\.language_model\.|model\.)?layers\.(\d+)\.(.+)$")
_PLE_SHARD_RE = re.compile(r"ngram_embedding\.shard_\d+\.weight$")

# Binary slab header for Tier 2 contiguous NVMe MoE storage
_SLAB_MAGIC = b"Q38M"
_SLAB_DTYPE_CODE = {
    torch.float8_e4m3fn: 1,
    torch.bfloat16: 2,
    torch.float32: 3,
}
_CODE_TO_DTYPE = {v: k for k, v in _SLAB_DTYPE_CODE.items()}


def write_moe_slab(path: str, payload: MoeExpertPayload) -> None:
    """Write one layer's packed MoE experts as a single contiguous binary file for max NVMe pread speed."""
    tensors = [payload.gate_up_proj, payload.down_proj]
    has_scales = payload.gate_up_scale is not None and payload.down_scale is not None
    if has_scales:
        assert payload.gate_up_scale is not None and payload.down_scale is not None
        tensors.extend([payload.gate_up_scale, payload.down_scale])

    meta = {
        "has_scales": has_scales,
        "tensors": [
            {
                "shape": list(t.shape),
                "dtype": _SLAB_DTYPE_CODE[t.dtype],
                "nbytes": t.numel() * t.element_size(),
            }
            for t in tensors
        ],
    }
    header_bytes = json.dumps(meta).encode("utf-8")
    # 4096-byte align data start for optimal NVMe page/DMA alignment
    raw_prefix_len = 8 + len(header_bytes)
    pad_len = (4096 - (raw_prefix_len % 4096)) % 4096

    tmp_path = f"{path}.tmp.{os.getpid()}"
    with open(tmp_path, "wb") as f:
        f.write(_SLAB_MAGIC)
        f.write(struct.pack("<I", len(header_bytes)))
        f.write(header_bytes)
        if pad_len:
            f.write(b"\x00" * pad_len)
        for t in tensors:
            t_cpu = t.detach().cpu().contiguous()
            f.write(t_cpu.view(torch.uint8).numpy().tobytes())
    os.replace(tmp_path, path)


def read_moe_slab_into(
    path: str,
    dst: MoeExpertPayload,
) -> MoeExpertPayload:
    """Sequentially read a contiguous Tier 2 NVMe MoE slab directly into pre-allocated pinned host buffers."""
    with open(path, "rb") as f:
        magic = f.read(4)
        if magic != _SLAB_MAGIC:
            raise ValueError(f"Invalid MoE slab magic in {path}: {magic!r}")
        (header_len,) = struct.unpack("<I", f.read(4))
        meta = json.loads(f.read(header_len).decode("utf-8"))
        raw_prefix_len = 8 + header_len
        pad_len = (4096 - (raw_prefix_len % 4096)) % 4096
        if pad_len:
            f.seek(pad_len, os.SEEK_CUR)

        targets = [dst.gate_up_proj, dst.down_proj]
        if meta["has_scales"]:
            assert dst.gate_up_scale is not None and dst.down_scale is not None
            targets.extend([dst.gate_up_scale, dst.down_scale])

        for t_info, dst_t in zip(meta["tensors"], targets, strict=True):
            nbytes = int(t_info["nbytes"])
            # Direct zero-copy readinto pinned buffer via memoryview (1 memcpy total!)
            mv = memoryview(dst_t.view(torch.uint8).numpy().reshape(-1))
            assert len(mv) == nbytes, f"Slab size mismatch: {len(mv)} vs {nbytes}"
            read_so_far = 0
            while read_so_far < nbytes:
                n = f.readinto(mv[read_so_far:])
                if not n:
                    raise EOFError(f"Unexpected EOF reading {path} at {read_so_far}/{nbytes}")
                read_so_far += n
    return dst


def allocate_moe_payload(
    num_experts: int,
    hidden_size: int,
    moe_intermediate_size: int,
    *,
    device: torch.device | str = "cpu",
    pin_memory: bool = False,
    weight_dtype: torch.dtype = torch.float8_e4m3fn,
    block_size: tuple[int, int] = (128, 128),
) -> MoeExpertPayload:
    """Pre-allocate one layer's packed MoE expert tensors (on GPU, pinned host, or pageable host)."""
    dev = torch.device(device)
    use_pin = pin_memory and dev.type == "cpu"
    gu = torch.empty((num_experts, 2 * moe_intermediate_size, hidden_size), dtype=weight_dtype, device=dev, pin_memory=use_pin)
    dn = torch.empty((num_experts, hidden_size, moe_intermediate_size), dtype=weight_dtype, device=dev, pin_memory=use_pin)

    if weight_dtype.itemsize == 1:
        bm, bn = block_size
        gu_s = torch.empty(
            (num_experts, (2 * moe_intermediate_size + bm - 1) // bm, (hidden_size + bn - 1) // bn),
            dtype=torch.float32,
            device=dev,
            pin_memory=use_pin,
        )
        dn_s = torch.empty(
            (num_experts, (hidden_size + bm - 1) // bm, (moe_intermediate_size + bn - 1) // bn),
            dtype=torch.float32,
            device=dev,
            pin_memory=use_pin,
        )
    else:
        gu_s, dn_s = None, None

    return MoeExpertPayload(gate_up_proj=gu, down_proj=dn, gate_up_scale=gu_s, down_scale=dn_s)


@dataclass
class Qwen38TextConfig:
    """Minimal normalized text configuration extracted from `config.json`."""

    vocab_size: int = 248320
    hidden_size: int = 2560
    num_hidden_layers: int = 48
    hc_count: int = 4
    hc_lowrank: int = 320
    num_attention_heads: int = 24
    num_key_value_heads: int = 2
    head_dim: int = 256
    linear_num_key_heads: int = 16
    linear_num_value_heads: int = 48
    linear_key_head_dim: int = 128
    linear_value_head_dim: int = 128
    linear_conv_kernel_dim: int = 4
    num_experts: int = 512
    num_experts_per_tok: int = 10
    moe_intermediate_size: int = 640
    shared_expert_intermediate_size: int = 640
    norm_topk_prob: bool = True
    rms_norm_eps: float = 1e-6
    output_gate_type: str = "sigmoid"
    rope_theta: float = 10_000_000.0
    partial_rotary_factor: float = 0.25
    indexer_budget: int = 2048
    indexer_compress_ratio: int = 4
    layer_types: list[str] = field(default_factory=list)
    ple_layer_ids: list[int] = field(default_factory=lambda: [2])
    ple_embed_dim: int = 2560
    ple_conv_kernel_size: int = 4
    ngram_size: int = 3
    heads_per_ngram: int = 8
    ngram_vocab_size_base: int = 20_000_000
    make_ngram_vocab_size_divisible_by: int = 128
    seed: int = 1234
    eos_token_id: int = 248044
    bos_token_id: int = 248044
    fp8_block_size: tuple[int, int] = (128, 128)
    is_fp8_experts: bool = True

    @classmethod
    def from_config_dict(cls, raw: dict[str, Any]) -> "Qwen38TextConfig":
        tc = raw.get("text_config", raw)
        qcfg = raw.get("quantization_config") or tc.get("quantization_config") or {}
        rope = tc.get("rope_parameters") or {}
        num_layers = int(tc.get("num_hidden_layers", 48))
        raw_lt = tc.get("layer_types")
        if not raw_lt:
            interval = int(tc.get("full_attention_interval", 4))
            layer_types = [
                "linear_attention" if (i + 1) % interval else "indexed_attention"
                for i in range(num_layers)
            ]
        else:
            layer_types = [
                "indexed_attention" if t == "full_attention" else t for t in raw_lt
            ]
        eos = tc.get("eos_token_id", raw.get("eos_token_id", 248044))
        if isinstance(eos, list):
            eos = eos[0]
        bos = tc.get("bos_token_id", raw.get("bos_token_id", eos))
        if isinstance(bos, list):
            bos = bos[0]
        bs = qcfg.get("weight_block_size", [128, 128])
        is_fp8 = str(qcfg.get("quant_method", "")).lower() == "fp8"
        hidden = int(tc.get("hidden_size", 2560))
        return cls(
            vocab_size=int(tc.get("vocab_size", 248320)),
            hidden_size=hidden,
            num_hidden_layers=num_layers,
            hc_count=int(tc.get("hc_count", 4)),
            hc_lowrank=int(tc.get("hc_lowrank", 320)),
            num_attention_heads=int(tc.get("num_attention_heads", 24)),
            num_key_value_heads=int(tc.get("num_key_value_heads", 2)),
            head_dim=int(tc.get("head_dim", 256)),
            linear_num_key_heads=int(tc.get("linear_num_key_heads", 16)),
            linear_num_value_heads=int(tc.get("linear_num_value_heads", 48)),
            linear_key_head_dim=int(tc.get("linear_key_head_dim", 128)),
            linear_value_head_dim=int(tc.get("linear_value_head_dim", 128)),
            linear_conv_kernel_dim=int(tc.get("linear_conv_kernel_dim", 4)),
            num_experts=int(tc.get("num_experts", 512)),
            num_experts_per_tok=int(tc.get("num_experts_per_tok", 10)),
            moe_intermediate_size=int(tc.get("moe_intermediate_size", 640)),
            shared_expert_intermediate_size=int(tc.get("shared_expert_intermediate_size", 640)),
            norm_topk_prob=bool(tc.get("norm_topk_prob", True)),
            rms_norm_eps=float(tc.get("rms_norm_eps", 1e-6)),
            output_gate_type=str(tc.get("output_gate_type") or tc.get("hidden_act") or "sigmoid"),
            rope_theta=float(rope.get("rope_theta", tc.get("rope_theta", 10_000_000.0))),
            partial_rotary_factor=float(rope.get("partial_rotary_factor", tc.get("partial_rotary_factor", 0.25))),
            indexer_budget=int(tc.get("indexer_budget", 2048)),
            indexer_compress_ratio=int(tc.get("indexer_compress_ratio", 4)),
            layer_types=layer_types,
            ple_layer_ids=[int(x) for x in (tc.get("ple_layer_ids") or [])],
            ple_embed_dim=int(tc.get("ple_embed_dim") or hidden),
            ple_conv_kernel_size=int(tc.get("ple_conv_kernel_size", 4)),
            ngram_size=int(tc.get("ngram_size", 3)),
            heads_per_ngram=int(tc.get("heads_per_ngram", 8)),
            ngram_vocab_size_base=int(tc.get("ngram_vocab_size_base", 20_000_000)),
            make_ngram_vocab_size_divisible_by=int(tc.get("make_ngram_vocab_size_divisible_by", 128)),
            seed=int(tc.get("seed", 1234)),
            eos_token_id=int(eos),
            bos_token_id=int(bos),
            fp8_block_size=(int(bs[0]), int(bs[1])),
            is_fp8_experts=is_fp8,
        )


@dataclass
class LoadedQwen38Model:
    """All dense GPU-resident weights + 3-tier MoE store + PLE provider ready for J-lens fitting."""

    config: Qwen38TextConfig
    plan: OffloadPlan
    device: torch.device
    compute_dtype: torch.dtype

    embed_tokens: torch.Tensor                        # [vocab, D] on GPU
    hyper_connection_mixer: GatedResidualWeights      # final collapse [10240 -> 2560] on GPU
    lm_head: torch.Tensor                             # [vocab, D] on CPU (moved to GPU for apply())
    layers: list[DecoderLayerDenseWeights]            # 48 dense layer weight structs on GPU

    # 3-Tier MoE expert storage:
    #   - Tier 0 (GPU_HBM), Tier 1 (HOST_PINNED / HOST_PAGEABLE): MoeExpertPayload
    #   - Tier 2 (NVME_SLAB): path string to contiguous binary slab file
    moe_tier_store: dict[int, MoeExpertPayload | str]

    # PLE provider (layer_idx -> MmapPleTable | PreextractedPleProvider | InMemoryPleProvider)
    ple_providers: dict[int, Any]


def _normalize_text_key(key: str) -> str | None:
    """Strip multimodal `model.language_model.` / `model.` prefix; return None for vision/MTP keys."""
    if key.startswith("mtp.") or key.startswith("model.visual.") or key.startswith("visual."):
        return None
    if key == "lm_head.weight":
        return "lm_head.weight"
    if key.startswith("model.language_model."):
        return key[len("model.language_model.") :]
    if key.startswith("model."):
        return key[len("model.") :]
    return key


def _safe_delete_hf_file(path: str) -> None:
    """Delete both the symlink and its underlying HF Hub blob so disk space is actually reclaimed."""
    try:
        real = os.path.realpath(path)
        if os.path.islink(path):
            os.unlink(path)
        if os.path.isfile(real):
            os.unlink(real)
        elif os.path.isfile(path):
            os.unlink(path)
    except OSError as exc:
        logger.debug("Could not delete cached shard %s: %s", path, exc)


def _assemble_layer_dense(
    layer_idx: int,
    raw_dense: dict[str, torch.Tensor],
    cfg: Qwen38TextConfig,
    device: torch.device,
    dtype: torch.dtype,
) -> DecoderLayerDenseWeights:
    def get(name: str, cast: torch.dtype | None = dtype) -> torch.Tensor:
        if name not in raw_dense:
            raise KeyError(f"Missing dense tensor for layer {layer_idx}: {name}")
        t = raw_dense.pop(name)
        return t.to(device=device, dtype=cast if cast is not None else t.dtype, non_blocking=True)

    attn_hc = GatedResidualWeights(
        hc_norm_weight=get("attn_hyper_connection.hc_norm.weight"),
        input_mix_weight_down=get("attn_hyper_connection.input_mix_weight_down.weight"),
        input_mix_weight_up=get("attn_hyper_connection.input_mix_weight_up.weight"),
        block_inject_weight=get("attn_hyper_connection.block_inject_weight.weight"),
        hc_count=cfg.hc_count,
        hidden_size=cfg.hidden_size,
        eps=cfg.rms_norm_eps,
    )
    mlp_hc = GatedResidualWeights(
        hc_norm_weight=get("mlp_hyper_connection.hc_norm.weight"),
        input_mix_weight_down=get("mlp_hyper_connection.input_mix_weight_down.weight"),
        input_mix_weight_up=get("mlp_hyper_connection.input_mix_weight_up.weight"),
        block_inject_weight=get("mlp_hyper_connection.block_inject_weight.weight"),
        hc_count=cfg.hc_count,
        hidden_size=cfg.hidden_size,
        eps=cfg.rms_norm_eps,
    )
    moe_dense = MoeDenseWeights(
        router_weight=get("mlp.gate.weight"),
        shared_gate_proj=get("mlp.shared_expert.gate_proj.weight"),
        shared_up_proj=get("mlp.shared_expert.up_proj.weight"),
        shared_down_proj=get("mlp.shared_expert.down_proj.weight"),
        shared_expert_gate=get("mlp.shared_expert_gate.weight"),
        num_experts=cfg.num_experts,
        top_k=cfg.num_experts_per_tok,
        norm_topk_prob=cfg.norm_topk_prob,
        fp8_block_size=cfg.fp8_block_size,
    )

    layer_type = cfg.layer_types[layer_idx]
    gdn = None
    qsa = None
    if layer_type == "linear_attention":
        conv_w = get("linear_attn.conv1d.weight")
        if conv_w.ndim == 3:
            conv_w = conv_w.squeeze(1)
        gdn = GdnLayerWeights(
            in_proj_qkv=get("linear_attn.in_proj_qkv.weight"),
            in_proj_z=get("linear_attn.in_proj_z.weight"),
            in_proj_b=get("linear_attn.in_proj_b.weight"),
            in_proj_a=get("linear_attn.in_proj_a.weight"),
            conv1d_weight=conv_w.contiguous(),
            dt_bias=get("linear_attn.dt_bias", cast=torch.float32),
            A_log=get("linear_attn.A_log", cast=torch.float32),
            norm_weight=get("linear_attn.norm.weight"),
            out_proj=get("linear_attn.out_proj.weight"),
            num_k_heads=cfg.linear_num_key_heads,
            num_v_heads=cfg.linear_num_value_heads,
            head_k_dim=cfg.linear_key_head_dim,
            head_v_dim=cfg.linear_value_head_dim,
            output_gate_type=cfg.output_gate_type,
            eps=cfg.rms_norm_eps,
        )
    else:
        qsa = QsaLayerWeights(
            q_proj=get("self_attn.q_proj.weight"),
            k_proj=get("self_attn.k_proj.weight"),
            v_proj=get("self_attn.v_proj.weight"),
            o_proj=get("self_attn.o_proj.weight"),
            q_norm=get("self_attn.q_norm.weight"),
            k_norm=get("self_attn.k_norm.weight"),
            num_q_heads=cfg.num_attention_heads,
            num_kv_heads=cfg.num_key_value_heads,
            head_dim=cfg.head_dim,
            indexer_budget=cfg.indexer_budget,
            indexer_compress_ratio=cfg.indexer_compress_ratio,
            eps=cfg.rms_norm_eps,
        )
        # Discard unused QSA indexer weights for T <= 2048 (or pop silently)
        for k in list(raw_dense.keys()):
            if k.startswith("self_attn.indexer."):
                raw_dense.pop(k, None)

    ple = None
    if (layer_idx + 1) in cfg.ple_layer_ids:
        ple = PleLayerWeights(
            key_proj=get("ple.key_proj.weight"),
            value_proj=get("ple.value_proj.weight"),
            norm_key=get("ple.norm_key.weight"),
            norm_query=get("ple.norm_query.weight"),
            norm_conv=get("ple.norm_conv.weight"),
            conv1d_weight=get("ple.conv1d.weight"),
            hc_count=cfg.hc_count,
            hidden_size=cfg.hidden_size,
            ngram_size=cfg.ngram_size,
            eps=cfg.rms_norm_eps,
        )

    return DecoderLayerDenseWeights(
        layer_idx=layer_idx,
        layer_type=layer_type,
        attn_hc=attn_hc,
        mlp_hc=mlp_hc,
        moe_dense=moe_dense,
        gdn=gdn,
        qsa=qsa,
        ple=ple,
    )


def load_qwen38_streaming(
    model_id_or_dir: str,
    plan: OffloadPlan,
    *,
    slab_dir: str = "/content/qwen38_moe_slabs",
    ple_dir: str | None = None,
    tokenized_prompts_for_preextract: Sequence[torch.Tensor] | None = None,
    revision: str | None = None,
    hf_token: str | None = None,
    compute_dtype: torch.dtype = torch.bfloat16,
    progress_callback: Callable[[str], None] | None = None,
) -> LoadedQwen38Model:
    """Stream-load Qwen3.8-Flash-Next-FP8 (from HF Hub or local dir) directly into our 3-tier hierarchy.

    Guarantees:
      - Zero Vision/MTP shard downloads.
      - Zero RAM allocated for the 51B PLE n-gram table (mmap or pre-extracted cache).
      - Peak disk stays well under Colab's 235 GiB limit via shard-by-shard pack-and-evict.
    """
    is_local = os.path.isdir(model_id_or_dir)
    device = torch.device("cuda:0") if plan.profile.has_cuda else torch.device("cpu")
    os.makedirs(slab_dir, exist_ok=True)

    def _log(msg: str) -> None:
        logger.info(msg)
        if progress_callback is not None:
            progress_callback(msg)

    def _fetch_file(filename: str, local_dir: str | None = None) -> str:
        if is_local:
            return os.path.join(model_id_or_dir, filename)
        from huggingface_hub import hf_hub_download

        kwargs: dict[str, Any] = {"repo_id": model_id_or_dir, "filename": filename}
        if revision is not None:
            kwargs["revision"] = revision
        if hf_token is not None:
            kwargs["token"] = hf_token
        if local_dir is not None:
            kwargs["local_dir"] = local_dir
        return hf_hub_download(**kwargs)

    # 1. Load config.json and model.safetensors.index.json
    config_path = _fetch_file("config.json")
    with open(config_path, "r", encoding="utf-8") as f:
        raw_cfg = json.load(f)
    cfg = Qwen38TextConfig.from_config_dict(raw_cfg)

    index_path = None
    try:
        index_path = _fetch_file("model.safetensors.index.json")
    except Exception:
        pass

    if index_path is not None and os.path.isfile(index_path):
        with open(index_path, "r", encoding="utf-8") as f:
            weight_map: dict[str, str] = json.load(f)["weight_map"]
    else:
        # Single-file safetensors model (e.g. unit tests / tiny synthetic model)
        single_path = _fetch_file("model.safetensors")
        with safe_open(single_path, framework="pt", device="cpu") as sf:
            weight_map = {k: "model.safetensors" for k in sf.keys()}

    # 2. Classify shards: PLE shards vs text backbone shards vs ignored (vision/MTP)
    shard_to_keys: dict[str, list[str]] = {}
    ple_shard_names: set[str] = set()
    ple_scale_keys: dict[int, tuple[str, str]] = {}  # layer_idx -> (shard_name, raw_key)
    layer_expert_shards: dict[int, set[str]] = {l: set() for l in range(cfg.num_hidden_layers)}
    needed_shards: set[str] = set()

    for raw_key, shard_name in weight_map.items():
        norm_key = _normalize_text_key(raw_key)
        if norm_key is None:
            continue
        shard_to_keys.setdefault(shard_name, []).append(raw_key)
        needed_shards.add(shard_name)

        if _PLE_SHARD_RE.search(norm_key):
            ple_shard_names.add(shard_name)
        elif norm_key.endswith("ngram_embedding.weight_scale"):
            m_l = _LAYER_RE.match(norm_key)
            if m_l:
                ple_scale_keys[int(m_l.group(1))] = (shard_name, raw_key)
        elif norm_key.endswith("ngram_embedding.weight"):
            ple_shard_names.add(shard_name)
        else:
            m_sp = _EXPERT_SPLIT_RE.match(raw_key)
            m_fu = _EXPERT_FUSED_RE.match(raw_key)
            if m_sp:
                layer_expert_shards[int(m_sp.group(1))].add(shard_name)
            elif m_fu:
                layer_expert_shards[int(m_fu.group(1))].add(shard_name)

    total_shards = len(set(weight_map.values()))
    skipped_shards = total_shards - len(needed_shards)
    _log(
        f"Index parsed: {len(needed_shards)}/{total_shards} shards needed "
        f"({len(ple_shard_names)} PLE shards, {skipped_shards} Vision/MTP shards skipped)."
    )

    # 3. Allocate layer buffers & tracking counters
    weight_dtype = torch.float8_e4m3fn if cfg.is_fp8_experts else compute_dtype
    dense_top: dict[str, torch.Tensor] = {}
    dense_layers: dict[int, dict[str, torch.Tensor]] = {l: {} for l in range(cfg.num_hidden_layers)}
    moe_tier_store: dict[int, MoeExpertPayload | str] = {}
    layer_buffers: dict[int, MoeExpertPayload] = {}
    layer_remaining_shards: dict[int, set[str]] = {
        l: set(s_set) for l, s_set in layer_expert_shards.items()
    }

    def _get_or_alloc_layer_buf(layer_idx: int) -> MoeExpertPayload:
        if layer_idx in layer_buffers:
            return layer_buffers[layer_idx]
        tier = plan.layer_tiers[layer_idx]
        if tier == StorageTier.GPU_HBM:
            buf = allocate_moe_payload(
                cfg.num_experts, cfg.hidden_size, cfg.moe_intermediate_size,
                device=device, pin_memory=False, weight_dtype=weight_dtype, block_size=cfg.fp8_block_size,
            )
        elif tier == StorageTier.HOST_PINNED:
            buf = allocate_moe_payload(
                cfg.num_experts, cfg.hidden_size, cfg.moe_intermediate_size,
                device="cpu", pin_memory=plan.profile.has_cuda, weight_dtype=weight_dtype, block_size=cfg.fp8_block_size,
            )
        else:
            # HOST_PAGEABLE or temporary CPU build buffer for NVME_SLAB (flushed & freed once layer completes!)
            buf = allocate_moe_payload(
                cfg.num_experts, cfg.hidden_size, cfg.moe_intermediate_size,
                device="cpu", pin_memory=False, weight_dtype=weight_dtype, block_size=cfg.fp8_block_size,
            )
        layer_buffers[layer_idx] = buf
        return buf

    def _finalize_layer_if_done(layer_idx: int) -> None:
        if layer_idx in moe_tier_store:
            return
        if layer_remaining_shards[layer_idx]:
            return
        buf = layer_buffers.pop(layer_idx)
        tier = plan.layer_tiers[layer_idx]
        if tier == StorageTier.NVME_SLAB:
            slab_path = os.path.join(slab_dir, f"layer_{layer_idx:02d}.q38slab")
            write_moe_slab(slab_path, buf)
            del buf
            gc.collect()
            moe_tier_store[layer_idx] = slab_path
        else:
            moe_tier_store[layer_idx] = buf

    def _consume_non_ple_tensors(shard_name: str, shard_path: str) -> None:
        keys = shard_to_keys.get(shard_name, [])
        touched_layers: set[int] = set()
        with safe_open(shard_path, framework="pt", device="cpu") as sf:
            for raw_key in keys:
                norm_key = _normalize_text_key(raw_key)
                if norm_key is None or _PLE_SHARD_RE.search(norm_key) or norm_key.endswith("ngram_embedding.weight"):
                    continue
                if norm_key.endswith("ngram_embedding.weight_scale"):
                    continue

                m_sp = _EXPERT_SPLIT_RE.match(raw_key)
                if m_sp:
                    l_idx = int(m_sp.group(1))
                    e_idx = int(m_sp.group(2))
                    proj = m_sp.group(3)
                    kind = m_sp.group(4)
                    buf = _get_or_alloc_layer_buf(l_idx)
                    t = sf.get_tensor(raw_key)
                    inter = cfg.moe_intermediate_size
                    if proj == "gate_proj":
                        if kind == "weight":
                            buf.gate_up_proj[e_idx, :inter].copy_(t, non_blocking=True)
                        elif buf.gate_up_scale is not None:
                            s_half = buf.gate_up_scale.shape[1] // 2
                            buf.gate_up_scale[e_idx, :s_half].copy_(t.float(), non_blocking=True)
                    elif proj == "up_proj":
                        if kind == "weight":
                            buf.gate_up_proj[e_idx, inter:].copy_(t, non_blocking=True)
                        elif buf.gate_up_scale is not None:
                            s_half = buf.gate_up_scale.shape[1] // 2
                            buf.gate_up_scale[e_idx, s_half:].copy_(t.float(), non_blocking=True)
                    else:  # down_proj
                        if kind == "weight":
                            buf.down_proj[e_idx].copy_(t, non_blocking=True)
                        elif buf.down_scale is not None:
                            buf.down_scale[e_idx].copy_(t.float(), non_blocking=True)
                    touched_layers.add(l_idx)
                    continue

                m_fu = _EXPERT_FUSED_RE.match(raw_key)
                if m_fu:
                    l_idx = int(m_fu.group(1))
                    field_name = m_fu.group(2)
                    buf = _get_or_alloc_layer_buf(l_idx)
                    t = sf.get_tensor(raw_key)
                    if field_name == "gate_up_proj":
                        buf.gate_up_proj.copy_(t, non_blocking=True)
                    elif field_name == "down_proj":
                        buf.down_proj.copy_(t, non_blocking=True)
                    elif field_name == "gate_up_proj_scale_inv" and buf.gate_up_scale is not None:
                        buf.gate_up_scale.copy_(t.float(), non_blocking=True)
                    elif field_name == "down_proj_scale_inv" and buf.down_scale is not None:
                        buf.down_scale.copy_(t.float(), non_blocking=True)
                    touched_layers.add(l_idx)
                    continue

                # Dense weight (always BF16/FP32)
                t = sf.get_tensor(raw_key)
                m_l = _LAYER_RE.match(norm_key)
                if m_l:
                    l_idx = int(m_l.group(1))
                    sub = m_l.group(2)
                    dense_layers[l_idx][sub] = t
                else:
                    dense_top[norm_key] = t

        for l_idx in touched_layers:
            layer_remaining_shards[l_idx].discard(shard_name)
            _finalize_layer_if_done(l_idx)

    # 4. Stage 1: Handle PLE shards FIRST (so if ple_mode == "preextract", we delete 48 GiB PLE shards BEFORE MoE!)
    ple_providers: dict[int, Any] = {}
    ple_local_paths: list[str] = []
    consumed_shards: set[str] = set()

    if cfg.ple_layer_ids and ple_shard_names:
        _log(f"Downloading / opening {len(ple_shard_names)} PLE shards (mode={plan.ple_mode})...")
        for s_name in sorted(ple_shard_names):
            p = _fetch_file(s_name, local_dir=ple_dir)
            ple_local_paths.append(p)
            # Extract any co-located non-PLE dense/expert tensors from this shard too
            _consume_non_ple_tensors(s_name, p)
            consumed_shards.add(s_name)

        # Also fetch scale shards if separate from PLE shards
        ple_scales: dict[int, float] = {}
        for l_idx, (s_name, r_key) in ple_scale_keys.items():
            sp = _fetch_file(s_name, local_dir=ple_dir if s_name in ple_shard_names else None)
            with safe_open(sp, framework="pt", device="cpu") as sf:
                ple_scales[l_idx] = float(sf.get_tensor(r_key).float().item())
            if s_name not in consumed_shards:
                _consume_non_ple_tensors(s_name, sp)
                consumed_shards.add(s_name)
                if plan.evict_raw_shards and not is_local and s_name not in ple_shard_names:
                    _safe_delete_hf_file(sp)

        for ple_order, one_based_lid in enumerate(cfg.ple_layer_ids):
            l_idx = one_based_lid - 1
            spec = PleHashSpec(
                vocab_size=cfg.vocab_size,
                ngram_size=cfg.ngram_size,
                heads_per_ngram=cfg.heads_per_ngram,
                ngram_vocab_size_base=cfg.ngram_vocab_size_base,
                make_ngram_vocab_size_divisible_by=cfg.make_ngram_vocab_size_divisible_by,
                ple_embed_dim=cfg.ple_embed_dim,
                seed=cfg.seed,
                eos_token_id=cfg.eos_token_id,
                ple_layer_index=ple_order,
            )
            mmap_tbl = MmapPleTable.from_safetensors_files(
                spec, ple_local_paths, weight_scale=ple_scales.get(l_idx)
            )
            if plan.ple_mode == "preextract":
                if tokenized_prompts_for_preextract is None:
                    raise ValueError(
                        "ple_mode='preextract' requires `tokenized_prompts_for_preextract`."
                    )
                _log(f"Pre-extracting PLE embeddings for {len(tokenized_prompts_for_preextract)} prompts...")
                provider = PreextractedPleProvider.from_mmap_and_prompts(
                    mmap_tbl, tokenized_prompts_for_preextract
                )
                mmap_tbl.close()
                ple_providers[l_idx] = provider
            else:
                ple_providers[l_idx] = mmap_tbl

        if plan.ple_mode == "preextract" and not is_local:
            _log("Evicting 48 GiB PLE shards from disk after pre-extraction...")
            for p in ple_local_paths:
                _safe_delete_hf_file(p)

    # 5. Stage 2: Stream remaining backbone shards in layer order (packing Tier 0/1/2 + immediate eviction!)
    remaining_shards = [s for s in sorted(needed_shards) if s not in consumed_shards]
    # Order remaining shards by earliest layer they contribute to -> minimizes simultaneous open layer buffers!
    shard_first_layer: dict[str, int] = {}
    for l_idx in range(cfg.num_hidden_layers):
        for s_name in layer_expert_shards[l_idx]:
            shard_first_layer.setdefault(s_name, l_idx)
    remaining_shards.sort(key=lambda s: (shard_first_layer.get(s, 999), s))

    _log(f"Streaming {len(remaining_shards)} backbone/MoE shards into 3-tier hierarchy...")
    for idx, s_name in enumerate(remaining_shards, start=1):
        sp = _fetch_file(s_name)
        _consume_non_ple_tensors(s_name, sp)
        if plan.evict_raw_shards and not is_local:
            _safe_delete_hf_file(sp)
        if idx % 10 == 0 or idx == len(remaining_shards):
            done_layers = len(moe_tier_store)
            _log(f"  shards {idx}/{len(remaining_shards)} unpacked ({done_layers}/{cfg.num_hidden_layers} MoE layers packed)")

    # Finalize any edge-case layers
    for l_idx in range(cfg.num_hidden_layers):
        _finalize_layer_if_done(l_idx)

    # 6. Assemble top-level & per-layer dense GPU weight structs
    embed_tokens = dense_top.pop("embed_tokens.weight").to(device=device, dtype=compute_dtype)
    if "lm_head.weight" in dense_top:
        lm_head = dense_top.pop("lm_head.weight").to(device="cpu", dtype=compute_dtype).contiguous()
    else:
        lm_head = embed_tokens.detach().cpu().contiguous()

    hc_mixer = GatedResidualWeights(
        hc_norm_weight=dense_top.pop("hyper_connection_mixer.hc_norm.weight").to(device=device, dtype=compute_dtype),
        input_mix_weight_down=dense_top.pop("hyper_connection_mixer.input_mix_weight_down.weight").to(device=device, dtype=compute_dtype),
        input_mix_weight_up=dense_top.pop("hyper_connection_mixer.input_mix_weight_up.weight").to(device=device, dtype=compute_dtype),
        block_inject_weight=None,
        hc_count=cfg.hc_count,
        hidden_size=cfg.hidden_size,
        eps=cfg.rms_norm_eps,
    )

    assembled_layers = [
        _assemble_layer_dense(l_idx, dense_layers[l_idx], cfg, device, compute_dtype)
        for l_idx in range(cfg.num_hidden_layers)
    ]

    if plan.profile.has_cuda:
        torch.cuda.synchronize()
    gc.collect()

    _log("Qwen3.8-Flash-Next 3-tier model load complete!")
    return LoadedQwen38Model(
        config=cfg,
        plan=plan,
        device=device,
        compute_dtype=compute_dtype,
        embed_tokens=embed_tokens,
        hyper_connection_mixer=hc_mixer,
        lm_head=lm_head,
        layers=assembled_layers,
        moe_tier_store=moe_tier_store,
        ple_providers=ple_providers,
    )


In [ ]:
%%writefile qwen38_jlens/checkpoint.py
"""Crash-proof, zero-GPU-stall checkpointing for Qwen3.8-Flash-Next J-Lens fitting.

Why not naive synchronous `torch.save(jacobian_sum, "/content/drive/...")`?
  - 48 layers x [2560, 10240] in fp32 is 4.69 GiB (5.03 GB); atomic temp+rename needs ~10 GB
    of Google Drive quota and blocks the GPU for 80-180 seconds over Drive FUSE!
  - Saving a single `.safetensors` file storing the running MEAN `J_mean = jacobian_sum / n_done`
    in fp16 (2.34 GiB, or optional fp32 4.69 GiB) with metadata embedded in the safetensors header:
      1. Cuts checkpoint size & Drive/HF upload time in half (2.34 GiB vs 4.69 GiB) with zero
         fp16 overflow risk regardless of `n_prompts` (because we store the mean, not the sum!).
      2. Makes EVERY in-progress checkpoint immediately loadable as a valid `Qwen38JacobianLens`!
      3. Writes to local NVMe in <1 second, then syncs asynchronously on a background thread to
         Google Drive and/or Hugging Face Hub (free unlimited storage, ~300-800 MB/s Xet upload)
         with ZERO GPU idle time!
      4. On resume, inspects local NVMe, Google Drive, and HF Hub headers (<1 ms via safetensors
         metadata) and automatically resumes from whichever copy has the highest `next_idx`!
"""

from __future__ import annotations

import json
import logging
import os
import shutil
import time
from concurrent.futures import Future, ThreadPoolExecutor
from dataclasses import dataclass
from typing import Any, Sequence

import torch
from safetensors import safe_open
from safetensors.torch import save_file

logger = logging.getLogger(__name__)

_DTYPE_MAP = {
    "float16": torch.float16,
    "fp16": torch.float16,
    "bfloat16": torch.bfloat16,
    "bf16": torch.bfloat16,
    "float32": torch.float32,
    "fp32": torch.float32,
}


def resolve_ckpt_dtype(dtype: str | torch.dtype) -> torch.dtype:
    if isinstance(dtype, torch.dtype):
        return dtype
    key = dtype.strip().lower()
    if key not in _DTYPE_MAP:
        raise ValueError(f"Unsupported checkpoint dtype {dtype!r}; choose from {sorted(_DTYPE_MAP)}")
    return _DTYPE_MAP[key]


def peek_checkpoint_metadata(path: str) -> dict[str, Any] | None:
    """Read checkpoint metadata from a `.safetensors` header in <1 ms (without loading 2-5 GiB weights).

    Also supports legacy `.pt` checkpoints transparently.
    """
    if not path or not os.path.isfile(path):
        return None
    if path.endswith(".safetensors"):
        try:
            with safe_open(path, framework="pt", device="cpu") as sf:
                raw_meta = sf.metadata() or {}
            if "n_done" not in raw_meta:
                return None
            return {
                "n_done": int(raw_meta["n_done"]),
                "next_idx": int(raw_meta.get("next_idx", raw_meta["n_done"])),
                "source_layers": json.loads(raw_meta["source_layers"]),
                "skip_first": int(raw_meta["skip_first"]),
                "d_model": int(raw_meta["d_model"]),
                "hc_count": int(raw_meta["hc_count"]),
                "model_name": raw_meta.get("model_name", "Qwen/Qwen3.8-Flash-Next-FP8"),
                "ckpt_dtype": raw_meta.get("ckpt_dtype", "float16"),
                "format": "safetensors",
            }
        except Exception as exc:
            logger.warning("Failed to peek safetensors checkpoint %s: %s", path, exc)
            return None

    # Legacy `.pt` fallback
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
        if "n_done" not in state:
            return None
        return {
            "n_done": int(state["n_done"]),
            "next_idx": int(state.get("next_idx", state["n_done"])),
            "source_layers": list(state["source_layers"]),
            "skip_first": int(state["skip_first"]),
            "d_model": int(state["d_model"]),
            "hc_count": int(state.get("hc_count", 4)),
            "model_name": state.get("model_name", "Qwen/Qwen3.8-Flash-Next-FP8"),
            "ckpt_dtype": "float32",
            "format": "pt",
        }
    except Exception as exc:
        logger.warning("Failed to peek legacy pt checkpoint %s: %s", path, exc)
        return None


def save_jlens_safetensors(
    path: str,
    jacobian_mean: dict[int, torch.Tensor],
    *,
    n_done: int,
    next_idx: int,
    skip_first: int,
    d_model: int,
    hc_count: int,
    model_name: str = "Qwen/Qwen3.8-Flash-Next-FP8",
    dtype: torch.dtype = torch.float16,
    include_collapsed: bool = False,
    divisor: float = 1.0,
) -> str:
    """Atomically write a single-file `.safetensors` J-lens / resumable checkpoint.

    Pass `divisor=float(n_done)` when supplying `jacobian_sum` to divide one layer at a time
    (avoiding a 4.69 GiB transient fp32 host RAM spike on 83.5 GiB A100 40G runtimes).
    """
    parent = os.path.dirname(os.path.abspath(path))
    if parent:
        os.makedirs(parent, exist_ok=True)

    source_layers = sorted(int(l) for l in jacobian_mean.keys())
    tensors: dict[str, torch.Tensor] = {}
    branch_norms: dict[str, list[float]] = {}
    scale = d_model ** 0.5
    inv_div = 1.0 / float(divisor) if divisor != 1.0 else 1.0

    for l in source_layers:
        raw = jacobian_mean[l].detach().cpu()
        J_f = (raw.float() * inv_div) if divisor != 1.0 else raw.float()
        J4 = J_f.view(d_model, hc_count, d_model)
        branch_norms[str(l)] = [round(float(J4[:, b, :].norm().item() / scale), 5) for b in range(hc_count)]
        tensors[f"J.layer_{l:02d}"] = J_f.to(dtype=dtype).contiguous()
        if include_collapsed:
            tensors[f"J_collapsed.layer_{l:02d}"] = J4.sum(dim=1).to(dtype=dtype).contiguous()
        del J_f, J4

    metadata = {
        "format_version": "qwen38_jlens_v1",
        "model_name": str(model_name),
        "n_done": str(int(n_done)),
        "next_idx": str(int(next_idx)),
        "source_layers": json.dumps(source_layers),
        "skip_first": str(int(skip_first)),
        "d_model": str(int(d_model)),
        "hc_count": str(int(hc_count)),
        "d_source": str(int(d_model * hc_count)),
        "ckpt_dtype": str(dtype).replace("torch.", ""),
        "branch_norms": json.dumps(branch_norms),
    }

    tmp_path = f"{path}.tmp.{os.getpid()}"
    save_file(tensors, tmp_path, metadata=metadata)
    os.replace(tmp_path, path)
    return path


def load_jlens_safetensors(path: str) -> tuple[dict[int, torch.Tensor], dict[str, Any]]:
    """Load `jacobian_mean` (`{layer: fp32 CPU Tensor}`) and metadata from a `.safetensors` file."""
    with safe_open(path, framework="pt", device="cpu") as sf:
        raw_meta = sf.metadata() or {}
        source_layers = json.loads(raw_meta["source_layers"])
        jacobian_mean: dict[int, torch.Tensor] = {}
        for l in source_layers:
            key = f"J.layer_{int(l):02d}"
            jacobian_mean[int(l)] = sf.get_tensor(key).float()

    meta = {
        "n_done": int(raw_meta["n_done"]),
        "next_idx": int(raw_meta.get("next_idx", raw_meta["n_done"])),
        "source_layers": [int(x) for x in source_layers],
        "skip_first": int(raw_meta.get("skip_first", 16)),
        "d_model": int(raw_meta["d_model"]),
        "hc_count": int(raw_meta.get("hc_count", 4)),
        "model_name": raw_meta.get("model_name", "Qwen/Qwen3.8-Flash-Next-FP8"),
        "ckpt_dtype": raw_meta.get("ckpt_dtype", "float16"),
    }
    return jacobian_mean, meta


@dataclass
class CheckpointConfig:
    """Multi-destination asynchronous checkpointing policy."""

    local_path: str                           # Fast local NVMe `.safetensors` path (~0.8s write!)
    drive_path: str | None = None             # Optional Google Drive mirror path (copied asynchronously)
    hf_repo_id: str | None = None             # Optional Hugging Face Hub repo (e.g. "user/qwen38-jlens")
    hf_path_in_repo: str | None = None        # Filename inside HF repo (defaults to basename of local_path)
    hf_token: str | None = None
    hf_private: bool = True
    ckpt_dtype: str = "float16"               # "float16" (2.34 GiB, default) or "float32" (4.69 GiB exact)
    every_n_prompts: int = 10                 # Trigger checkpoint every N prompts
    every_minutes: float = 10.0               # OR every M minutes (whichever comes first!)
    max_sync_retries: int = 3
    model_name: str = "Qwen/Qwen3.8-Flash-Next-FP8"


class AsyncCheckpointManager:
    """Non-blocking checkpoint manager: fast local NVMe write + background Drive & HF Hub sync.

    Guarantees:
      - Zero GPU stalls waiting on Google Drive FUSE or HF Hub network uploads.
      - Never piles up queued background uploads (if a previous remote sync is still running,
        local NVMe is still updated and the next completed window triggers the remote sync;
        `flush()` waits cleanly on completion or `KeyboardInterrupt`).
      - Multi-source resume automatically chooses the freshest checkpoint across Local NVMe,
        Google Drive, and Hugging Face Hub!
    """

    def __init__(self, cfg: CheckpointConfig) -> None:
        self.cfg = cfg
        self.dtype = resolve_ckpt_dtype(cfg.ckpt_dtype)
        self._pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="jlens-ckpt")
        self._remote_future: Future | None = None
        self._last_ckpt_time = time.monotonic()
        self._last_local_next_idx = -1
        self._last_synced_next_idx = -1
        self._hf_repo_ready = False
        self.last_status: str = "idle"

    @property
    def has_remote(self) -> bool:
        return bool(self.cfg.drive_path or self.cfg.hf_repo_id)

    def should_checkpoint(self, prompts_since_last: int, *, force: bool = False) -> bool:
        if force:
            return True
        if self.cfg.every_n_prompts > 0 and prompts_since_last >= self.cfg.every_n_prompts:
            return True
        if self.cfg.every_minutes > 0 and (time.monotonic() - self._last_ckpt_time) >= self.cfg.every_minutes * 60.0:
            return True
        return False

    def _copy_to_drive(self, src_path: str, dst_path: str) -> None:
        """Atomic copy to Google Drive FUSE with fallback + retry for transient FUSE errors."""
        parent = os.path.dirname(os.path.abspath(dst_path))
        os.makedirs(parent, exist_ok=True)
        tmp_dst = f"{dst_path}.uploading"
        last_exc: Exception | None = None
        for attempt in range(1, self.cfg.max_sync_retries + 1):
            try:
                shutil.copyfile(src_path, tmp_dst)
                os.replace(tmp_dst, dst_path)
                return
            except Exception as exc:
                last_exc = exc
                logger.warning("Drive checkpoint sync attempt %d/%d failed: %s", attempt, self.cfg.max_sync_retries, exc)
                time.sleep(min(5.0, attempt * 2.0))
        if last_exc is not None:
            raise last_exc

    def _upload_to_hf(self, src_path: str, next_idx: int, n_done: int) -> None:
        """Upload checkpoint to Hugging Face Hub (Xet chunk-deduplicated, free unlimited storage)."""
        assert self.cfg.hf_repo_id is not None
        from huggingface_hub import HfApi

        api = HfApi(token=self.cfg.hf_token)
        if not self._hf_repo_ready:
            api.create_repo(
                repo_id=self.cfg.hf_repo_id,
                repo_type="model",
                private=self.cfg.hf_private,
                exist_ok=True,
            )
            self._hf_repo_ready = True

        path_in_repo = self.cfg.hf_path_in_repo or os.path.basename(self.cfg.local_path)
        last_exc: Exception | None = None
        for attempt in range(1, self.cfg.max_sync_retries + 1):
            try:
                api.upload_file(
                    path_or_fileobj=src_path,
                    path_in_repo=path_in_repo,
                    repo_id=self.cfg.hf_repo_id,
                    repo_type="model",
                    commit_message=f"Checkpoint J-lens (n_done={n_done}, next_idx={next_idx})",
                )
                return
            except Exception as exc:
                last_exc = exc
                logger.warning("HF Hub checkpoint upload attempt %d/%d failed: %s", attempt, self.cfg.max_sync_retries, exc)
                time.sleep(min(8.0, attempt * 2.5))
        if last_exc is not None:
            raise last_exc

    def _sync_remote_task(self, staged_path: str, next_idx: int, n_done: int) -> None:
        t0 = time.perf_counter()
        targets: list[str] = []
        try:
            if self.cfg.drive_path:
                self._copy_to_drive(staged_path, self.cfg.drive_path)
                targets.append("Drive")
            if self.cfg.hf_repo_id:
                self._upload_to_hf(staged_path, next_idx, n_done)
                targets.append(f"HF:{self.cfg.hf_repo_id}")
            self._last_synced_next_idx = next_idx
            dt = time.perf_counter() - t0
            self.last_status = f"synced@{next_idx} ({'+'.join(targets)} in {dt:.1f}s)"
            logger.info("Async remote checkpoint synced: %s", self.last_status)
        except Exception as exc:
            self.last_status = f"remote_err@{next_idx}: {exc}"
            logger.error("Async remote checkpoint sync failed (local NVMe copy is still safe!): %s", exc)
        finally:
            # Remove immutable staging hardlink/copy once remote sync finishes
            if staged_path != self.cfg.local_path and os.path.exists(staged_path):
                try:
                    os.unlink(staged_path)
                except OSError:
                    pass

    def save_async(
        self,
        jacobian_sum: dict[int, torch.Tensor],
        *,
        n_done: int,
        next_idx: int,
        skip_first: int,
        d_model: int,
        hc_count: int,
        wait_remote: bool = False,
    ) -> None:
        """Write local NVMe `.safetensors` immediately (~0.8s), then push to Drive/HF in background."""
        if n_done <= 0:
            return

        if self._last_local_next_idx != next_idx or not os.path.isfile(self.cfg.local_path):
            t0 = time.perf_counter()
            save_jlens_safetensors(
                self.cfg.local_path,
                jacobian_sum,
                n_done=n_done,
                next_idx=next_idx,
                skip_first=skip_first,
                d_model=d_model,
                hc_count=hc_count,
                model_name=self.cfg.model_name,
                dtype=self.dtype,
                include_collapsed=False,
                divisor=float(n_done),
            )
            self._last_local_next_idx = next_idx
            self._last_ckpt_time = time.monotonic()
            dt_local = time.perf_counter() - t0
            self.last_status = f"local@{next_idx} ({dt_local:.1f}s)"
            logger.info("Local NVMe checkpoint saved: %s (n_done=%d, next_idx=%d, %.2fs)", self.cfg.local_path, n_done, next_idx, dt_local)

        if not self.has_remote:
            return

        # If a previous background upload is still running:
        #   - during fitting (`wait_remote=False`), don't pile up uploads; local copy is already fresh!
        #   - on final/interrupt flush (`wait_remote=True`), wait for previous upload then launch final sync.
        if self._remote_future is not None and not self._remote_future.done():
            if not wait_remote:
                logger.info("Previous remote sync still in flight; keeping local NVMe checkpoint fresh without stalling GPU.")
                return
            self._remote_future.result()

        # Hardlink local file to an immutable staging path (0 extra disk space, 0 ms!) so
        # subsequent local writes never clobber bytes being read by the background upload thread!
        staged_path = f"{self.cfg.local_path}.sync.{next_idx}"
        if os.path.exists(staged_path):
            os.unlink(staged_path)
        try:
            os.link(self.cfg.local_path, staged_path)
        except OSError:
            shutil.copyfile(self.cfg.local_path, staged_path)

        self._remote_future = self._pool.submit(self._sync_remote_task, staged_path, next_idx, n_done)
        if wait_remote:
            self._remote_future.result()

    def load_best_resume(
        self,
        *,
        expected_source_layers: Sequence[int],
        expected_skip_first: int,
    ) -> tuple[dict[int, torch.Tensor], int, int] | None:
        """Inspect Local NVMe, Google Drive, and HF Hub; resume from whichever has the highest `next_idx`."""
        candidates: list[tuple[int, str, str]] = []  # (next_idx, source_label, local_file_path)

        for label, path in (("local", self.cfg.local_path), ("drive", self.cfg.drive_path)):
            if path and os.path.isfile(path):
                meta = peek_checkpoint_metadata(path)
                if meta is not None:
                    candidates.append((meta["next_idx"], label, path))

        if self.cfg.hf_repo_id:
            try:
                from huggingface_hub import hf_hub_download

                path_in_repo = self.cfg.hf_path_in_repo or os.path.basename(self.cfg.local_path)
                dl_path = hf_hub_download(
                    repo_id=self.cfg.hf_repo_id,
                    filename=path_in_repo,
                    repo_type="model",
                    token=self.cfg.hf_token,
                )
                meta = peek_checkpoint_metadata(dl_path)
                if meta is not None:
                    candidates.append((meta["next_idx"], f"hf:{self.cfg.hf_repo_id}", dl_path))
            except Exception as exc:
                logger.info("No resumable checkpoint found on HF Hub %s (%s).", self.cfg.hf_repo_id, exc)

        if not candidates:
            return None

        # Pick the candidate with the largest `next_idx`
        candidates.sort(key=lambda item: item[0], reverse=True)
        best_next, best_label, best_path = candidates[0]
        logger.info("Selected freshest checkpoint from %s (%s, next_idx=%d).", best_label, best_path, best_next)

        if best_path.endswith(".safetensors"):
            j_mean, meta = load_jlens_safetensors(best_path)
            if list(meta["source_layers"]) != list(expected_source_layers) or meta["skip_first"] != expected_skip_first:
                raise ValueError(
                    f"Checkpoint at {best_path} has incompatible source_layers/skip_first "
                    f"({meta['source_layers']}/{meta['skip_first']} vs {list(expected_source_layers)}/{expected_skip_first}); "
                    "pass resume=False to discard it."
                )
            n_done = meta["n_done"]
            next_idx = meta["next_idx"]
            jacobian_sum = {l: (J * float(n_done)) for l, J in j_mean.items()}
            return jacobian_sum, n_done, next_idx

        # Legacy `.pt` resume
        state = torch.load(best_path, map_location="cpu", weights_only=True)
        if state.get("source_layers") != list(expected_source_layers) or state.get("skip_first") != expected_skip_first:
            raise ValueError(f"Legacy checkpoint at {best_path} has incompatible source_layers/skip_first.")
        return state["jacobian_sum"], int(state["n_done"]), int(state["next_idx"])

    def flush(self) -> None:
        """Wait for any in-flight remote upload to finish cleanly."""
        if self._remote_future is not None:
            self._remote_future.result()

    def close(self) -> None:
        self.flush()
        self._pool.shutdown(wait=True)


In [ ]:
%%writefile qwen38_jlens/lens.py
"""Qwen38JacobianLens container, readout evaluation, and interactive HTML slice visualizer.

Compatible with `anthropics/jacobian-lens` (`jlens`) while natively supporting Qwen3.8-Flash-Next's
4-branch Gated Residual (`hc_count = 4`, residual stream width `10240 -> 2560`).

Format saved to `.pt`:
  {
    "J": {layer_idx: Tensor[2560, 10240] (fp16)},
    "J_collapsed": {layer_idx: Tensor[2560, 2560] (fp16)},   # sum across 4 branches (identical-perturbation J)
    "branch_norms": {layer_idx: [norm_b0, norm_b1, norm_b2, norm_b3]},
    "n_prompts": int,
    "source_layers": list[int],
    "d_model": 2560,
    "d_source": 10240,
    "hc_count": 4,
    "model_name": str,
  }
"""

from __future__ import annotations

import html as _html
import json
import os
from typing import Any, Sequence

import torch
import torch.nn.functional as F


class Qwen38JacobianLens:
    """Fitted Jacobian Lens for Qwen3.8-Flash-Next (`qwen4_exp`, 4-branch Gated Residual)."""

    def __init__(
        self,
        jacobians: dict[int, torch.Tensor],
        *,
        n_prompts: int,
        d_model: int = 2560,
        hc_count: int = 4,
        model_name: str = "Qwen/Qwen3.8-Flash-Next-FP8",
    ) -> None:
        self.jacobians = {int(layer): J.float().cpu() for layer, J in jacobians.items()}
        self.source_layers = sorted(self.jacobians.keys())
        self.n_prompts = int(n_prompts)
        self.d_model = int(d_model)
        self.hc_count = int(hc_count)
        self.d_source = self.d_model * self.hc_count
        self.model_name = model_name

    def __repr__(self) -> str:
        rng = f"{self.source_layers[0]}..{self.source_layers[-1]}" if self.source_layers else "empty"
        return (
            f"Qwen38JacobianLens(model={self.model_name!r}, d_out={self.d_model}, "
            f"d_src={self.d_source} ({self.hc_count}x{self.d_model}), "
            f"n_prompts={self.n_prompts}, layers=[{rng}] ({len(self.source_layers)}))"
        )

    def branch_norms(self) -> dict[int, list[float]]:
        """Per-branch Frobenius norms `||J_l[:, b, :]||_F / sqrt(d_model)` across the 4 GR branches."""
        scale = self.d_model ** 0.5
        out: dict[int, list[float]] = {}
        for l in self.source_layers:
            J4 = self.jacobians[l].view(self.d_model, self.hc_count, self.d_model)
            out[l] = [float(J4[:, b, :].norm().item() / scale) for b in range(self.hc_count)]
        return out

    def collapsed_jacobians(self) -> dict[int, torch.Tensor]:
        """Sum across the 4 branches: `J_sum[l] = sum_{b=0..3} d h_final / d H_l^(b)` (`[d_model, d_model]`)."""
        return {
            l: self.jacobians[l].view(self.d_model, self.hc_count, self.d_model).sum(dim=1)
            for l in self.source_layers
        }

    def transport(self, H_l: torch.Tensor, layer: int) -> torch.Tensor:
        """Transport 4-branch residual `H_l` (`[..., hc_count * d_model]`) -> `[..., d_model]`."""
        J = self.jacobians[layer].to(device=H_l.device, dtype=torch.float32)
        return F.linear(H_l.float(), J)

    def save(self, path: str, *, dtype: torch.dtype = torch.float16, include_collapsed: bool = True) -> None:
        """Atomically save fitted lens to `.safetensors` (recommended, fast, zero-pickle) or `.pt`."""
        if path.endswith(".safetensors"):
            from qwen38_jlens.checkpoint import save_jlens_safetensors

            save_jlens_safetensors(
                path,
                self.jacobians,
                n_done=self.n_prompts,
                next_idx=self.n_prompts,
                skip_first=16,
                d_model=self.d_model,
                hc_count=self.hc_count,
                model_name=self.model_name,
                dtype=dtype,
                include_collapsed=include_collapsed,
            )
            return

        parent = os.path.dirname(os.path.abspath(path))
        if parent:
            os.makedirs(parent, exist_ok=True)
        payload: dict[str, Any] = {
            "J": {l: J.to(dtype) for l, J in self.jacobians.items()},
            "branch_norms": self.branch_norms(),
            "n_prompts": self.n_prompts,
            "source_layers": self.source_layers,
            "d_model": self.d_model,
            "d_source": self.d_source,
            "hc_count": self.hc_count,
            "model_name": self.model_name,
        }
        if include_collapsed:
            payload["J_collapsed"] = {l: J.to(dtype) for l, J in self.collapsed_jacobians().items()}
        tmp = f"{path}.tmp.{os.getpid()}"
        torch.save(payload, tmp)
        os.replace(tmp, path)

    @classmethod
    def load(cls, path: str) -> "Qwen38JacobianLens":
        """Load a saved lens or ANY in-progress `.safetensors` checkpoint directly as a usable lens!"""
        if path.endswith(".safetensors"):
            from qwen38_jlens.checkpoint import load_jlens_safetensors

            jacobians, meta = load_jlens_safetensors(path)
            return cls(
                jacobians=jacobians,
                n_prompts=meta["n_done"],
                d_model=meta["d_model"],
                hc_count=meta["hc_count"],
                model_name=meta["model_name"],
            )

        ckpt = torch.load(path, map_location="cpu", weights_only=True)
        if "J" not in ckpt:
            raise ValueError(f"{path} is not a saved Qwen38JacobianLens file (keys={sorted(ckpt)})")
        return cls(
            jacobians=ckpt["J"],
            n_prompts=ckpt["n_prompts"],
            d_model=ckpt["d_model"],
            hc_count=ckpt.get("hc_count", ckpt["J"][ckpt["source_layers"][0]].shape[1] // ckpt["d_model"]),
            model_name=ckpt.get("model_name", "Qwen/Qwen3.8-Flash-Next-FP8"),
        )

    @classmethod
    def merge(cls, lenses: Sequence["Qwen38JacobianLens"]) -> "Qwen38JacobianLens":
        """Combine lenses fitted on disjoint prompt slices (n_prompts-weighted mean)."""
        if not lenses:
            raise ValueError("merge() requires at least one lens.")
        first = lenses[0]
        for other in lenses[1:]:
            if (
                other.source_layers != first.source_layers
                or other.d_model != first.d_model
                or other.hc_count != first.hc_count
            ):
                raise ValueError("Cannot merge lenses with mismatched layers / d_model / hc_count.")
        n_total = sum(l.n_prompts for l in lenses)
        merged = {}
        for layer in first.source_layers:
            acc = sum(l.jacobians[layer] * l.n_prompts for l in lenses)
            merged[layer] = acc / n_total
        return cls(
            jacobians=merged,
            n_prompts=n_total,
            d_model=first.d_model,
            hc_count=first.hc_count,
            model_name=first.model_name,
        )


def render_slice_html(
    tokens: Sequence[str],
    layer_top1: dict[int, list[tuple[str, float]]],
    model_top1: list[tuple[str, float]],
    title: str = "Qwen3.8-Flash-Next Jacobian Lens Slice",
) -> str:
    """Build a lightweight, self-contained interactive HTML layer x position heatmap for Colab display."""
    layers = sorted(layer_top1.keys())
    rows_html = []
    for l in layers:
        cells = []
        for tok_str, prob in layer_top1[l]:
            alpha = min(0.85, max(0.08, prob))
            bg = f"rgba(56, 139, 253, {alpha:.2f})"
            esc = _html.escape(tok_str.replace("\n", "\\n"))
            cells.append(
                f'<td style="background:{bg};padding:4px 6px;border:1px solid #2d333b;'
                f'font-family:monospace;font-size:12px;white-space:nowrap;" '
                f'title="L{l} p={prob:.3f}">{esc}<sup style="opacity:.7">{int(prob*100)}%</sup></td>'
            )
        rows_html.append(
            f'<tr><th style="position:sticky;left:0;background:#161b22;color:#8b949e;padding:4px 8px;'
            f'border:1px solid #2d333b;font-family:monospace;font-size:12px;">L{l}</th>{"".join(cells)}</tr>'
        )
    # Final model output row
    m_cells = []
    for tok_str, prob in model_top1:
        alpha = min(0.90, max(0.12, prob))
        bg = f"rgba(63, 185, 80, {alpha:.2f})"
        esc = _html.escape(tok_str.replace("\n", "\\n"))
        m_cells.append(
            f'<td style="background:{bg};padding:4px 6px;border:1px solid #2d333b;'
            f'font-family:monospace;font-size:12px;font-weight:bold;white-space:nowrap;" '
            f'title="Model p={prob:.3f}">{esc}<sup style="opacity:.7">{int(prob*100)}%</sup></td>'
        )
    rows_html.append(
        f'<tr><th style="position:sticky;left:0;background:#161b22;color:#3fb950;padding:4px 8px;'
        f'border:1px solid #2d333b;font-family:monospace;font-size:12px;">OUT</th>{"".join(m_cells)}</tr>'
    )

    header_cells = "".join(
        f'<th style="background:#0d1117;color:#c9d1d9;padding:5px 6px;border:1px solid #2d333b;'
        f'font-family:monospace;font-size:12px;white-space:nowrap;">{_html.escape(t.replace(chr(10), "⏎"))}</th>'
        for t in tokens
    )
    return (
        f'<div style="background:#0d1117;color:#e6edf3;padding:12px;border-radius:8px;overflow-x:auto;">'
        f'<div style="font-family:sans-serif;font-weight:600;margin-bottom:8px;">{_html.escape(title)}</div>'
        f'<table style="border-collapse:collapse;">'
        f'<thead><tr><th style="background:#0d1117;color:#8b949e;padding:5px 8px;border:1px solid #2d333b;">Layer</th>'
        f'{header_cells}</tr></thead><tbody>{"".join(rows_html)}</tbody></table></div>'
    )


In [ ]:
%%writefile qwen38_jlens/engine.py
"""Async double-buffered 3-tier weight streamer and single-sweep layer-by-layer J-lens engine.

Architecture of one prompt pass (`jacobian_for_prompt_fast`):
  1. **Forward sweep (layers 0 -> L-1, B = 1)**:
     - Streams offloaded MoE layers asynchronously via `AsyncMoEStreamer`.
     - Saves ONLY each layer's input residual `H_{l-1}` (`[1, T, 10240]` = 2.5 MB/layer -> 120 MB total!).
     - Computes `h_final = hyper_connection_mixer(H_{L-1})`.
  2. **Backward sweep (layers L-1 -> min_source + 1, ALL K = k_pass cotangents at once!)**:
     - Initializes ONE in-place GPU cotangent buffer `G` (`[k_pass, T, 10240]`, 6.25 GiB for full K=2560!).
     - At each layer `l` (going downward):
       a) Async prefetch kicks off layer `l - 1` (H2D DMA) AND the next Tier-2 NVMe slab (background thread).
       b) Recomputes ONE tiny `B = 1` forward of layer `l` (~0.5 ms), dequantizing ONLY the hit active
          experts ONCE into BF16.
       c) Executes exact analytical VJP for ALL `k_pass` cotangents (`decoder_layer_vjp_inplace`) updating
          `G` in-place while layer `l` sits hot in GPU HBM!
       d) Immediately reduces `J_{l-1} = G[:, valid_positions, :].float().mean(dim=1)` onto CPU!
     - **Result**: Each offloaded layer's FP8 weights cross PCIe/NVMe AT MOST ONCE per sweep
       (vs 320x in Anthropic's reference `dim_batch=8` loop -> 40x-120x faster!).
"""

from __future__ import annotations

import logging
import math
import os
import time
from concurrent.futures import Future, ThreadPoolExecutor
from dataclasses import dataclass
from typing import Any, Sequence

import torch
import torch.nn.functional as F

from qwen38_jlens.hardware import StorageTier
from qwen38_jlens.kernels import GatedDeltaRuleBackend
from qwen38_jlens.lens import Qwen38JacobianLens
from qwen38_jlens.loader import (
    LoadedQwen38Model,
    allocate_moe_payload,
    read_moe_slab_into,
)
from qwen38_jlens.model_ops import (
    MoeExpertPayload,
    build_rope_cos_sin,
    decoder_layer_fwd,
    decoder_layer_vjp_inplace,
    gated_residual_fwd,
    gated_residual_vjp,
)

logger = logging.getLogger(__name__)

SKIP_FIRST_N_POSITIONS = 16


def valid_position_mask(seq_len: int, *, skip_first: int = SKIP_FIRST_N_POSITIONS) -> torch.Tensor:
    """Boolean mask `[seq_len]` matching `anthropics/jacobian-lens` (`[skip_first : seq_len - 1]`)."""
    if skip_first < 0:
        raise ValueError(f"skip_first must be >= 0, got {skip_first}")
    mask = torch.zeros(seq_len, dtype=torch.bool)
    mask[skip_first : seq_len - 1] = True
    if int(mask.sum().item()) == 0:
        raise ValueError(f"prompt too short: seq_len={seq_len}, need > {skip_first + 1} tokens")
    return mask


def _copy_payload_async(dst: MoeExpertPayload, src: MoeExpertPayload, non_blocking: bool) -> None:
    dst.gate_up_proj.copy_(src.gate_up_proj, non_blocking=non_blocking)
    dst.down_proj.copy_(src.down_proj, non_blocking=non_blocking)
    if dst.gate_up_scale is not None and src.gate_up_scale is not None:
        dst.gate_up_scale.copy_(src.gate_up_scale, non_blocking=non_blocking)
    if dst.down_scale is not None and src.down_scale is not None:
        dst.down_scale.copy_(src.down_scale, non_blocking=non_blocking)


class AsyncMoEStreamer:
    """Zero-stall double-buffered 3-tier MoE expert weight streamer.

    Manages:
      - Tier 0 (`GPU_HBM`): zero-copy direct return.
      - Tier 1 (`HOST_PINNED`): async H2D DMA on dedicated `h2d_stream` into 2-slot GPU staging ring.
      - Tier 1 (`HOST_PAGEABLE`) & Tier 2 (`NVME_SLAB`): background I/O thread populates 2-slot
        pinned host staging ring (overlapped across interleaved Tier 0/1 layers!), then `h2d_stream`
        DMAs into the 2-slot GPU staging ring!
    """

    def __init__(self, model: LoadedQwen38Model) -> None:
        self.model = model
        self.plan = model.plan
        self.cfg = model.config
        self.device = model.device
        self.has_cuda = self.device.type == "cuda"

        weight_dtype = torch.float8_e4m3fn if self.cfg.is_fp8_experts else model.compute_dtype

        self._gpu_slots: list[MoeExpertPayload] = [
            allocate_moe_payload(
                self.cfg.num_experts,
                self.cfg.hidden_size,
                self.cfg.moe_intermediate_size,
                device=self.device,
                pin_memory=False,
                weight_dtype=weight_dtype,
                block_size=self.cfg.fp8_block_size,
            )
            for _ in range(self.plan.gpu_staging_slots)
        ]
        self._host_slots: list[MoeExpertPayload] = [
            allocate_moe_payload(
                self.cfg.num_experts,
                self.cfg.hidden_size,
                self.cfg.moe_intermediate_size,
                device="cpu",
                pin_memory=self.has_cuda,
                weight_dtype=weight_dtype,
                block_size=self.cfg.fp8_block_size,
            )
            for _ in range(self.plan.host_pinned_staging_slots)
        ]

        if self.has_cuda:
            self._h2d_stream = torch.cuda.Stream(device=self.device)
            self._gpu_ready = [torch.cuda.Event() for _ in self._gpu_slots]
            self._gpu_free = [torch.cuda.Event() for _ in self._gpu_slots]
            self._host_free = [torch.cuda.Event() for _ in self._host_slots]
        else:
            self._h2d_stream = None
            self._gpu_ready = []
            self._gpu_free = []
            self._host_free = []

        self._io_pool = ThreadPoolExecutor(max_workers=1)
        # CPU-only scratch buffer for NVMe slabs
        self._cpu_nvme_buf: MoeExpertPayload | None = None
        if not self.has_cuda and any(
            t == StorageTier.NVME_SLAB for t in self.plan.layer_tiers.values()
        ):
            self._cpu_nvme_buf = allocate_moe_payload(
                self.cfg.num_experts,
                self.cfg.hidden_size,
                self.cfg.moe_intermediate_size,
                device="cpu",
                pin_memory=False,
                weight_dtype=weight_dtype,
                block_size=self.cfg.fp8_block_size,
            )

    def close(self) -> None:
        self._io_pool.shutdown(wait=True)

    def _stage_to_host_slot(self, layer_idx: int, host_slot_idx: int) -> int:
        """Runs on background I/O thread: waits for host slot DMA completion, then loads layer into pinned host slot."""
        if self.has_cuda:
            self._host_free[host_slot_idx].synchronize()
        dst = self._host_slots[host_slot_idx]
        item = self.model.moe_tier_store[layer_idx]
        if isinstance(item, str):
            read_moe_slab_into(item, dst)
        else:
            _copy_payload_async(dst, item, non_blocking=False)
        return host_slot_idx

    def iter_sweep(self, layer_order: Sequence[int]):
        """Context-style generator yielding `(layer_idx, gpu_moe_payload)` with full async double-buffering."""
        if not self.has_cuda or not self._gpu_slots:
            # Pure GPU-resident (all Tier 0) or CPU test path
            for l_idx in layer_order:
                item = self.model.moe_tier_store[l_idx]
                if isinstance(item, str):
                    assert self._cpu_nvme_buf is not None
                    yield l_idx, read_moe_slab_into(item, self._cpu_nvme_buf)
                else:
                    yield l_idx, item
            return

        order = list(layer_order)
        tiers = self.plan.layer_tiers

        # Pre-index all slow layers (Tier 1 pageable + Tier 2 NVMe) that require host staging
        slow_positions = [
            pos
            for pos, l_idx in enumerate(order)
            if tiers[l_idx] in (StorageTier.HOST_PAGEABLE, StorageTier.NVME_SLAB)
        ]
        # Track active host futures: layer_idx -> (Future, host_slot_idx)
        host_futures: dict[int, tuple[Future, int]] = {}
        next_slow_ptr = 0
        next_host_slot = 0
        n_host_slots = len(self._host_slots)

        def _kick_host_prefetch() -> None:
            nonlocal next_slow_ptr, next_host_slot
            if n_host_slots == 0:
                return
            while next_slow_ptr < len(slow_positions) and len(host_futures) < n_host_slots:
                l_slow = order[slow_positions[next_slow_ptr]]
                h_slot = next_host_slot % n_host_slots
                next_host_slot += 1
                fut = self._io_pool.submit(self._stage_to_host_slot, l_slow, h_slot)
                host_futures[l_slow] = (fut, h_slot)
                next_slow_ptr += 1

        # Active GPU H2D transfers: layer_idx -> gpu_slot_idx
        gpu_inflight: dict[int, int] = {}
        next_gpu_slot = 0
        n_gpu_slots = len(self._gpu_slots)

        def _kick_h2d(l_idx: int) -> None:
            nonlocal next_gpu_slot
            t = tiers[l_idx]
            if t == StorageTier.GPU_HBM or l_idx in gpu_inflight:
                return
            g_slot = next_gpu_slot % n_gpu_slots
            next_gpu_slot += 1

            if t == StorageTier.HOST_PINNED:
                src_payload = self.model.moe_tier_store[l_idx]
                assert isinstance(src_payload, MoeExpertPayload)
                h_slot_used = None
            else:
                # Ensure background I/O finished staging into pinned host slot
                fut, h_slot_used = host_futures.pop(l_idx)
                fut.result()
                src_payload = self._host_slots[h_slot_used]

            with torch.cuda.stream(self._h2d_stream):
                self._h2d_stream.wait_event(self._gpu_free[g_slot])
                _copy_payload_async(self._gpu_slots[g_slot], src_payload, non_blocking=True)
                self._gpu_ready[g_slot].record(self._h2d_stream)
                if h_slot_used is not None:
                    self._host_free[h_slot_used].record(self._h2d_stream)
            gpu_inflight[l_idx] = g_slot
            # Immediately queue the next slow layer onto the freed host slot!
            if h_slot_used is not None:
                _kick_host_prefetch()

        # Prime pipeline: kick off up to 2 slow-tier host prefetches + first offloaded H2D
        _kick_host_prefetch()
        if order:
            _kick_h2d(order[0])

        compute_stream = torch.cuda.current_stream(self.device)
        for pos, l_idx in enumerate(order):
            # Look ahead to prefetch the NEXT offloaded layer in parallel with current layer's compute!
            for look_pos in range(pos + 1, min(len(order), pos + 3)):
                cand = order[look_pos]
                if tiers[cand] != StorageTier.GPU_HBM:
                    # Only launch H2D if we have a free GPU slot (at most 1 ahead of current)
                    if len(gpu_inflight) < n_gpu_slots:
                        _kick_h2d(cand)
                    break

            t = tiers[l_idx]
            if t == StorageTier.GPU_HBM:
                payload = self.model.moe_tier_store[l_idx]
                assert isinstance(payload, MoeExpertPayload)
                yield l_idx, payload
            else:
                if l_idx not in gpu_inflight:
                    _kick_h2d(l_idx)
                g_slot = gpu_inflight.pop(l_idx)
                compute_stream.wait_event(self._gpu_ready[g_slot])
                try:
                    yield l_idx, self._gpu_slots[g_slot]
                finally:
                    self._gpu_free[g_slot].record(compute_stream)


@dataclass
class FitStats:
    prompt_idx: int
    seq_len: int
    n_valid: int
    elapsed_s: float
    fwd_s: float
    bwd_s: float
    max_norm: float
    mean_rel_change: float


class Qwen38JLensEngine:
    """High-performance single-sweep Jacobian Lens fitter and inference applier for Qwen3.8-Flash-Next."""

    def __init__(self, model: LoadedQwen38Model) -> None:
        self.model = model
        self.cfg = model.config
        self.plan = model.plan
        self.device = model.device
        self.dtype = model.compute_dtype
        self.gdn_backend = GatedDeltaRuleBackend(preferred=self.plan.gdn_backend)
        self.streamer = AsyncMoEStreamer(model)

    def close(self) -> None:
        self.streamer.close()

    def _get_ple_embeddings(self, input_ids: torch.Tensor) -> dict[int, torch.Tensor]:
        """Gather `B=1` PLE n-gram embeddings on GPU for all PLE layers (takes <2 ms via mmap)."""
        out: dict[int, torch.Tensor] = {}
        for l_idx, provider in self.model.ple_providers.items():
            out[l_idx] = provider.lookup(input_ids, device=self.device).to(self.dtype)
        return out

    @torch.no_grad()
    def forward_capture(
        self,
        input_ids: torch.Tensor,  # [1, T]
        *,
        capture_layers: Sequence[int] | None = None,
    ) -> tuple[torch.Tensor, dict[int, torch.Tensor]]:
        """Run ONE fast `B=1` forward pass streaming layers 0..L-1; returns `(h_final, captured_H)`."""
        input_ids = input_ids.to(self.device)
        T = input_ids.shape[1]
        capture_set = set(range(self.cfg.num_hidden_layers)) if capture_layers is None else set(capture_layers)

        cos, sin = build_rope_cos_sin(
            seq_len=T,
            head_dim=self.cfg.head_dim,
            partial_rotary_factor=self.cfg.partial_rotary_factor,
            rope_theta=self.cfg.rope_theta,
            device=self.device,
            dtype=self.dtype,
        )
        ple_embs = self._get_ple_embeddings(input_ids)

        emb = F.embedding(input_ids, self.model.embed_tokens)  # [1, T, D]
        H = emb.repeat(1, 1, self.cfg.hc_count).contiguous()   # [1, T, hc_count * D]
        captured: dict[int, torch.Tensor] = {}

        for l_idx, experts in self.streamer.iter_sweep(range(self.cfg.num_hidden_layers)):
            H, _ = decoder_layer_fwd(
                H,
                self.model.layers[l_idx],
                experts,
                cos,
                sin,
                self.gdn_backend,
                ple_embeddings=ple_embs.get(l_idx),
                dequant_chunk_size=self.plan.expert_dequant_chunk,
                use_triton_dequant=self.plan.use_triton_fp8_dequant,
                keep_experts_dequantized=False,
            )
            if l_idx in capture_set:
                captured[l_idx] = H.detach()

        h_final, _, _ = gated_residual_fwd(H, self.model.hyper_connection_mixer)
        return h_final, captured

    @torch.no_grad()
    def jacobian_for_tokens(
        self,
        input_ids: torch.Tensor,  # [1, T]
        source_layers: Sequence[int],
        *,
        skip_first: int = SKIP_FIRST_N_POSITIONS,
    ) -> tuple[dict[int, torch.Tensor], int, int, float, float]:
        """Compute exact per-prompt `J_l = E[d h_final / d H_l]` (`[D, hc_count * D]` fp32 CPU) for all `source_layers`.

        Uses single-sweep layer-by-layer reverse propagation:
          - Forward sweep (0 -> L-1): saves ONLY `[1, T, 10240]` layer inputs (~120 MB total!).
          - Backward sweep (L-1 -> min_source + 1): holds ALL `k_pass` cotangents in ONE in-place
            GPU buffer `G`, dequantizing each layer's hit experts ONCE and overlapping next-layer
            DMA/NVMe prefetch 100% behind backward compute!
        """
        t0 = time.perf_counter()
        input_ids = input_ids.to(self.device)
        T = int(input_ids.shape[1])
        pos_mask = valid_position_mask(T, skip_first=skip_first)
        valid_pos = pos_mask.nonzero(as_tuple=True)[0].to(self.device)
        n_valid = int(valid_pos.numel())

        D = self.cfg.hidden_size
        hc = self.cfg.hc_count
        D_hc = D * hc
        n_layers = self.cfg.num_hidden_layers
        src_list = sorted(set(source_layers))
        src_set = set(src_list)
        min_src = src_list[0]

        cos, sin = build_rope_cos_sin(
            seq_len=T,
            head_dim=self.cfg.head_dim,
            partial_rotary_factor=self.cfg.partial_rotary_factor,
            rope_theta=self.cfg.rope_theta,
            device=self.device,
            dtype=self.dtype,
        )
        ple_embs = self._get_ple_embeddings(input_ids)

        # ---- 1. Forward Sweep (0 -> L-1, B = 1) ----
        emb = F.embedding(input_ids, self.model.embed_tokens)
        H = emb.repeat(1, 1, hc).contiguous()

        # Save each layer's input H_{l-1} for layers >= min_src + 1 (only ~2.5 MB/layer!)
        layer_inputs: dict[int, torch.Tensor] = {}
        for l_idx, experts in self.streamer.iter_sweep(range(n_layers)):
            if l_idx > min_src:
                layer_inputs[l_idx] = H
            H, _ = decoder_layer_fwd(
                H,
                self.model.layers[l_idx],
                experts,
                cos,
                sin,
                self.gdn_backend,
                ple_embeddings=ple_embs.get(l_idx),
                dequant_chunk_size=self.plan.expert_dequant_chunk,
                use_triton_dequant=self.plan.use_triton_fp8_dequant,
                keep_experts_dequantized=False,
            )

        # Final hyper_connection_mixer forward (H_{L-1} -> h_final [1, T, D])
        _, _, mixer_saved = gated_residual_fwd(H, self.model.hyper_connection_mixer)
        if self.device.type == "cuda":
            torch.cuda.synchronize()
        t_fwd = time.perf_counter() - t0

        # ---- 2. Backward Sweep (L-1 -> min_src + 1, k_pass cotangents at once!) ----
        t_bwd_start = time.perf_counter()
        jacobians = {l: torch.zeros((D, D_hc), dtype=torch.float32) for l in src_list}
        k_pass = min(self.plan.k_pass, D)
        bwd_layers = list(range(n_layers - 1, min_src, -1))

        for k_start in range(0, D, k_pass):
            k_end = min(k_start + k_pass, D)
            K_cur = k_end - k_start

            # Build initial one-hot cotangent at target h_final for valid positions,
            # then immediately backprop through hyper_connection_mixer -> G_{L-1} [K_cur, T, D_hc]!
            G = torch.zeros((K_cur, T, D_hc), device=self.device, dtype=self.dtype)
            d_target_chunk = torch.zeros((min(self.plan.moe_k_chunk, K_cur), T, D), device=self.device, dtype=self.dtype)
            k_sub_idx = torch.arange(d_target_chunk.shape[0], device=self.device)
            for c0 in range(0, K_cur, d_target_chunk.shape[0]):
                c1 = min(c0 + d_target_chunk.shape[0], K_cur)
                nc = c1 - c0
                d_target_chunk[:nc].zero_()
                d_target_chunk[
                    k_sub_idx[:nc, None],
                    valid_pos[None, :],
                    (k_start + c0) + k_sub_idx[:nc, None],
                ] = 1.0
                G[c0:c1] = gated_residual_vjp(
                    d_target_chunk[:nc], None, self.model.hyper_connection_mixer, mixer_saved
                )
            del d_target_chunk

            # Layer L-1 output H_{L-1} Jacobian!
            if (n_layers - 1) in src_set:
                rows = G[:, valid_pos, :].float().mean(dim=1).cpu()
                jacobians[n_layers - 1][k_start:k_end, :] = rows

            # Reverse layer sweep: L-1 -> min_src + 1
            for l_idx, experts in self.streamer.iter_sweep(bwd_layers):
                # Recompute ONE tiny B=1 forward of layer l_idx (~0.5 ms), keeping active experts
                # dequantized in BF16 when E_hit <= expert_dequant_chunk!
                _, saved_l = decoder_layer_fwd(
                    layer_inputs[l_idx],
                    self.model.layers[l_idx],
                    experts,
                    cos,
                    sin,
                    self.gdn_backend,
                    ple_embeddings=ple_embs.get(l_idx),
                    dequant_chunk_size=self.plan.expert_dequant_chunk,
                    use_triton_dequant=self.plan.use_triton_fp8_dequant,
                    keep_experts_dequantized=True,
                )

                # In-place VJP across ALL K_cur cotangents!
                decoder_layer_vjp_inplace(
                    G,
                    self.model.layers[l_idx],
                    experts,
                    saved_l,
                    self.gdn_backend,
                    attn_k_chunk=self.plan.attn_k_chunk,
                    moe_k_chunk=self.plan.moe_k_chunk,
                    dequant_chunk_size=self.plan.expert_dequant_chunk,
                    use_triton_dequant=self.plan.use_triton_fp8_dequant,
                    use_triton_swiglu=self.plan.use_triton_swiglu_vjp,
                )
                del saved_l

                # After layer l_idx backward, G is now d h_final / d H_{l_idx - 1}!
                prev_layer = l_idx - 1
                if prev_layer in src_set:
                    rows = G[:, valid_pos, :].float().mean(dim=1).cpu()
                    jacobians[prev_layer][k_start:k_end, :] = rows

            del G

        if self.device.type == "cuda":
            torch.cuda.synchronize()
        t_bwd = time.perf_counter() - t_bwd_start
        return jacobians, T, n_valid, t_fwd, t_bwd

    def fit(
        self,
        tokenized_prompts: Sequence[torch.Tensor],
        *,
        source_layers: Sequence[int] | None = None,
        skip_first: int = SKIP_FIRST_N_POSITIONS,
        checkpoint_path: str | None = None,
        checkpoint_every: int = 10,
        checkpoint_manager: Any | None = None,
        resume: bool = True,
        on_prompt_done: Any = None,
    ) -> Qwen38JacobianLens:
        """Fit `J_l` over tokenized prompts `[1, T]` with zero-GPU-stall async checkpointing & resume.

        Supports either:
          - `checkpoint_manager=AsyncCheckpointManager(...)` (recommended: local NVMe + async Drive/HF Hub,
            count + time triggers, single-file `.safetensors`, directly loadable as a lens on interrupt!), OR
          - `checkpoint_path=".../ckpt.safetensors"` (or legacy `.pt`) for simple local checkpointing.
        """
        from qwen38_jlens.checkpoint import AsyncCheckpointManager, CheckpointConfig

        n_layers = self.cfg.num_hidden_layers
        D = self.cfg.hidden_size
        D_hc = D * self.cfg.hc_count
        src_layers = list(range(n_layers)) if source_layers is None else sorted(set(source_layers))

        owned_mgr = False
        mgr: AsyncCheckpointManager | None = checkpoint_manager
        if mgr is None and checkpoint_path is not None:
            if checkpoint_path.endswith(".safetensors"):
                mgr = AsyncCheckpointManager(
                    CheckpointConfig(
                        local_path=checkpoint_path,
                        every_n_prompts=max(1, checkpoint_every),
                        every_minutes=0.0,
                        ckpt_dtype="float16",
                    )
                )
                owned_mgr = True

        # ---- Resume from freshest checkpoint (Local / Drive / HF Hub or legacy .pt) ----
        jacobian_sum: dict[int, torch.Tensor]
        n_done: int
        next_idx: int
        resumed = None
        if resume and mgr is not None:
            resumed = mgr.load_best_resume(
                expected_source_layers=src_layers, expected_skip_first=skip_first
            )
        elif resume and checkpoint_path is not None and os.path.exists(checkpoint_path):
            state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
            if state.get("source_layers") != src_layers or state.get("skip_first") != skip_first:
                raise ValueError(
                    f"Checkpoint at {checkpoint_path} has incompatible source_layers/skip_first; pass resume=False."
                )
            resumed = (state["jacobian_sum"], int(state["n_done"]), int(state["next_idx"]))

        if resumed is not None:
            jacobian_sum, n_done, next_idx = resumed
            logger.info("Resuming J-lens fit: %d/%d prompts already processed (n_done=%d).", next_idx, len(tokenized_prompts), n_done)
        else:
            jacobian_sum = {l: torch.zeros((D, D_hc), dtype=torch.float32) for l in src_layers}
            n_done = 0
            next_idx = 0
        resumed_next_idx = next_idx

        def _save_legacy_pt() -> None:
            if checkpoint_path is None:
                return
            parent = os.path.dirname(os.path.abspath(checkpoint_path))
            if parent:
                os.makedirs(parent, exist_ok=True)
            tmp = f"{checkpoint_path}.tmp.{os.getpid()}"
            torch.save(
                {
                    "jacobian_sum": jacobian_sum,
                    "n_done": n_done,
                    "next_idx": next_idx,
                    "source_layers": src_layers,
                    "skip_first": skip_first,
                    "d_model": D,
                    "hc_count": self.cfg.hc_count,
                },
                tmp,
            )
            os.replace(tmp, checkpoint_path)

        sqrt_d = math.sqrt(D)
        prompts_since_ckpt = 0
        try:
            for p_idx, ids in enumerate(tokenized_prompts):
                if p_idx < next_idx:
                    continue
                t0 = time.perf_counter()
                try:
                    per_J, seq_len, n_valid, t_fwd, t_bwd = self.jacobian_for_tokens(
                        ids, src_layers, skip_first=skip_first
                    )
                except ValueError as exc:
                    logger.warning("Skipping prompt %d: %s", p_idx, exc)
                    next_idx = p_idx + 1
                    continue

                max_norm = max(per_J[l].norm().item() for l in src_layers) / sqrt_d
                if n_done > 0:
                    rel_change = max(
                        (
                            (per_J[l] - jacobian_sum[l] / n_done).norm()
                            / ((n_done + 1) * max(1e-8, (jacobian_sum[l] / n_done).norm().item()))
                        ).item()
                        for l in src_layers
                    )
                else:
                    rel_change = float("nan")

                for l in src_layers:
                    jacobian_sum[l].add_(per_J[l])
                n_done += 1
                next_idx = p_idx + 1
                prompts_since_ckpt += 1
                elapsed = time.perf_counter() - t0

                stats = FitStats(
                    prompt_idx=p_idx,
                    seq_len=seq_len,
                    n_valid=n_valid,
                    elapsed_s=elapsed,
                    fwd_s=t_fwd,
                    bwd_s=t_bwd,
                    max_norm=max_norm,
                    mean_rel_change=rel_change,
                )
                logger.info(
                    "Prompt %d/%d  T=%d (valid=%d)  %.2fs (fwd=%.2fs bwd=%.2fs)  max||J||/sqrt(d)=%.3f  d_mean=%.2e",
                    p_idx + 1, len(tokenized_prompts), seq_len, n_valid,
                    elapsed, t_fwd, t_bwd, max_norm, rel_change,
                )
                if on_prompt_done is not None:
                    on_prompt_done(stats)

                if mgr is not None and mgr.should_checkpoint(prompts_since_ckpt):
                    mgr.save_async(
                        jacobian_sum,
                        n_done=n_done,
                        next_idx=next_idx,
                        skip_first=skip_first,
                        d_model=D,
                        hc_count=self.cfg.hc_count,
                        wait_remote=False,
                    )
                    prompts_since_ckpt = 0
                elif mgr is None and checkpoint_path is not None and checkpoint_every > 0 and prompts_since_ckpt >= checkpoint_every:
                    _save_legacy_pt()
                    prompts_since_ckpt = 0

        except KeyboardInterrupt:
            logger.warning("KeyboardInterrupt caught at next_idx=%d (n_done=%d); flushing checkpoint safely before exit!", next_idx, n_done)
            if mgr is not None and n_done > 0 and next_idx > resumed_next_idx:
                mgr.save_async(
                    jacobian_sum,
                    n_done=n_done,
                    next_idx=next_idx,
                    skip_first=skip_first,
                    d_model=D,
                    hc_count=self.cfg.hc_count,
                    wait_remote=True,
                )
            elif checkpoint_path is not None and n_done > 0 and next_idx > resumed_next_idx:
                _save_legacy_pt()
            if owned_mgr and mgr is not None:
                mgr.close()
            raise

        # Final checkpoint flush (wait for Drive / HF Hub upload to finish!)
        if mgr is not None and n_done > 0:
            mgr.flush()
            if next_idx > resumed_next_idx:
                needs_final = (
                    prompts_since_ckpt > 0
                    or mgr._last_local_next_idx < next_idx
                    or (mgr.has_remote and mgr._last_synced_next_idx < next_idx)
                )
                if needs_final:
                    mgr.save_async(
                        jacobian_sum,
                        n_done=n_done,
                        next_idx=next_idx,
                        skip_first=skip_first,
                        d_model=D,
                        hc_count=self.cfg.hc_count,
                        wait_remote=True,
                    )
            if owned_mgr:
                mgr.close()
        elif checkpoint_path is not None and n_done > 0 and next_idx > resumed_next_idx:
            _save_legacy_pt()

        if n_done == 0:
            raise ValueError("No prompts were long enough to fit the Jacobian lens.")

        # Divide in-place to avoid allocating a duplicate 4.69 GiB fp32 dict on tight-RAM hosts!
        for l in src_layers:
            jacobian_sum[l].div_(float(n_done))
        return Qwen38JacobianLens(
            jacobians=jacobian_sum,
            n_prompts=n_done,
            d_model=D,
            hc_count=self.cfg.hc_count,
        )

    @torch.no_grad()
    def apply_lens(
        self,
        lens: Qwen38JacobianLens,
        input_ids: torch.Tensor,  # [1, T]
        *,
        layers: Sequence[int] | None = None,
        positions: Sequence[int] | None = None,
        use_jacobian: bool = True,
    ) -> tuple[dict[int, torch.Tensor], torch.Tensor]:
        """Apply fitted J-lens (or logit-lens baseline if `use_jacobian=False`) on `input_ids`.

        Returns `(lens_logits_by_layer, model_final_logits)`, each of shape `[n_positions, vocab_size]` on CPU.
        """
        req_layers = lens.source_layers if layers is None else sorted(set(layers))
        h_final, captured = self.forward_capture(input_ids, capture_layers=req_layers)

        def _sel(x: torch.Tensor) -> torch.Tensor:
            t0 = x[0]  # [T, ...]
            return t0 if positions is None else t0[list(positions)]

        # Unembed via CPU or GPU lm_head
        lm_w = self.model.lm_head.float()  # [vocab, D] on CPU (avoids extra 1.2 GiB VRAM peak)
        model_logits = F.linear(_sel(h_final).float().cpu(), lm_w)

        lens_logits: dict[int, torch.Tensor] = {}
        for l in req_layers:
            H_sel = _sel(captured[l])  # [P, hc_count * D]
            if use_jacobian:
                h_proj = lens.transport(H_sel, l).cpu()
            else:
                # Logit-lens baseline for 4-branch GR: pass H_l through final hyper_connection_mixer!
                h_proj, _, _ = gated_residual_fwd(H_sel.unsqueeze(0), self.model.hyper_connection_mixer)
                h_proj = h_proj[0].float().cpu()
            lens_logits[l] = F.linear(h_proj, lm_w)

        return lens_logits, model_logits


In [ ]:
%%writefile qwen38_jlens/__init__.py
"""Ultra-fast Jacobian Lens (J-lens) fitting engine for Qwen3.8-Flash-Next (qwen4_exp)."""

from qwen38_jlens.checkpoint import (
    AsyncCheckpointManager,
    CheckpointConfig,
    load_jlens_safetensors,
    peek_checkpoint_metadata,
    save_jlens_safetensors,
)
from qwen38_jlens.engine import FitStats, Qwen38JLensEngine, valid_position_mask
from qwen38_jlens.hardware import HardwareProfile, OffloadPlan, StorageTier, build_offload_plan, detect_hardware
from qwen38_jlens.kernels import GatedDeltaRuleBackend, dequantize_fp8_blockwise, grouped_mm, swiglu_vjp_pack
from qwen38_jlens.lens import Qwen38JacobianLens, render_slice_html
from qwen38_jlens.loader import LoadedQwen38Model, Qwen38TextConfig, load_qwen38_streaming
from qwen38_jlens.ple_mmap import InMemoryPleProvider, MmapPleTable, PleHashSpec, PreextractedPleProvider

__all__ = [
    "AsyncCheckpointManager",
    "CheckpointConfig",
    "FitStats",
    "GatedDeltaRuleBackend",
    "HardwareProfile",
    "InMemoryPleProvider",
    "LoadedQwen38Model",
    "MmapPleTable",
    "OffloadPlan",
    "PleHashSpec",
    "PreextractedPleProvider",
    "Qwen38JLensEngine",
    "Qwen38JacobianLens",
    "Qwen38TextConfig",
    "StorageTier",
    "build_offload_plan",
    "dequantize_fp8_blockwise",
    "detect_hardware",
    "grouped_mm",
    "load_jlens_safetensors",
    "load_qwen38_streaming",
    "peek_checkpoint_metadata",
    "render_slice_html",
    "save_jlens_safetensors",
    "swiglu_vjp_pack",
    "valid_position_mask",
]


## 4. Live Hardware Auto-Detection, Concrete Offload Plan & Kernel Micro-Benchmark

In [ ]:
import logging, time, torch
from qwen38_jlens import (
    GatedDeltaRuleBackend,
    StorageTier,
    build_offload_plan,
    dequantize_fp8_blockwise,
    detect_hardware,
    grouped_mm,
    swiglu_vjp_pack,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)

SOURCE_LAYERS = list(range(0, 48, LAYER_STRIDE))
if 47 not in SOURCE_LAYERS:
    SOURCE_LAYERS.append(47)

profile = detect_hardware(LOCAL_WORK_DIR)
plan = build_offload_plan(
    profile,
    num_layers=48,
    d_model=2560,
    hc_count=4,
    num_experts=512,
    moe_intermediate_size=640,
    max_seq_len=MAX_SEQ_LEN,
    num_source_layers=len(SOURCE_LAYERS),
    force_ple_mode=None if PLE_MODE == "auto" else PLE_MODE,
)
print(plan.summary())

# Show exact interleaved layer -> tier map (visual check!)
tier_sym = {
    StorageTier.GPU_HBM: "G",
    StorageTier.HOST_PINNED: "P",
    StorageTier.HOST_PAGEABLE: "R",
    StorageTier.NVME_SLAB: "D",
}
print("\nInterleaved Layer Placement Map (G=GPU HBM, P=Pinned RAM, R=Pageable RAM, D=NVMe Slab):")
for row_start in range(0, 48, 12):
    print("  " + " ".join(f"L{l:02d}:{tier_sym[plan.layer_tiers[l]]}" for l in range(row_start, row_start + 12)))

# ---- 3-Second Live Kernel Verification & Speed Micro-Benchmark ----
print("\nRunning live GPU kernel parity & bandwidth benchmark...")
gdn_bench = GatedDeltaRuleBackend(preferred=plan.gdn_backend, verify_on_init=True)
print(f"  [OK] Active GDN kernel : {gdn_bench.name} (direct={gdn_bench._use_direct})")

dev = torch.device("cuda:0")
E_act = min(380, plan.expert_dequant_chunk)
w_fp8 = torch.randn(E_act, 1280, 2560, device=dev).to(torch.float8_e4m3fn)
s_fp32 = torch.ones(E_act, 10, 20, device=dev, dtype=torch.float32) * 0.05
_ = dequantize_fp8_blockwise(w_fp8, s_fp32, use_triton=plan.use_triton_fp8_dequant)
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(5):
    w_bf16 = dequantize_fp8_blockwise(w_fp8, s_fp32, use_triton=plan.use_triton_fp8_dequant)
torch.cuda.synchronize()
dt_deq = (time.perf_counter() - t0) / 5
bytes_rw = E_act * 1280 * 2560 * (1 + 2)
print(f"  [OK] Triton FP8->BF16 dequant ({E_act} experts): {dt_deq*1e3:.2f} ms ({bytes_rw / dt_deq / 1e9:.0f} GB/s)")

K_blk = plan.moe_k_chunk
S_rows = 128 * 10
counts = torch.full((E_act,), max(1, S_rows // E_act), device=dev, dtype=torch.int32)
offs_K = torch.cumsum(counts * K_blk, dim=0, dtype=torch.int32)
M_tot = int(offs_K[-1].item())
dGU = torch.randn(M_tot, 1280, device=dev, dtype=torch.bfloat16)
_ = grouped_mm(dGU, w_bf16, offs_K)
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(3):
    out_gmm = grouped_mm(dGU, w_bf16, offs_K)
torch.cuda.synchronize()
dt_gmm = (time.perf_counter() - t0) / 3
tflops = (2.0 * M_tot * 1280 * 2560) / dt_gmm / 1e12
print(f"  [OK] MoE grouped_mm VJP (M={M_tot}, K=1280, N=2560): {dt_gmm*1e3:.2f} ms ({tflops:.1f} TFLOPS)")

del w_fp8, s_fp32, w_bf16, dGU, out_gmm
torch.cuda.empty_cache()


## 5. Load Tokenizer & Pretraining-Like Fitting Corpus (`Salesforce/wikitext`)
Matches `anthropics/jacobian-lens`: streams `Salesforce/wikitext` (`wikitext-103-raw-v1`, `min_chars=600`), truncates to `MAX_SEQ_LEN=128` tokens, and prepends BOS when missing.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=False)

def encode_prompt(text: str, max_len: int = MAX_SEQ_LEN) -> torch.Tensor:
    ids = tokenizer.encode(text, add_special_tokens=False)
    bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
    if bos_id is not None and (not ids or ids[0] != bos_id):
        ids = [bos_id] + ids
    ids = ids[:max_len]
    return torch.tensor([ids], dtype=torch.long)

def load_wikitext_slice(offset: int, n_prompts: int, min_chars: int = 600) -> list[str]:
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
    out = []
    seen = 0
    for rec in ds:
        txt = rec["text"]
        if len(txt.strip()) >= min_chars:
            if seen >= offset:
                out.append(txt)
                if len(out) >= n_prompts:
                    break
            seen += 1
    return out

raw_prompts = load_wikitext_slice(PROMPT_OFFSET, N_PROMPTS)
tokenized_prompts = [encode_prompt(p, MAX_SEQ_LEN) for p in raw_prompts]

EVAL_PROMPTS = [
    "Fact: The currency used in the country shaped like a boot is",
    "The capital of the country where Mount Everest is located is",
    "     _______     \n   /         \\   \n  /  ~     ~  \\  \n (   o     o   ) \n |      ^      | \n  \\   \u2014\u2014\u2014   /  \n   \\_________/   \n\nWhat is this?",
]
eval_tokenized = [encode_prompt(p, max_len=256) for p in EVAL_PROMPTS]

lengths = [int(t.shape[1]) for t in tokenized_prompts]
print(f"Prepared {len(tokenized_prompts)} WikiText prompts (offset={PROMPT_OFFSET}): "
      f"min_len={min(lengths)}, median={sorted(lengths)[len(lengths)//2]}, max_len={max(lengths)}")


## 6. Stream-Load `Qwen/Qwen3.8-Flash-Next-FP8` Directly Into the 3-Tier Hierarchy
- Skips Vision encoder and MTP shards automatically.
- Loads all dense BF16 weights (`~8.2 GiB`) directly to GPU (immune to `transformers#48349`).
- Keeps 51B PLE n-gram table out of RAM (`MADV_RANDOM` NVMe mmap or tight-disk pre-extraction).
- Unpacks each raw safetensors shard straight into **Tier 0 GPU HBM / Tier 1 Pinned+Pageable RAM / Tier 2 contiguous NVMe slabs**, evicting raw HF blobs immediately so Colab disk stays well below 235 GiB.

In [ ]:
import shutil, time, psutil, torch
from qwen38_jlens import Qwen38JLensEngine, load_qwen38_streaming

t_load = time.perf_counter()
loaded_model = load_qwen38_streaming(
    MODEL_ID,
    plan,
    slab_dir=SLAB_DIR,
    ple_dir=PLE_DIR,
    tokenized_prompts_for_preextract=(tokenized_prompts + eval_tokenized) if plan.ple_mode == "preextract" else None,
    hf_token=HF_TOKEN,
    compute_dtype=torch.bfloat16,
    progress_callback=lambda msg: print(f"[Loader {time.perf_counter()-t_load:5.0f}s] {msg}", flush=True),
)
engine = Qwen38JLensEngine(loaded_model)

vram_used = torch.cuda.memory_allocated() / 1024**3
ram_used = psutil.virtual_memory().used / 1024**3
disk_free = shutil.disk_usage(LOCAL_WORK_DIR).free / 1024**3
print(f"\nReady for J-Lens fitting in {time.perf_counter()-t_load:.1f}s!")
print(f"  Active GDN Kernel : {engine.gdn_backend.name} (direct={engine.gdn_backend._use_direct})")
print(f"  Live Telemetry    : VRAM alloc={vram_used:.1f} GiB | RAM used={ram_used:.1f} GiB | Disk free={disk_free:.1f} GiB")


## 7. Fit Qwen3.8-Flash-Next Jacobian Lens (`J_l ∈ R^{2560 × 10240}`) with Zero-Stall Async Checkpointing
- Saves `J_mean` as a single `.safetensors` file (`2.34 GiB` in `fp16`) to local NVMe in `< 1 s`, then syncs in the background to Google Drive and/or Hugging Face Hub with **0 s GPU stall**.
- Interrupting the cell (`Stop` / `KeyboardInterrupt`) cleanly flushes the latest checkpoint before exiting.
- Re-running after any Colab disconnect automatically resumes from whichever target (Local / Drive / HF Hub) has the highest `next_idx`!

In [ ]:
import os, shutil, time, torch
from tqdm.auto import tqdm
from qwen38_jlens import AsyncCheckpointManager, CheckpointConfig, peek_checkpoint_metadata

ckpt_mgr = AsyncCheckpointManager(
    CheckpointConfig(
        local_path=LOCAL_CKPT_PATH,
        drive_path=DRIVE_CKPT_PATH,
        hf_repo_id=HF_CHECKPOINT_REPO,
        hf_path_in_repo=CKPT_FILENAME,
        hf_token=HF_TOKEN,
        hf_private=HF_REPO_PRIVATE,
        ckpt_dtype=CKPT_DTYPE,
        every_n_prompts=CHECKPOINT_EVERY_PROMPTS,
        every_minutes=CHECKPOINT_EVERY_MINUTES,
        model_name=MODEL_ID,
    )
)

# Quick preview of existing checkpoints before starting
for lbl, p in (("Local NVMe", LOCAL_CKPT_PATH), ("Google Drive", DRIVE_CKPT_PATH)):
    meta = peek_checkpoint_metadata(p) if p else None
    if meta:
        print(f"Found existing {lbl} checkpoint: next_idx={meta['next_idx']}, n_done={meta['n_done']} ({meta['ckpt_dtype']})")

pbar = tqdm(total=len(tokenized_prompts), desc="Fitting Qwen3.8 J-Lens", unit="prompt")
times = []

def _on_prompt(stats):
    times.append(stats.elapsed_s)
    avg_s = sum(times[-10:]) / len(times[-10:])
    vram_pk = torch.cuda.max_memory_allocated() / 1024**3
    pbar.n = stats.prompt_idx + 1
    pbar.set_postfix(
        sec=f"{stats.elapsed_s:.1f}s",
        fwd=f"{stats.fwd_s:.1f}s",
        bwd=f"{stats.bwd_s:.1f}s",
        avg=f"{avg_s:.1f}s/p",
        d_mean=f"{stats.mean_rel_change:.1e}",
        vram=f"{vram_pk:.1f}G",
        ckpt=ckpt_mgr.last_status,
    )
    pbar.refresh()

torch.cuda.reset_peak_memory_stats()
t_fit = time.perf_counter()

try:
    lens = engine.fit(
        tokenized_prompts,
        source_layers=SOURCE_LAYERS,
        skip_first=SKIP_FIRST,
        checkpoint_manager=ckpt_mgr,
        resume=True,
        on_prompt_done=_on_prompt,
    )
finally:
    pbar.close()
    ckpt_mgr.close()

# Save final lens (.safetensors with collapsed J included; optional legacy .pt)
lens.save(FINAL_LENS_LOCAL, dtype=torch.float16, include_collapsed=True)
if FINAL_LENS_DRIVE:
    shutil.copyfile(FINAL_LENS_LOCAL, FINAL_LENS_DRIVE + ".tmp")
    os.replace(FINAL_LENS_DRIVE + ".tmp", FINAL_LENS_DRIVE)

if ALSO_SAVE_LEGACY_PT:
    pt_path = os.path.join(OUT_DIR, f"qwen38_flash_next_jlens_{TAG}.pt")
    lens.save(pt_path, dtype=torch.float16, include_collapsed=True)
    print(f"Saved legacy .pt lens: {pt_path}")

if HF_CHECKPOINT_REPO:
    from huggingface_hub import HfApi
    _hf_api = HfApi(token=HF_TOKEN)
    _hf_api.create_repo(
        repo_id=HF_CHECKPOINT_REPO,
        repo_type="model",
        private=HF_REPO_PRIVATE,
        exist_ok=True,
    )
    _hf_api.upload_file(
        path_or_fileobj=FINAL_LENS_LOCAL,
        path_in_repo=LENS_FILENAME,
        repo_id=HF_CHECKPOINT_REPO,
        repo_type="model",
        commit_message=f"Final Qwen3.8-Flash-Next J-lens (n={lens.n_prompts})",
    )
    print(f"Uploaded final lens to HF Hub: {HF_CHECKPOINT_REPO}/{LENS_FILENAME}")

total_min = (time.perf_counter() - t_fit) / 60.0
peak_vram = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nSaved final lens: {FINAL_LENS_DRIVE or FINAL_LENS_LOCAL}")
print(f"  Summary   : {lens}")
print(f"  Total time: {total_min:.2f} min ({(total_min*60)/max(1, len(times)):.2f} s/prompt)")
print(f"  Peak VRAM : {peak_vram:.2f} GiB")


## 8. Verify Fitted J-Lens: Multi-Hop Readout, 4-Branch Gated Residual Norms & Interactive Slice
Compares **J-Lens** (`lm_head(H_l @ J_l.T)`) vs **Logit-Lens** (`lm_head(hyper_connection_mixer(H_l))`) on two-hop reasoning prompts and inspects how signal distributes across Qwen3.8-Flash-Next's 4 Gated Residual branches.

In [ ]:
import torch
from IPython.display import HTML, display
from qwen38_jlens import render_slice_html

# 1. Inspect 4-branch Gated Residual Frobenius norms across depth
bnorms = lens.branch_norms()
print("4-Branch Gated Residual Jacobian Norms ||J_l[:, b, :]||_F / sqrt(2560):")
for l in [0, 8, 16, 24, 32, 40, 47]:
    if l in bnorms:
        b_str = "  ".join(f"b{b}={v:.3f}" for b, v in enumerate(bnorms[l]))
        print(f"  L{l:02d}: {b_str}")

# 2. Multi-hop J-lens vs Logit-lens readout at final token
test_prompt = EVAL_PROMPTS[0]
test_ids = eval_tokenized[0]
probe_layers = [l for l in (8, 16, 24, 32, 40, 46, 47) if l in lens.jacobians]

jl_logits, model_logits = engine.apply_lens(lens, test_ids, layers=probe_layers, positions=[-1], use_jacobian=True)
ll_logits, _ = engine.apply_lens(lens, test_ids, layers=probe_layers, positions=[-1], use_jacobian=False)

def top5(logits_1d):
    vals, idxs = torch.softmax(logits_1d.float(), dim=-1).topk(5)
    return [f"{tokenizer.decode([int(i)])!r}({float(v)*100:.0f}%)" for v, i in zip(vals, idxs)]

print(f"\nPrompt: {test_prompt!r}")
for l in probe_layers:
    print(f"  L{l:02d} Logit-Lens: {top5(ll_logits[l][0])}")
    print(f"  L{l:02d} J-Lens    : {top5(jl_logits[l][0])}")
print(f"  Model Output   : {top5(model_logits[0])}")

# 3. Interactive position x layer HTML slice heatmap (every 4th layer for clean viewing)
slice_layers = [l for l in lens.source_layers if l % 4 == 0 or l == lens.source_layers[-1]]
all_jl, all_model = engine.apply_lens(lens, test_ids, layers=slice_layers, positions=None, use_jacobian=True)

tok_strs = [tokenizer.decode([int(t)]) for t in test_ids[0].tolist()]
layer_top1 = {}
for l in slice_layers:
    probs = torch.softmax(all_jl[l].float(), dim=-1)
    pv, pi = probs.max(dim=-1)
    layer_top1[l] = [(tokenizer.decode([int(i)]), float(v)) for v, i in zip(pv, pi)]

m_probs = torch.softmax(all_model.float(), dim=-1)
mpv, mpi = m_probs.max(dim=-1)
model_top1 = [(tokenizer.decode([int(i)]), float(v)) for v, i in zip(mpv, mpi)]

display(HTML(render_slice_html(tok_strs, layer_top1, model_top1, title=f"Qwen3.8-Flash-Next J-Lens (n={lens.n_prompts})")))


## 9. (Optional) Merge Parallel Colab Shards (`n = 1000`) or Use an Interrupted Checkpoint Directly
- Because checkpoints store `J_mean` + `n_done` in `.safetensors`, **any interrupted checkpoint** can be loaded directly via `Qwen38JacobianLens.load(ckpt_path)`.
- Run multiple Colab sessions with `PROMPT_OFFSET = 0, 100, 200, ...` pointing to the same Drive folder or HF Hub repo, then merge them into one exact `n_prompts`-weighted lens.

In [ ]:
# Uncomment to merge multiple sharded `.safetensors` lenses (or interrupted checkpoints!):
# import glob
# from qwen38_jlens import Qwen38JacobianLens
# shard_files = sorted(glob.glob(f"{OUT_DIR}/qwen38_flash_next_jlens_off*.safetensors"))
# print("Merging:", shard_files)
# merged = Qwen38JacobianLens.merge([Qwen38JacobianLens.load(p) for p in shard_files])
# merged_path = f"{OUT_DIR}/qwen38_flash_next_jlens_merged_n{merged.n_prompts}.safetensors"
# merged.save(merged_path, include_collapsed=True)
# print("Saved merged lens:", merged_path, merged)

# Optional cleanup of temporary local NVMe slabs / PLE shards:
# engine.close()
